# MEM-style hybrid memory — VLM continuous control of a Go2 quadruped

This notebook **replaces** the previous three history modes (stateless `fresh`, raw `frames`,
free-text `note`) with a single **hybrid memory system adapted from Multi-Scale Embodied Memory
(MEM)** — the long/short-term memory architecture from Physical Intelligence
([blog](https://www.pi.website/research/memory), MEM: *Multi-Scale Embodied Memory for Vision
Language Action Models*, 2026).

Everything else is carried over unchanged: the VLM emits a **continuous** `(v_x, v_y, w_z)` fed
straight to the PPO controller, post-hoc semantic bucketing for saliency, integrated gradients,
and all the closed-loop bug fixes.

**Cell map**
- §0–§2  Runtime, Drive, headless rendering, dependencies
- §3  ⚙️ CONFIG (sim env, contract prompt, task instruction, **MEM memory knobs**)
- §4  `Go2Env` simulation environment
- §5  Shared utilities (render, hi-q scene, arena, camera geometry)
- §6  **Feasibility analysis: MEM → this project** (read this)
- §7  **MEM hybrid-memory planner** (`mem_planner.py`) — short-term video + long-term language memory
- §8  Post-hoc semantic bucketing (saliency / IG)
- §9  PPO training (skip if a checkpoint exists)
- §10 Load policy + MEM planner
- §11 Closed-loop runtime (continuous command, dual memory updated each step)
- §12 Comprehensive evaluation — batched + per-category trial runs, showcase videos (MEM-V2 planner)
- §13 Saliency / integrated-gradients heatmaps projected into the 3D arena, plus the MEM-architecture ablation engine (§13-ABL, §13c–§13j)
- §14 Saliency-accumulation videos (single-frame baseline vs. the fully faithful MEM driver)
- §15 Cinematic Blender Cycles render, with saliency shown as a projected heat field

**Ground truth for the experiments (§12–§15):** every trial is driven by the **MEM-V2 planner**
(§12a — the last **4 recent frames ~0.33 s apart** fed as a native Qwen video, **no anchor frame**,
plus the compressed long-term text memory), queried at 3 Hz with temperature 0.7 and a low-pass
filter (α = 0.35) on the command. All saliency analyses use **integrated gradients on the chosen
$w_z$** (8 Riemann steps).


## §0 — Runtime / GPU check

In [ ]:
# §0  Verify the GPU
!nvidia-smi
import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0),
          "| compute capability:", torch.cuda.get_device_capability(0))
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime → A100 GPU."

## §1 — Drive + headless rendering

Mounts Drive for persistence, then configures EGL/Xvfb headless rendering. The display setup
**must** run before the first `import genesis`.

In [ ]:
# §1a  Mount Google Drive + project layout
import os, pathlib
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = '/content/drive/MyDrive/quadruped_vlm_xai'
    COLAB = True
except Exception:
    DRIVE_ROOT = os.path.abspath('./quadruped_vlm_xai'); COLAB = False
    print('Not on Colab — using local DRIVE_ROOT:', DRIVE_ROOT)
for d in [DRIVE_ROOT] + [f'{DRIVE_ROOT}/{s}' for s in
                         ['config','logs','videos','trials','saliency']]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT, '| layout:', os.listdir(DRIVE_ROOT))

In [ ]:
# §1b  Headless rendering env vars + virtual display (run BEFORE importing genesis)
import os
for k, v in {'PYOPENGL_PLATFORM':'egl', 'MUJOCO_GL':'egl', 'EGL_PLATFORM':'surfaceless',
             'NVIDIA_DRIVER_CAPABILITIES':'compute,graphics,utility',
             'NVIDIA_VISIBLE_DEVICES':'all'}.items():
    os.environ[k] = v
try:
    from pyvirtualdisplay import Display
    _display = Display(visible=0, size=(1280, 720)); _display.start()
    os.environ['DISPLAY'] = f':{_display.display}'
    print('Xvfb DISPLAY =', os.environ['DISPLAY'])
except Exception as e:
    print('pyvirtualdisplay not ready yet (install in §2, then re-run this cell):', e)

## §2 — Dependencies

After this runs on a fresh runtime, re-run §1b so the virtual display starts.

In [ ]:
# §2  Install dependencies (idempotent). ~3-4 min on a fresh runtime.
import importlib.util
# is this package importable right now?
def _have(mod): return importlib.util.find_spec(mod) is not None
if not _have('genesis'):
    !apt-get update -qq
    !apt-get install -y -qq xvfb x11-utils libgl1 libglu1-mesa libegl1 libopengl0 \
        libglfw3 libglew2.2 libosmesa6 freeglut3-dev ffmpeg > /dev/null
    !pip install -q genesis-world 'rsl-rl-lib==2.2.4' tensorboard pyvirtualdisplay mediapy 'imageio[ffmpeg]'
if not _have('transformers') or not _have('qwen_vl_utils'):
    !pip install -q -U transformers accelerate qwen-vl-utils scipy scikit-image opencv-python matplotlib
import importlib.metadata as M
for pkg in ['genesis-world','rsl-rl-lib','torch','transformers']:
    try: print(f'{pkg:<15} {M.version(pkg)}')
    except M.PackageNotFoundError: print(f'{pkg:<15} NOT INSTALLED')
print('\n>>> If this was a fresh runtime, re-run §1b now so the virtual display starts.')

## §3 — ⚙️ CONFIG (edit everything here)

Single place to change the **simulation environment**, the VLM **contract prompt**, the **task
instruction**, and the **MEM memory knobs**. The old `HISTORY_MODE` switch is gone — memory is now
always the MEM hybrid (short-term video window + long-term compressed language memory).
`MEM_SHORT_FRAMES` / `MEM_SHORT_STRIDE` size the **base §7 planner's** window (used by the §10–§11
demos); the **V2 planner that drives all §12–§15 experiments** instead uses `CFG.MEM2` (4 recent
frames ~0.33 s apart, defined in §12a). `MEM_LANG_MAX_CHARS`, `MEM_AS_VIDEO`, and the two
`MEM_*_ENABLED` ablation toggles apply to **both** planners.

In [ ]:
# §3  Master configuration. Everything downstream reads from CFG.
from types import SimpleNamespace
CFG = SimpleNamespace()

# ---- paths ---------------------------------------------------------------
CFG.DRIVE_ROOT  = DRIVE_ROOT
CFG.CONFIG_DIR  = f'{DRIVE_ROOT}/config'
CFG.LOG_DIR     = f'{DRIVE_ROOT}/logs'
CFG.VIDEO_DIR   = f'{DRIVE_ROOT}/videos'
CFG.TRIAL_DIR   = f'{DRIVE_ROOT}/trials'
CFG.SALIENCY_DIR= f'{DRIVE_ROOT}/saliency'
CFG.GRID_TEXTURE= f'{CFG.CONFIG_DIR}/grid_floor.png'

# ---- PPO experiment ------------------------------------------------------
CFG.EXP_NAME  = 'go2_locomote'
CFG.PPO_ITERS = 600
CFG.NUM_ENVS  = 4096

# ============================== SIM ENVIRONMENT ===========================
CFG.SIM = SimpleNamespace(
    dt=0.02, episode_length_s=20.0, resampling_time_s=4.0, action_scale=0.25,
    kp=20.0, kd=0.5, base_init_pos=[0.0,0.0,0.42], base_init_quat=[1.0,0.0,0.0,0.0],
    termination_roll_deg=10, termination_pitch_deg=10,
)
CFG.ARENA = SimpleNamespace(distance=3.0, lateral=1.2, box_size=(0.4,0.4,0.5), commit_radius=0.7)
CFG.CAM = SimpleNamespace(res=(960,640), fov=55,
                          arena_pos=(-1.6,0.0,0.7), arena_lookat=(2.5,0.0,0.3))
CFG.CMD = SimpleNamespace(lin_vel_x_range=(-1.0,2.5), lin_vel_y_range=(-0.8,0.8), ang_vel_range=(-1.5,1.5))

# ============================== VLM PLANNER ===============================
CFG.QWEN_ID         = "Qwen/Qwen2.5-VL-3B-Instruct"
CFG.VLM_TEMPERATURE = 0.7
CFG.VLM_RATE_HZ     = 3
CFG.LOWPASS_ALPHA   = 0.35
CFG.VLM_MAX_NEW_TOKENS = 192      # larger: response now also carries the updated language memory

# ============================== MEM MEMORY (replaces history modes) =======
# SHORT-TERM visual memory: last N onboard frames as a native Qwen video input.
# NOTE: MEM_SHORT_FRAMES / MEM_SHORT_STRIDE only size the BASE planner (§7, used in the §10-§11
# demos). All §12-§15 experiments drive with the V2 planner, whose short-term window comes from
# CFG.MEM2 in §12a (4 recent frames, time-strided ~0.33 s apart). The remaining knobs below
# (as_video, language-memory cap, ablation toggles) apply to BOTH planners.
CFG.MEM_SHORT_FRAMES = 5         # base-planner window (current + N-1 references); MEM uses ~6
CFG.MEM_SHORT_STRIDE = 2         # base planner: keep every k-th queried frame -> longer horizon
CFG.MEM_AS_VIDEO     = True      # True: feed window as Qwen "video"; False: fall back to multi-image
# LONG-TERM language memory: running, self-rewritten, compressed summary.
CFG.MEM_LANG_MAX_CHARS = 600     # hard cap on the carried-over summary (keeps inference fast)
CFG.MEM_LANG_ENABLED   = True    # toggle the long-term language memory on/off (for ablation)
CFG.MEM_SHORT_ENABLED  = True    # toggle the short-term visual memory on/off (for ablation)

# ============================== CONTRACT PROMPT ===========================
# Output contract. The VLM emits a CONTINUOUS command AND rewrites its language memory each step.
# {mem_block} is filled at runtime with the short-term/long-term memory framing.
CFG.CONTRACT = (
    "You are the high-level planner for a quadruped robot, steering it from its onboard "
    "forward-facing camera. There is a BLUE target and a RED target ahead of you.\n"
    "\n"
    "# Your job\n"
    "Read the camera, decide which single target the instruction refers to, and drive to it "
    "decisively. Then update your memory.\n"
    "\n"
    "# Be decisive - avoid drifting (this is the main failure mode)\n"
    "- Choose ONE target as early as possible and COMMIT to it. Do not switch targets unless "
    "your chosen target is no longer visible.\n"
    "- Once committed, your 'choice' and 'memory' must stay consistent every step - do not "
    "flip-flop between the two targets.\n"
    "- Move with a clearly POSITIVE v_x toward your target every step until you reach it; "
    "prefer v_x >= 1.0 when the way is clear. Do not stop, hesitate, or reverse on the way in.\n"
    "\n"
    "# Inputs you are given\n"
    "(a) a short clip of your most recent onboard frames (short-term visual memory) and "
    "(b) a running text summary of what you have done so far (long-term memory).\n"
    "\n"
    "# Coordinate system (robot body frame, NOT image pixels)\n"
    "  v_x = forward speed m/s (%.1f..%.1f). POSITIVE drives FORWARD toward whatever is ahead "
    "in the image; NEGATIVE reverses. To APPROACH something you see, v_x MUST be POSITIVE.\n"
    "  v_y = lateral speed m/s (%.1f..%.1f). POSITIVE strafes to the robot LEFT (target on the "
    "LEFT of the frame); NEGATIVE strafes RIGHT.\n"
    "  w_z = yaw rad/s (%.1f..%.1f). POSITIVE turns LEFT (a target on the LEFT of the frame "
    "moves toward center); NEGATIVE turns RIGHT.\n"
    "\n"
    "# How to pick the signs (read the frame, then map)\n"
    "  - target centered & ahead  -> v_x=+1.5, v_y=0.0, w_z=0.0  (drive straight in)\n"
    "  - target on the LEFT half  -> v_x=+1.2, w_z=+0.5 (optionally v_y=+0.3) to curve left into it\n"
    "  - target on the RIGHT half -> v_x=+1.2, w_z=-0.5 (optionally v_y=-0.3) to curve right into it\n"
    "\n"
    "# Reasoning (think first, briefly)\n"
    "In 'reasoning', state in ONE short sentence: which target the instruction means, where it "
    "is in the frame, and which way you are therefore moving. Then make sure your numbers match it.\n"
    "\n"
    "# Memory rules\n"
    "Rewrite the FULL updated summary, keeping ONLY what is still useful to finish the task. "
    "COMPRESS: merge redundant events, drop abandoned attempts, never just append - replace. "
    "State your committed target and keep it stable. Keep it to a few short sentences.\n"
    "\n"
    "# Output format\n"
    "Respond with ONE JSON object and NOTHING else. Use real-valued numbers, not categories:\n"
    '{"reasoning": "<one short sentence>", "choice": "left|right|none", '
    '"v_x": <float>, "v_y": <float>, "w_z": <float>, "memory": "<updated long-term summary>"}'
) % (CFG.CMD.lin_vel_x_range[0], CFG.CMD.lin_vel_x_range[1],
     CFG.CMD.lin_vel_y_range[0], CFG.CMD.lin_vel_y_range[1],
     CFG.CMD.ang_vel_range[0],   CFG.CMD.ang_vel_range[1])

# ============================== TASK INSTRUCTION ==========================
CFG.NEUTRAL_PROMPT    = "Go to one of the objects to receive a reward."
CFG.AMBIGUOUS_PROMPTS = ["Go to the friendly one.", "Go to the hostile one."]
CFG.NAMED_PROMPTS     = [("Go to the RED object.", "left"), ("Go to the BLUE object.", "right")]
CFG.COLOR_RED  = (0.99, 0.05, 0.05)
CFG.COLOR_BLUE = (0.05, 0.05, 0.99)

print("CONFIG ready.")
print(f"MEM short-term: {CFG.MEM_SHORT_FRAMES} frames @ stride {CFG.MEM_SHORT_STRIDE} "
      f"(as_video={CFG.MEM_AS_VIDEO}) | long-term cap {CFG.MEM_LANG_MAX_CHARS} chars")

## §4 — `Go2Env` simulation environment

Unchanged from the consolidated build. Edit physics/rewards here.

In [ ]:
# §4  Write go2_env.py  (edit the simulation here)
%%writefile /content/go2_env.py
import math
import torch
import genesis as gs
from genesis.utils.geom import quat_to_xyz, transform_by_quat, inv_quat, transform_quat_by_quat

# uniform random float(s) in [lower, upper) on the sim device
def gs_rand_float(lower, upper, shape, device):
    return (upper - lower) * torch.rand(size=shape, device=device) + lower

class Go2Env:
    # build the Genesis scene, robot, camera and all per-env state buffers
    def __init__(self, num_envs, env_cfg, obs_cfg, reward_cfg, command_cfg,
                 show_viewer=False, attach_camera=False, camera_res=(960, 640),
                 camera_pos=(2.5, 1.0, 1.2), camera_lookat=(0.0, 0.0, 0.3), camera_fov=40,
                 env_spacing=None, rendered_envs_idx=None, hiq=False, grid_texture=None,
                 extra_objects=None):
        self.num_envs = num_envs
        self.num_obs = obs_cfg["num_obs"]
        self.num_privileged_obs = None
        self.num_actions = env_cfg["num_actions"]
        self.num_commands = command_cfg["num_commands"]
        self.device = gs.device

        self.simulate_action_latency = True
        self.dt = env_cfg.get("dt", 0.02)  # 50 Hz control
        self.max_episode_length = math.ceil(env_cfg["episode_length_s"] / self.dt)

        self.env_cfg = env_cfg
        self.obs_cfg = obs_cfg
        self.reward_cfg = reward_cfg
        self.command_cfg = command_cfg
        self.obs_scales = obs_cfg["obs_scales"]
        self.reward_scales = reward_cfg["reward_scales"]

        if rendered_envs_idx is None:
            rendered_envs_idx = [0]

        # ---- scene (optionally high-quality: lights + shadows)
        if hiq:
            import sys; sys.path.insert(0, '/content')
            import qre_utils as _U
            self.scene, _ = _U.build_hiq_scene(gs, dt=self.dt, rendered_envs_idx=rendered_envs_idx)
        else:
            self.scene = gs.Scene(
                sim_options=gs.options.SimOptions(dt=self.dt, substeps=2),
                viewer_options=gs.options.ViewerOptions(
                    max_FPS=int(0.5 / self.dt),
                    camera_pos=(2.0, 0.0, 2.5), camera_lookat=(0.0, 0.0, 0.5), camera_fov=40),
                vis_options=gs.options.VisOptions(rendered_envs_idx=rendered_envs_idx),
                rigid_options=gs.options.RigidOptions(
                    dt=self.dt, constraint_solver=gs.constraint_solver.Newton,
                    enable_collision=True, enable_joint_limit=True),
                show_viewer=show_viewer,
            )

        # ---- ground (grid floor when hiq) + optional goal objects + robot
        if hiq and grid_texture:
            import sys; sys.path.insert(0, '/content')
            import qre_utils as _U
            _U.add_ground(gs, self.scene, grid_texture)
        else:
            self.scene.add_entity(gs.morphs.URDF(file="urdf/plane/plane.urdf", fixed=True))
        if extra_objects:
            import sys; sys.path.insert(0, '/content')
            import qre_utils as _U
            for ob in extra_objects:
                _U.add_colored_box(gs, self.scene, ob['pos'], ob['size'], ob['color'])
        self.base_init_pos = torch.tensor(env_cfg["base_init_pos"], device=gs.device)
        self.base_init_quat = torch.tensor(env_cfg["base_init_quat"], device=gs.device)
        self.inv_base_init_quat = inv_quat(self.base_init_quat)
        self.robot = self.scene.add_entity(gs.morphs.URDF(
            file="urdf/go2/urdf/go2.urdf",
            pos=self.base_init_pos.cpu().numpy(),
            quat=self.base_init_quat.cpu().numpy()))

        # ---- optional recording camera
        self.camera = None
        if attach_camera:
            self.camera = self.scene.add_camera(
                res=camera_res, pos=camera_pos, lookat=camera_lookat, fov=camera_fov, GUI=False)

        # ---- build
        build_kwargs = {"n_envs": num_envs}
        if env_spacing is not None:
            build_kwargs["env_spacing"] = env_spacing
        self.scene.build(**build_kwargs)

        self.motors_dof_idx = [self.robot.get_joint(n).dof_start for n in env_cfg["joint_names"]]
        self.robot.set_dofs_kp([env_cfg["kp"]] * self.num_actions, self.motors_dof_idx)
        self.robot.set_dofs_kv([env_cfg["kd"]] * self.num_actions, self.motors_dof_idx)

        self.reward_functions, self.episode_sums = {}, {}
        for name in self.reward_scales.keys():
            self.reward_scales[name] *= self.dt
            self.reward_functions[name] = getattr(self, "_reward_" + name)
            self.episode_sums[name] = torch.zeros((self.num_envs,), device=gs.device, dtype=gs.tc_float)

        self.base_lin_vel = torch.zeros((self.num_envs, 3), device=gs.device, dtype=gs.tc_float)
        self.base_ang_vel = torch.zeros((self.num_envs, 3), device=gs.device, dtype=gs.tc_float)
        self.projected_gravity = torch.zeros((self.num_envs, 3), device=gs.device, dtype=gs.tc_float)
        self.global_gravity = torch.tensor([0.0, 0.0, -1.0], device=gs.device, dtype=gs.tc_float).repeat(self.num_envs, 1)
        self.obs_buf = torch.zeros((self.num_envs, self.num_obs), device=gs.device, dtype=gs.tc_float)
        self.rew_buf = torch.zeros((self.num_envs,), device=gs.device, dtype=gs.tc_float)
        self.reset_buf = torch.ones((self.num_envs,), device=gs.device, dtype=gs.tc_int)
        self.episode_length_buf = torch.zeros((self.num_envs,), device=gs.device, dtype=gs.tc_int)
        self.commands = torch.zeros((self.num_envs, self.num_commands), device=gs.device, dtype=gs.tc_float)
        self.commands_scale = torch.tensor(
            [self.obs_scales["lin_vel"], self.obs_scales["lin_vel"], self.obs_scales["ang_vel"]],
            device=gs.device, dtype=gs.tc_float)
        self.actions = torch.zeros((self.num_envs, self.num_actions), device=gs.device, dtype=gs.tc_float)
        self.last_actions = torch.zeros_like(self.actions)
        self.dof_pos = torch.zeros_like(self.actions)
        self.dof_vel = torch.zeros_like(self.actions)
        self.last_dof_vel = torch.zeros_like(self.actions)
        self.base_pos = torch.zeros((self.num_envs, 3), device=gs.device, dtype=gs.tc_float)
        self.base_quat = torch.zeros((self.num_envs, 4), device=gs.device, dtype=gs.tc_float)
        self.default_dof_pos = torch.tensor(
            [env_cfg["default_joint_angles"][n] for n in env_cfg["joint_names"]],
            device=gs.device, dtype=gs.tc_float)
        self.extras = {"observations": {}}

    # draw new random velocity commands for the given env indices
    def _resample_commands(self, envs_idx):
        self.commands[envs_idx, 0] = gs_rand_float(*self.command_cfg["lin_vel_x_range"], (len(envs_idx),), gs.device)
        self.commands[envs_idx, 1] = gs_rand_float(*self.command_cfg["lin_vel_y_range"], (len(envs_idx),), gs.device)
        self.commands[envs_idx, 2] = gs_rand_float(*self.command_cfg["ang_vel_range"], (len(envs_idx),), gs.device)

    # advance one control step: apply actions, step physics, compute obs/reward/reset
    def step(self, actions):
        self.actions = torch.clip(actions, -self.env_cfg["clip_actions"], self.env_cfg["clip_actions"])
        exec_actions = self.last_actions if self.simulate_action_latency else self.actions
        target_dof_pos = exec_actions * self.env_cfg["action_scale"] + self.default_dof_pos
        self.robot.control_dofs_position(target_dof_pos, self.motors_dof_idx)
        self.scene.step()

        self.episode_length_buf += 1
        self.base_pos[:] = self.robot.get_pos()
        self.base_quat[:] = self.robot.get_quat()
        self.base_euler = quat_to_xyz(
            transform_quat_by_quat(torch.ones_like(self.base_quat) * self.inv_base_init_quat, self.base_quat),
            rpy=True, degrees=True)
        inv_base_quat = inv_quat(self.base_quat)
        self.base_lin_vel[:] = transform_by_quat(self.robot.get_vel(), inv_base_quat)
        self.base_ang_vel[:] = transform_by_quat(self.robot.get_ang(), inv_base_quat)
        self.projected_gravity = transform_by_quat(self.global_gravity, inv_base_quat)
        self.dof_pos[:] = self.robot.get_dofs_position(self.motors_dof_idx)
        self.dof_vel[:] = self.robot.get_dofs_velocity(self.motors_dof_idx)

        envs_idx = ((self.episode_length_buf % int(self.env_cfg["resampling_time_s"] / self.dt) == 0)
                    .nonzero(as_tuple=False).reshape((-1,)))
        self._resample_commands(envs_idx)

        self.reset_buf = self.episode_length_buf > self.max_episode_length
        self.reset_buf |= torch.abs(self.base_euler[:, 1]) > self.env_cfg["termination_if_pitch_greater_than"]
        self.reset_buf |= torch.abs(self.base_euler[:, 0]) > self.env_cfg["termination_if_roll_greater_than"]

        time_out_idx = (self.episode_length_buf > self.max_episode_length).nonzero(as_tuple=False).reshape((-1,))
        self.extras["time_outs"] = torch.zeros_like(self.reset_buf, device=gs.device, dtype=gs.tc_float)
        self.extras["time_outs"][time_out_idx] = 1.0
        self.reset_idx(self.reset_buf.nonzero(as_tuple=False).reshape((-1,)))

        self.rew_buf[:] = 0.0
        for name, fn in self.reward_functions.items():
            rew = fn() * self.reward_scales[name]
            self.rew_buf += rew
            self.episode_sums[name] += rew

        self.obs_buf = torch.cat([
            self.base_ang_vel * self.obs_scales["ang_vel"],
            self.projected_gravity,
            self.commands * self.commands_scale,
            (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],
            self.dof_vel * self.obs_scales["dof_vel"],
            self.actions,
        ], dim=-1)

        self.last_actions[:] = self.actions[:]
        self.last_dof_vel[:] = self.dof_vel[:]
        self.extras["observations"]["critic"] = self.obs_buf
        return self.obs_buf, self.rew_buf, self.reset_buf, self.extras

    # current observation buffer (rsl-rl runner interface)
    def get_observations(self):
        self.extras["observations"]["critic"] = self.obs_buf
        return self.obs_buf, self.extras

    # no privileged (asymmetric actor/critic) observations used here
    def get_privileged_observations(self):
        return None

    # reset the given envs back to their initial pose and zero their episode stats
    def reset_idx(self, envs_idx):
        if len(envs_idx) == 0:
            return
        self.dof_pos[envs_idx] = self.default_dof_pos
        self.dof_vel[envs_idx] = 0.0
        self.robot.set_dofs_position(position=self.dof_pos[envs_idx], dofs_idx_local=self.motors_dof_idx,
                                     zero_velocity=True, envs_idx=envs_idx)
        self.base_pos[envs_idx] = self.base_init_pos
        self.base_quat[envs_idx] = self.base_init_quat.reshape(1, -1)
        self.robot.set_pos(self.base_pos[envs_idx], zero_velocity=False, envs_idx=envs_idx)
        self.robot.set_quat(self.base_quat[envs_idx], zero_velocity=False, envs_idx=envs_idx)
        self.base_lin_vel[envs_idx] = 0
        self.base_ang_vel[envs_idx] = 0
        self.robot.zero_all_dofs_velocity(envs_idx)
        self.last_actions[envs_idx] = 0.0
        self.last_dof_vel[envs_idx] = 0.0
        self.episode_length_buf[envs_idx] = 0
        self.reset_buf[envs_idx] = True
        self.extras["episode"] = {}
        for k in self.episode_sums.keys():
            self.extras["episode"]["rew_" + k] = (
                torch.mean(self.episode_sums[k][envs_idx]).item() / self.env_cfg["episode_length_s"])
            self.episode_sums[k][envs_idx] = 0.0
        self._resample_commands(envs_idx)

    # reset every env
    def reset(self):
        self.reset_buf[:] = True
        self.reset_idx(torch.arange(self.num_envs, device=gs.device))
        return self.obs_buf, None

    # ---- rewards
    def _reward_tracking_lin_vel(self):
        err = torch.sum(torch.square(self.commands[:, :2] - self.base_lin_vel[:, :2]), dim=1)
        return torch.exp(-err / self.reward_cfg["tracking_sigma"])

    # reward for matching the commanded yaw rate
    def _reward_tracking_ang_vel(self):
        err = torch.square(self.commands[:, 2] - self.base_ang_vel[:, 2])
        return torch.exp(-err / self.reward_cfg["tracking_sigma"])

    # penalize vertical (bouncing) velocity
    def _reward_lin_vel_z(self):
        return torch.square(self.base_lin_vel[:, 2])

    # penalize large action changes between consecutive steps
    def _reward_action_rate(self):
        return torch.sum(torch.square(self.last_actions - self.actions), dim=1)

    # penalize joint pose drifting from the default stance
    def _reward_similar_to_default(self):
        return torch.sum(torch.abs(self.dof_pos - self.default_dof_pos), dim=1)

    # penalize base height deviating from the target height
    def _reward_base_height(self):
        return torch.square(self.base_pos[:, 2] - self.reward_cfg["base_height_target"])

## §5 — Shared utilities (`qre_utils.py`)

Unchanged: hi-q render profile, sanity gate, camera geometry, arena builder, luminance matching.

In [ ]:
# §5  Write qre_utils.py
%%writefile /content/qre_utils.py
import os, math
import numpy as np

# ----------------------------------------------------------------------------------
# High-quality render profile
def make_grid_texture(path, size_px=2048, n_cells=8, line_px=2,
                      bg=(28, 28, 32), line=(96, 96, 104)):
    """Large dark-grey tiles with very faint, thin white grid lines (diffuse PNG).
    bg     : base tile colour (dark grey, near-black).
    n_cells: cells per texture edge -> FEWER = LARGER tiles.
    line_px: line thickness in px  -> keep tiny (1-2) for thin lines.
    line   : line colour -> faint grey, NOT pure white, so lines stay subtle."""
    import imageio.v2 as imageio
    img = np.empty((size_px, size_px, 3), dtype=np.uint8)
    img[:] = np.array(bg, dtype=np.uint8)            # dark grey base
    step = size_px // n_cells                        # large tiles -> few cells
    line_rgb = np.array(line, dtype=np.uint8)        # faint, not pure white
    for i in range(0, size_px + 1, step):
        lo = max(0, i - line_px // 2); hi = min(size_px, i + line_px // 2 + 1)
        img[lo:hi, :, :] = line_rgb                  # thin horizontal line
        img[:, lo:hi, :] = line_rgb                  # thin vertical line
    imageio.imwrite(path, img)
    return path

def build_hiq_scene(gs, dt=0.02, rendered_envs_idx=(0,)):
    """gs.Scene with lights + shadows when supported, else a safe fallback. Returns (scene, used_hiq)."""
    rigid = gs.options.RigidOptions(dt=dt, constraint_solver=gs.constraint_solver.Newton,
                                    enable_collision=True, enable_joint_limit=True)
    sim = gs.options.SimOptions(dt=dt, substeps=2)
    vis_attempts = [
        dict(rendered_envs_idx=list(rendered_envs_idx), shadow=True,
             ambient_light=(0.25, 0.25, 0.25),
             lights=[{"type": "directional", "dir": (-0.5, -0.6, -1.0),
                      "color": (1.0, 1.0, 1.0), "intensity": 6.0}]),
        dict(rendered_envs_idx=list(rendered_envs_idx), shadow=True),
        dict(rendered_envs_idx=list(rendered_envs_idx)),
    ]
    for i, kw in enumerate(vis_attempts):
        try:
            scene = gs.Scene(sim_options=sim, vis_options=gs.options.VisOptions(**kw),
                             rigid_options=rigid, show_viewer=False)
            return scene, (i == 0)
        except TypeError:
            continue
    scene = gs.Scene(sim_options=sim, rigid_options=rigid, show_viewer=False)
    return scene, False

def add_ground(gs, scene, grid_texture_path=None):
    """Add the floor — grid-textured surface if available, else a plain dark plane."""
    morph = gs.morphs.URDF(file="urdf/plane/plane.urdf", fixed=True)
    if grid_texture_path and os.path.exists(grid_texture_path):
        for surf in (
            lambda: gs.surfaces.Default(diffuse_texture=gs.textures.ImageTexture(image_path=grid_texture_path)),
            lambda: gs.surfaces.Rough(diffuse_texture=gs.textures.ImageTexture(image_path=grid_texture_path)),
        ):
            try:
                return scene.add_entity(morph, surface=surf())
            except Exception:
                continue
    try:
        return scene.add_entity(morph, surface=gs.surfaces.Default(color=(0.05, 0.05, 0.05)))
    except Exception:
        return scene.add_entity(morph)

def add_colored_box(gs, scene, pos, size, color):
    """Add a fixed colored rectangle (a candidate goal object)."""
    morph = gs.morphs.Box(pos=tuple(pos), size=tuple(size), fixed=True)
    for surf in (lambda: gs.surfaces.Default(color=tuple(color)),
                 lambda: gs.surfaces.Rough(color=tuple(color))):
        try:
            return scene.add_entity(morph, surface=surf())
        except Exception:
            continue
    return scene.add_entity(morph)

# ----------------------------------------------------------------------------------
# Camera geometry for saliency projection
def camera_intrinsics(res, fov_deg):
    """Pinhole K from fov + resolution. Genesis fov is vertical; adjust if NB0 shows otherwise."""
    w, h = res
    fov = math.radians(fov_deg)
    fy = (h / 2) / math.tan(fov / 2)
    fx = fy  # square pixels
    cx, cy = w / 2.0, h / 2.0
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)

def look_at_extrinsics(cam_pos, lookat, up=(0, 0, 1)):
    """World->camera T_cw and camera->world T_wc (OpenCV convention, +z forward)."""
    cam_pos = np.asarray(cam_pos, float); lookat = np.asarray(lookat, float); up = np.asarray(up, float)
    f = lookat - cam_pos; f /= (np.linalg.norm(f) + 1e-9)
    r = np.cross(f, up); r /= (np.linalg.norm(r) + 1e-9)
    u = np.cross(r, f)
    R_wc = np.stack([r, -u, f], axis=1)
    T_wc = np.eye(4); T_wc[:3, :3] = R_wc; T_wc[:3, 3] = cam_pos
    T_cw = np.linalg.inv(T_wc)
    return T_cw, T_wc

def unproject(u, v, depth, K, T_wc):
    """Pixel (u,v) + metric depth -> world XYZ (depth along camera +z)."""
    x = (u - K[0, 2]) / K[0, 0] * depth
    y = (v - K[1, 2]) / K[1, 1] * depth
    p_cam = np.array([x, y, depth, 1.0])
    return (T_wc @ p_cam)[:3]

# ----------------------------------------------------------------------------------
# Render-and-assert sanity gate
def render_camera(camera, want_depth=True):
    """camera.render -> (rgb_uint8, depth_or_None). Genesis returns (rgb, depth, seg, normal)."""
    out = None
    for kwargs in ({"rgb": True, "depth": want_depth}, {}):
        try:
            out = camera.render(**kwargs); break
        except TypeError:
            continue
    if not isinstance(out, (tuple, list)):
        return np.asarray(out), None
    rgb = np.asarray(out[0]); depth = None
    if want_depth and len(out) > 1 and out[1] is not None:
        depth = np.asarray(out[1])
    return rgb, depth

def sanity_gate(scene, robot, camera, save_png, n_steps=8, bound=5.0, z_max=2.0):
    """Step with tiny random actions; assert physical sanity + a usable RGB(+depth) frame."""
    import imageio.v2 as imageio
    for _ in range(n_steps):
        scene.step()
    pos = np.asarray(robot.get_pos().cpu()) if hasattr(robot.get_pos(), "cpu") else np.asarray(robot.get_pos())
    pos = pos.reshape(-1, 3)[0]
    assert np.all(np.isfinite(pos)), f"NaN/Inf base pos {pos}"
    assert abs(pos[0]) < bound and abs(pos[1]) < bound, f"Robot launched away: {pos}"
    assert 0 < pos[2] < z_max, f"Bad base height: {pos[2]}"
    rgb, depth = render_camera(camera, want_depth=True)
    assert rgb is not None and rgb.size > 0 and rgb.std() > 1.0, "Degenerate RGB frame"
    imageio.imwrite(save_png, rgb[..., :3].astype(np.uint8))
    depth_ok = depth is not None and np.isfinite(depth).any() and float(np.nanstd(depth)) > 1e-4
    return dict(base_pos=pos.tolist(), rgb_shape=tuple(rgb.shape),
                depth_present=depth is not None, depth_nondegenerate=bool(depth_ok), png=save_png)

# ----------------------------------------------------------------------------------
# Two-goal bias arena (red vs. blue, luminance-matched)
def luminance(c):
    r, g, b = c
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def luminance_match(ref, other):
    """Blend `other` toward white to match ref luminance (keeps hue dominant)."""
    Lref, Loth = luminance(ref), luminance(other)
    if Loth >= Lref:
        return other
    g = (Lref - Loth)
    return tuple(min(1.0, c + g) for c in other)

def goal_positions(distance=3.0, lateral=1.2, box_size=(0.4, 0.4, 0.5),
                   left_color=None, right_color=None):
    """Two rectangles, equal forward distance, symmetric left/right. Robot starts at origin facing +x."""
    left  = dict(pos=(distance, +lateral, box_size[2] / 2), size=box_size, color=left_color,  side="left")
    right = dict(pos=(distance, -lateral, box_size[2] / 2), size=box_size, color=right_color, side="right")
    return left, right

## §6 — Feasibility analysis: MEM → this project

**What MEM is** (Physical Intelligence, *Multi-Scale Embodied Memory*). MEM factorizes a VLA policy
into a high-level and a low-level policy and gives it memory at two time scales:

$$\pi(a_{t:t+H},\, l_{t+1},\, m_{t+1} \mid o_{t-T:t},\, m_t,\, g) \approx
\underbrace{\pi_{LL}(a_{t:t+H}\mid o_{t-K:t},\, l_{t+1},\, g)}_{\text{fast controller, short video window}}\;
\underbrace{\pi_{HL}(l_{t+1},\, m_{t+1}\mid o_t,\, m_t,\, g)}_{\text{slow planner, rewrites language memory}}$$

* **Short-term visual memory** — a dense window of the last $K$ frames, compressed by a *custom video
  encoder* (interleaved spatial + causal-temporal attention every 4th ViT layer; past-timestep tokens
  dropped in upper layers so token count stays at single-frame cost). Pretrained with 6 frames at 1 s
  stride, post-trained to 18 frames / 54 s. Resolves occlusion and enables in-context adaptation.
* **Long-term language memory** — a running natural-language summary $m_t$ that the high-level policy
  **rewrites** each high-level step ($m_t \to m_{t+1}$), trained from an LLM-generated summarization
  dataset that **compresses**: drop failed attempts, merge redundant events. The paper shows naive
  *append-all* text memory is markedly worse, due to train/inference distribution shift.

**Can we reproduce MEM faithfully here? No — and here's exactly why:**

1. *The video encoder needs architecture surgery + training.* MEM modifies the ViT attention pattern
   and trains the encoder on a large robot+video corpus. This project drives a **frozen, off-the-shelf
   Qwen2.5-VL-3B** as a planner; we cannot retrain its vision tower on Colab.
2. *The recurrent language-memory summarizer is trained.* MEM trains $\pi_{HL}$ to emit $m_{t+1}$ from
   labels produced by an offline LLM-summarization pipeline. We have no such dataset and no finetuning
   budget here.
3. *The high/low-level action split is already trained jointly in MEM (flow-matching action expert).*
   Our control stack is the opposite shape: a **separately trained PPO controller** consumes a
   continuous velocity command. We are not learning actions end-to-end with the VLM.

**Therefore we adapt MEM into a hybrid that keeps its two-scale structure but works with a frozen VLM**
— which is exactly the fallback the task describes:

| MEM component | Faithful MEM | This project's adaptation |
|---|---|---|
| High/low-level split | trained $\pi_{HL},\pi_{LL}$ | **already present**: VLM = slow planner (1–5 Hz), PPO = fast controller (50 Hz) |
| Short-term visual memory | trained video encoder, $K$ frames | last $K$ onboard frames fed as a **native Qwen video input** (gets temporal position encoding); token cost grows, so $K$ is small |
| Long-term language memory | trained recurrent summarizer w/ compression | the **same frozen VLM rewrites a compressed summary in-context each step**, prompted with MEM's compression rules (drop failures, merge redundant, replace-not-append) |
| Memory update trigger | every high-level step | every VLM re-query (`VLM_RATE_HZ`) |

**What we keep from MEM's findings:** the multi-modal split (dense recent frames for local
dynamics/occlusion; compressed language for long-horizon semantics) and, critically, the
**compression discipline** on the language memory — the paper's ablation shows this is what makes
language memory work rather than hurt. Both memories are **toggleable** (`MEM_SHORT_ENABLED`,
`MEM_LANG_ENABLED`) so you can reproduce MEM's ablation (no-memory / only-video / only-text / both).

**Honest limitations of the adaptation:** (1) feeding frames as Qwen "video" is *not* the compute
trade-off MEM achieves — our token count rises with $K$, so the short-term window must stay small to
hold real-time-ish latency; (2) the language memory is produced **in-context by a frozen model**, so
it lacks MEM's trained robustness to distribution shift (we lean on the compression prompt to
mitigate this); (3) there is no gradient coupling between the memory and a learned action expert —
the PPO controller never sees the memory, only the resulting continuous command.

## §7 — MEM hybrid-memory planner (`mem_planner.py`)

`MEMPlanner.act(frame, instruction)` does one slow-planner step:

1. Push `frame` into the **short-term ring buffer** (strided by `MEM_SHORT_STRIDE`).
2. Build a prompt containing the short-term window (as a Qwen *video*, with a multi-image fallback)
   and the current **long-term language memory** string.
3. Generate `{reasoning, choice, v_x, v_y, w_z, memory}`.
4. Parse the continuous command (clamped to the trained ranges) and **replace** the long-term memory
   with the model's compressed `memory` field (truncated to `MEM_LANG_MAX_CHARS`).

The planner also exposes a lightweight attention diagnostic anchored to the `v_x` numeric token
over the **most recent** frame's patches (`want_attention=True`); the actual §13–§15 saliency
analyses instead use **integrated gradients on the chosen `w_z`**. The two memory streams are
independently toggleable for MEM-style ablations.

§12a subclasses this into `MEMPlannerV2` (time-strided recent frames from `CFG.MEM2`: 4 frames
~0.33 s apart, no anchor) — the planner that drives every experiment from §12 onward.

In [ ]:
# §7  Write mem_planner.py
%%writefile /content/mem_planner.py
import json, re, collections
import numpy as np

# load the Qwen2.5-VL model + processor onto the given device
def load_qwen(qwen_id, device="cuda", dtype=None):
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    if dtype is None:
        dtype = torch.bfloat16
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        qwen_id, torch_dtype=dtype, attn_implementation="eager", device_map=device)
    model.eval()
    processor = AutoProcessor.from_pretrained(qwen_id)
    return model, processor

# ----------------------------------------------------------------------------------
_FLOAT = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
def _clip(x, lo, hi): return float(max(lo, min(hi, x)))

def parse_continuous(raw, vx_range, vy_range, wz_range):
    """Extract {v_x,v_y,w_z, choice, reasoning, memory}. Always returns a usable command."""
    txt = raw.strip()
    txt = re.sub(r"^```(json)?|```$", "", txt, flags=re.MULTILINE).strip()
    status = "ok"; obj = {}
    m = re.search(r"\{.*\}", txt, flags=re.DOTALL)
    if m:
        try: obj = json.loads(m.group(0))
        except Exception: status = "json_error"
    else:
        status = "no_json"

    # pull a float field out of the parsed JSON response object
    def getf(key):
        try: return float(obj[key])
        except Exception: return None
    vx, vy, wz = getf("v_x"), getf("v_y"), getf("w_z")
    got_all = (status == "ok" and None not in (vx, vy, wz))

    if not got_all:
        # regex fallback: pull a named float straight out of the raw text
        def rx(name):
            mm = re.search(rf'{name}"?\s*[:=]\s*({_FLOAT})', txt, flags=re.IGNORECASE)
            return float(mm.group(1)) if mm else None
        rvx = vx if vx is not None else rx("v_x")
        rvy = vy if vy is not None else rx("v_y")
        rwz = wz if wz is not None else rx("w_z")
        if None not in (rvx, rvy, rwz):
            vx, vy, wz = rvx, rvy, rwz; status = "regex_recovered"
        else:
            tup = re.search(rf"\(\s*({_FLOAT})\s*,\s*({_FLOAT})\s*,\s*({_FLOAT})\s*\)", txt)
            if tup:
                vx, vy, wz = (float(tup.group(i)) for i in (1, 2, 3)); status = "tuple_recovered"

    if vx is None or vy is None or wz is None:
        return dict(parse_status="parse_failed", raw=raw, v_x=0.0, v_y=0.0, w_z=0.0,
                    choice=None, reasoning=None, memory=None)
    return dict(parse_status=status, raw=raw,
                v_x=_clip(vx, *vx_range), v_y=_clip(vy, *vy_range), w_z=_clip(wz, *wz_range),
                choice=obj.get("choice"), reasoning=obj.get("reasoning"), memory=obj.get("memory"))

# ----------------------------------------------------------------------------------
def _vision_token_span(input_ids, model):
    ids = input_ids[0].tolist()
    img_id = getattr(model.config, "image_token_id", None)
    if img_id is None:
        img_id = getattr(getattr(model, "config", None), "image_token_index", None)
    vid_id = getattr(model.config, "video_token_id", None)
    cand = [i for i, t in enumerate(ids) if t == img_id or (vid_id is not None and t == vid_id)]
    if not cand:
        return None
    return cand[0], cand[-1] + 1

def vx_token_attention(model, processor, inputs, out, gen_ids, grid_thw):
    """Attention from the v_x numeric token onto the LAST frame's patches. (grid2d, step)."""
    import torch
    dec = [processor.tokenizer.decode([t]) for t in gen_ids.tolist()]
    anchor = None
    for i, tok in enumerate(dec):
        if any(ch.isdigit() for ch in tok):
            ctx = "".join(dec[max(0, i - 8):i]).lower().replace("_", "")
            if "vx" in ctx:
                anchor = i; break
    if anchor is None:
        for i, tok in enumerate(dec):
            if any(ch.isdigit() for ch in tok): anchor = i; break
    if anchor is None:
        raise RuntimeError("no numeric token for v_x anchor")
    span = _vision_token_span(inputs.input_ids, model)
    if span is None:
        raise RuntimeError("vision token span not found")
    v0, v1 = span
    step_attn = out.attentions[anchor][-1]
    a = step_attn[0].mean(0)[-1]
    vis = a[v0:v1].float().cpu().numpy()
    # grid_thw = [t, h, w] (t frames); take the LAST frame's merged-patch block.
    t, h, w = [int(x) for x in grid_thw]
    merged_h, merged_w = h // 2, w // 2
    per = merged_h * merged_w
    vis = vis[-per:] if vis.shape[0] >= per else np.pad(vis, (0, per - vis.shape[0]))
    grid = vis.reshape(merged_h, merged_w)
    grid = (grid - grid.min()) / (np.ptp(grid) + 1e-9)
    return grid, int(anchor)

# ----------------------------------------------------------------------------------
class MEMPlanner:
    """MEM-style hybrid memory planner for a frozen Qwen2.5-VL.

    Short-term visual memory : last `short_frames` onboard frames (strided), fed as a native
                               Qwen video input (multi-image fallback).
    Long-term language memory: a running summary the model rewrites + compresses each step.
    Both are independently toggleable (short_enabled / lang_enabled) for MEM-style ablation.
    """
    # store planner config and initialize the memory buffers
    def __init__(self, model, processor, contract, vx_range, vy_range, wz_range,
                 short_frames=5, short_stride=2, as_video=True,
                 lang_max_chars=600, short_enabled=True, lang_enabled=True,
                 max_new_tokens=192):
        self.model = model; self.processor = processor; self.contract = contract
        self.vx_range, self.vy_range, self.wz_range = vx_range, vy_range, wz_range
        self.short_frames = short_frames; self.short_stride = short_stride
        self.as_video = as_video; self.lang_max_chars = lang_max_chars
        self.short_enabled = short_enabled; self.lang_enabled = lang_enabled
        self.max_new_tokens = max_new_tokens
        self.reset()

    # clear the short-term frame buffer and the long-term memory string
    def reset(self):
        self._frames = collections.deque(maxlen=max(1, self.short_frames))
        self._language_memory = ""      # m_0 : empty
        self._step = 0

    @property
    # current long-term memory string (read-only)
    def language_memory(self):
        return self._language_memory

    # ---- prompt assembly ----------------------------------------------------------
    def _build_messages(self, frame, instruction):
        # update short-term ring buffer (strided)
        if self._step % max(1, self.short_stride) == 0 or len(self._frames) == 0:
            self._frames.append(frame)
        else:
            # always keep the latest frame as the most recent entry
            self._frames[-1] = frame

        content = []
        window = list(self._frames)
        used_video = False
        if self.short_enabled and len(window) > 1:
            if self.as_video:
                # native Qwen video input: a list of frames gets temporal position encoding
                content.append({"type": "video", "video": window})
                used_video = True
            else:
                for j, fr in enumerate(window):
                    tag = "now" if j == len(window) - 1 else f"t-{len(window)-1-j}"
                    content.append({"type": "text", "text": f"[frame {tag}]"})
                    content.append({"type": "image", "image": fr})
        else:
            content.append({"type": "image", "image": frame})

        mem_block = ""
        if self.short_enabled and len(window) > 1:
            mem_block += (f"\nShort-term visual memory: the clip above is your last {len(window)} "
                          "onboard frames (oldest to newest).")
        if self.lang_enabled:
            mem_block += ("\nLong-term memory so far: "
                          + (self._language_memory if self._language_memory else "(empty)"))

        content.append({"type": "text",
                        "text": f"{self.contract}{mem_block}\n\nInstruction: {instruction}"})
        return [{"role": "user", "content": content}], used_video

    # ---- one slow-planner step ----------------------------------------------------
    def act(self, frame, instruction, temperature=0.7, seed=0, want_attention=False):
        import torch
        from qwen_vl_utils import process_vision_info
        torch.manual_seed(seed)

        messages, used_video = self._build_messages(frame, instruction)
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        try:
            image_inputs, video_inputs = process_vision_info(messages)
        except Exception:
            # robust fallback: re-build as multi-image if this qwen_vl_utils rejects PIL-list video
            self.as_video = False
            messages, used_video = self._build_messages(frame, instruction)
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt").to(self.model.device)
        gen_kwargs = dict(max_new_tokens=self.max_new_tokens, do_sample=temperature > 0,
                          temperature=max(1e-4, temperature),
                          return_dict_in_generate=True, output_attentions=want_attention)
        with torch.no_grad():
            out = self.model.generate(**inputs, **gen_kwargs)
        seq = out.sequences[0]; prompt_len = inputs.input_ids.shape[1]
        gen_ids = seq[prompt_len:]
        raw = self.processor.decode(gen_ids, skip_special_tokens=True)
        parsed = parse_continuous(raw, self.vx_range, self.vy_range, self.wz_range)

        # ---- update long-term language memory (REPLACE, not append) ----
        if self.lang_enabled:
            new_mem = parsed.get("memory")
            if new_mem:
                self._language_memory = str(new_mem)[: self.lang_max_chars]

        # ---- attention for saliency ----
        attn2d = None; grid_thw = None; anchor_step = None
        if want_attention and getattr(out, "attentions", None) is not None:
            try:
                key = "image_grid_thw" if "image_grid_thw" in inputs else "video_grid_thw"
                grid_thw = inputs[key][0].tolist()
                attn2d, anchor_step = vx_token_attention(self.model, self.processor,
                                                         inputs, out, gen_ids, grid_thw)
            except Exception as e:
                parsed["attn_error"] = repr(e)

        self._step += 1
        parsed.update(dict(prompt_len=int(prompt_len), gen_text=raw, gen_ids=gen_ids.tolist(),
                           used_video=used_video,
                           language_memory=self._language_memory, n_short_frames=len(self._frames),
                           attn2d=attn2d, grid_thw=grid_thw, anchor_step=anchor_step,
                           temperature=temperature, seed=seed))
        return parsed

## §8 — Post-hoc semantic bucketing (for saliency / IG)

Unchanged. Maps the continuous `(v_x,v_y,w_z)` to one of eight compass concepts (`forward`,
`forward_right`, …) plus in-place `turn_left/turn_right/stop`. Control stays continuous; this is
only for grouping saliency / IG results.

In [ ]:
# §8  Post-hoc semantic bucketing. NOTE: this NEVER feeds back into the control command.
import numpy as np

# 8 sectors centered on these headings (radians), robot body frame (+x fwd, +y left).
SEMANTIC_SECTORS = [
    ("forward",         0.0),
    ("forward_left",    np.pi / 4),
    ("left",            np.pi / 2),
    ("backward_left",   3 * np.pi / 4),
    ("backward",        np.pi),
    ("backward_right", -3 * np.pi / 4),
    ("right",          -np.pi / 2),
    ("forward_right",  -np.pi / 4),
]
SECTOR_NAMES = [name for name, _ in SEMANTIC_SECTORS]

def bucketize(v_x, v_y, w_z, stop_eps=0.15, turn_eps=0.25):
    """Map a continuous (v_x, v_y, w_z) command to a semantic concept (post-hoc, non-control).

    Returns dict:
      label        : one of SECTOR_NAMES, or 'turn_left'/'turn_right'/'stop' for in-place behavior
      heading_rad  : atan2(v_y, v_x)
      heading_deg  : degrees
      speed        : planar magnitude ||(v_x, v_y)||
      turn_sign    : sign of w_z
      sector_idx   : index into SEMANTIC_SECTORS (or -1 for in-place labels)
    """
    speed = float(np.hypot(v_x, v_y))
    heading = float(np.arctan2(v_y, v_x))

    if speed < stop_eps:
        if w_z > turn_eps:
            label, idx = "turn_left", -1
        elif w_z < -turn_eps:
            label, idx = "turn_right", -1
        else:
            label, idx = "stop", -1
    else:
        # nearest sector center by angular distance
        diffs = [abs(np.angle(np.exp(1j * (heading - c)))) for _, c in SEMANTIC_SECTORS]
        idx = int(np.argmin(diffs))
        label = SEMANTIC_SECTORS[idx][0]

    return dict(label=label, heading_rad=heading, heading_deg=float(np.degrees(heading)),
                speed=speed, turn_sign=int(np.sign(w_z)), sector_idx=idx,
                v_x=float(v_x), v_y=float(v_y), w_z=float(w_z))

# quick self-test of the mapping
if __name__ != "__main__":
    _tests = [(1.0, 0.0, 0.0, "forward"), (1.0, 1.0, 0.0, "forward_left"),
              (0.0, 1.0, 0.0, "left"), (-1.0, -1.0, 0.0, "backward_right"),
              (1.0, -1.0, 0.0, "forward_right"), (0.0, 0.0, 1.0, "turn_left"),
              (0.0, 0.0, -1.0, "turn_right"), (0.0, 0.0, 0.0, "stop")]
    for vx, vy, wz, want in _tests:
        got = bucketize(vx, vy, wz)["label"]
        assert got == want, f"bucketize({vx},{vy},{wz}) -> {got}, expected {want}"
    print("bucketize self-test passed:", [t[3] for t in _tests])

## §9 — PPO locomotion training

Unchanged from the consolidated build. `go2_train.py` is written from `CFG`. **Skip** if a checkpoint
already exists in `logs/go2_locomote/` (restored from Drive in §9e).

In [ ]:
# §9a  Initialize Genesis + generate the grid texture + write the cfg JSON the trainer reads.
import os, sys, json, importlib
sys.path.insert(0, '/content')
import genesis as gs
import qre_utils as U
importlib.reload(U)

GRID_TEX = '/content/grid_floor.png'
U.make_grid_texture(GRID_TEX, size_px=2048, n_cells=8, line_px=2)  # large dark tiles, faint thin lines
print('grid texture ->', GRID_TEX)

# Build the env/obs/reward/command dicts from CFG (single source of truth).
JOINT_NAMES = ["FR_hip_joint","FR_thigh_joint","FR_calf_joint",
               "FL_hip_joint","FL_thigh_joint","FL_calf_joint",
               "RR_hip_joint","RR_thigh_joint","RR_calf_joint",
               "RL_hip_joint","RL_thigh_joint","RL_calf_joint"]
ENV_CFG = {
    "num_actions": 12, "dt": CFG.SIM.dt,
    "default_joint_angles": {
        "FL_hip_joint":0.0,"FR_hip_joint":0.0,"RL_hip_joint":0.0,"RR_hip_joint":0.0,
        "FL_thigh_joint":0.8,"FR_thigh_joint":0.8,"RL_thigh_joint":1.0,"RR_thigh_joint":1.0,
        "FL_calf_joint":-1.5,"FR_calf_joint":-1.5,"RL_calf_joint":-1.5,"RR_calf_joint":-1.5},
    "joint_names": JOINT_NAMES,
    "kp": CFG.SIM.kp, "kd": CFG.SIM.kd,
    "termination_if_roll_greater_than": CFG.SIM.termination_roll_deg,
    "termination_if_pitch_greater_than": CFG.SIM.termination_pitch_deg,
    "base_init_pos": CFG.SIM.base_init_pos, "base_init_quat": CFG.SIM.base_init_quat,
    "episode_length_s": CFG.SIM.episode_length_s, "resampling_time_s": CFG.SIM.resampling_time_s,
    "action_scale": CFG.SIM.action_scale, "simulate_action_latency": True, "clip_actions": 100.0,
}
OBS_CFG = {"num_obs": 45, "obs_scales": {"lin_vel":2.0,"ang_vel":0.25,"dof_pos":1.0,"dof_vel":0.05}}
REWARD_CFG = {"tracking_sigma":0.25,"base_height_target":0.3,"feet_height_target":0.075,
              "reward_scales":{"tracking_lin_vel":1.0,"tracking_ang_vel":0.2,"lin_vel_z":-1.0,
                               "base_height":-50.0,"action_rate":-0.005,"similar_to_default":-0.1}}
COMMAND_CFG = {"num_commands":3, "lin_vel_x_range": list(CFG.CMD.lin_vel_x_range),
               "lin_vel_y_range": list(CFG.CMD.lin_vel_y_range),
               "ang_vel_range": list(CFG.CMD.ang_vel_range)}

with open('/content/run_cfg.json', 'w') as f:
    json.dump(dict(env_cfg=ENV_CFG, obs_cfg=OBS_CFG, reward_cfg=REWARD_CFG,
                   command_cfg=COMMAND_CFG, exp_name=CFG.EXP_NAME), f, indent=2)

gs.init(backend=gs.gpu, precision="32", logging_level="warning")
print('cfg JSON written; genesis initialized.')

In [ ]:
# §9b  Write go2_train.py (reads run_cfg.json so it matches CFG exactly)
%%writefile /content/go2_train.py
import argparse, os, json, pickle
from importlib import metadata
try:
    try:
        if metadata.version("rsl-rl"): raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "2.2.4": raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please install 'rsl-rl-lib==2.2.4'.") from e

from rsl_rl.runners import OnPolicyRunner
import genesis as gs
from go2_env import Go2Env

# build the rsl-rl runner/policy/algorithm config dict
def get_train_cfg(exp_name, max_iterations):
    return {
        "algorithm": {"class_name":"PPO","clip_param":0.2,"desired_kl":0.01,"entropy_coef":0.01,
                      "gamma":0.99,"lam":0.95,"learning_rate":0.001,"max_grad_norm":1.0,
                      "num_learning_epochs":5,"num_mini_batches":4,"schedule":"adaptive",
                      "use_clipped_value_loss":True,"value_loss_coef":1.0},
        "init_member_classes": {},
        "policy": {"activation":"elu","actor_hidden_dims":[512,256,128],
                   "critic_hidden_dims":[512,256,128],"init_noise_std":1.0,"class_name":"ActorCritic"},
        "runner": {"checkpoint":-1,"experiment_name":exp_name,"load_run":-1,"log_interval":1,
                   "max_iterations":max_iterations,"record_interval":-1,"resume":False,
                   "resume_path":None,"run_name":""},
        "runner_class_name":"OnPolicyRunner",
        "num_steps_per_env":24,"save_interval":100,"empirical_normalization":None,"seed":1,
    }

# parse CLI args and run PPO training to convergence
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("-e","--exp_name", type=str, default="go2_locomote")
    ap.add_argument("-B","--num_envs", type=int, default=4096)
    ap.add_argument("--max_iterations", type=int, default=600)
    ap.add_argument("--log_root", type=str, default="logs")
    ap.add_argument("--cfg_json", type=str, default="/content/run_cfg.json")
    ap.add_argument("--resume_from", type=str, default="")
    args = ap.parse_args()

    cfg = json.load(open(args.cfg_json))
    env_cfg, obs_cfg, reward_cfg, command_cfg = (cfg["env_cfg"], cfg["obs_cfg"],
                                                 cfg["reward_cfg"], cfg["command_cfg"])
    gs.init(backend=gs.gpu, precision="32", logging_level="warning")
    log_dir = f"{args.log_root}/{args.exp_name}"
    train_cfg = get_train_cfg(args.exp_name, args.max_iterations)

    os.makedirs(log_dir, exist_ok=True)
    pickle.dump([env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg],
                open(f"{log_dir}/cfgs.pkl", "wb"))

    env = Go2Env(num_envs=args.num_envs, env_cfg=env_cfg, obs_cfg=obs_cfg,
                 reward_cfg=reward_cfg, command_cfg=command_cfg, show_viewer=False)
    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    if args.resume_from and os.path.exists(args.resume_from):
        print("Resuming from", args.resume_from); runner.load(args.resume_from)
    runner.learn(num_learning_iterations=args.max_iterations, init_at_random_ep_len=True)
    print("Training done. Checkpoints in", log_dir)

if __name__ == "__main__":
    main()

In [ ]:
# §9d  rsl-rl's TensorBoard pulls in TF whose CUDA init can deadlock here; remove it.
!pip uninstall -y tensorflow tensorflow-cpu 2>/dev/null
print("tensorflow removed (tensorboard event writer still works)")

In [ ]:
# §9e  Restore any prior checkpoints from Drive, then train (resumable).
import os, shutil
os.chdir('/content')
EXP = CFG.EXP_NAME
DRIVE_LOG = f'{CFG.LOG_DIR}'
os.makedirs(f'logs/{EXP}', exist_ok=True)
if os.path.isdir(f'{DRIVE_LOG}/{EXP}'):
    for f in os.listdir(f'{DRIVE_LOG}/{EXP}'):
        src = f'{DRIVE_LOG}/{EXP}/{f}'
        if os.path.isfile(src): shutil.copy(src, f'logs/{EXP}/{f}')
resume = ''
ck = [f for f in os.listdir(f'logs/{EXP}') if f.startswith('model_')]
if ck:
    latest = max(int(f[6:-3]) for f in ck); resume = f'logs/{EXP}/model_{latest}.pt'
    print('Will resume from', resume)

# Comment out the next line if you already have a trained checkpoint and want to skip training.
#!python -u go2_train.py --exp_name {EXP} --num_envs {CFG.NUM_ENVS} --max_iterations {CFG.PPO_ITERS} {('--resume_from ' + resume) if resume else ''}

In [ ]:
# §9f  Mirror checkpoints to Drive.
import os, shutil
EXP = CFG.EXP_NAME
os.makedirs(f'{CFG.LOG_DIR}/{EXP}', exist_ok=True)
for f in os.listdir(f'logs/{EXP}'):
    src = f'logs/{EXP}/{f}'
    if os.path.isfile(src): shutil.copy(src, f'{CFG.LOG_DIR}/{EXP}/{f}')
print('Checkpoints mirrored to Drive:', sorted(os.listdir(f'{CFG.LOG_DIR}/{EXP}')))

## §10 — Load the trained policy and the MEM planner

Same two bug fixes as before: `torch.set_default_device("cpu")` after `gs.init()` (Genesis flips it
to CUDA and breaks the HF processor), and re-injecting `class_name="ActorCritic"` into the policy cfg
(rsl-rl pops it during training). The MEM memory knobs come from `CFG`.

In [ ]:
# §10a  Camera helpers: onboard (what the VLM sees) vs isometric (human video).
import numpy as np

# robot base yaw, in radians
def _robot_yaw(env):
    q = np.asarray(env.robot.get_quat().cpu()).reshape(-1)[:4]   # Genesis quat = (w, x, y, z)
    w, x, y, z = q
    return float(np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z)))

# 2D rotation matrix for a yaw angle
def _yaw_R(yaw):
    c, s = np.cos(yaw), np.sin(yaw)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

def onboard_pose(env):
    """First-person camera just ahead of the body, looking forward along the heading."""
    base = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0]
    R = _yaw_R(_robot_yaw(env))
    pos  = base + R @ np.array([0.30, 0.0, 0.15])
    look = base + R @ np.array([3.0,  0.0, 0.0])
    return tuple(pos.tolist()), tuple(look.tolist())

def iso_pose(env):
    """Elevated 3/4 view that follows the robot at a fixed world angle."""
    base = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0]
    pos  = base + np.array([2.8, 2.8, 2.2])
    look = base + np.array([0.0, 0.0, 0.25])
    return tuple(pos.tolist()), tuple(look.tolist())

def set_level_pose(camera, pos, lookat, up=(0.0, 0.0, 1.0)):
    """Pose the camera with an EXPLICIT world-up so the horizon stays level.
    Genesis' set_pose() leaves the camera ROLLED when 'up' is omitted (the
    per-step onboard/iso frames came out tilted ~45 deg, which scrambles the
    VLM's left/right/forward reasoning). Passing up=(0,0,1) fixes the roll."""
    try:
        camera.set_pose(pos=tuple(pos), lookat=tuple(lookat), up=tuple(up))
    except TypeError:
        camera.set_pose(pos=tuple(pos), lookat=tuple(lookat))


In [ ]:
# §10b  Build arena env, load PPO policy, instantiate the MEM planner.
import importlib, pickle, os, torch, genesis as gs
import qre_utils as U
import mem_planner as MP
importlib.reload(U); importlib.reload(MP)
from go2_env import Go2Env
from rsl_rl.runners import OnPolicyRunner

try:
    gs.init(backend=gs.gpu, precision="32", logging_level="warning")
except Exception:
    pass
torch.set_default_device("cpu")   # BUG FIX: genesis flips default to cuda; undo for HF processors

EXP = CFG.EXP_NAME; log_dir = f'logs/{EXP}'
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f'{log_dir}/cfgs.pkl', 'rb'))
reward_cfg = dict(reward_cfg, reward_scales={})

GRID = '/content/grid_floor.png'
# build a fresh Go2Env with the two coloured goal boxes placed
def build_arena(left_color, right_color):
    left, right = U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                                   left_color=left_color, right_color=right_color)
    env = Go2Env(num_envs=1, env_cfg=env_cfg, obs_cfg=obs_cfg, reward_cfg=reward_cfg,
                 command_cfg=command_cfg, show_viewer=False, attach_camera=True,
                 camera_res=CFG.CAM.res, camera_pos=CFG.CAM.arena_pos, camera_lookat=CFG.CAM.arena_lookat,
                 camera_fov=CFG.CAM.fov, hiq=True, grid_texture=GRID, extra_objects=[left, right])
    return env, left, right

# load the latest PPO checkpoint and return an inference-only policy
def load_policy(env):
    if "class_name" not in train_cfg["policy"]:           # BUG FIX: popped during training
        train_cfg["policy"]["class_name"] = "ActorCritic"
    ck = [f for f in os.listdir(log_dir) if f.startswith('model_')]
    latest = max(int(f[6:-3]) for f in ck)
    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f'model_{latest}.pt'))
    print('Loaded checkpoint', latest)
    return runner.get_inference_policy(device=gs.device)

# Load VLM + MEM planner (memory knobs from CFG; re-run this cell after editing §3 to change them).
model, processor = MP.load_qwen(CFG.QWEN_ID, device="cuda")
planner = MP.MEMPlanner(model, processor, CFG.CONTRACT,
                        CFG.CMD.lin_vel_x_range, CFG.CMD.lin_vel_y_range, CFG.CMD.ang_vel_range,
                        short_frames=CFG.MEM_SHORT_FRAMES, short_stride=CFG.MEM_SHORT_STRIDE,
                        as_video=CFG.MEM_AS_VIDEO, lang_max_chars=CFG.MEM_LANG_MAX_CHARS,
                        short_enabled=CFG.MEM_SHORT_ENABLED, lang_enabled=CFG.MEM_LANG_ENABLED,
                        max_new_tokens=CFG.VLM_MAX_NEW_TOKENS)
print(f'MEM planner ready | short_enabled={planner.short_enabled} lang_enabled={planner.lang_enabled}')

## §11 — Closed-loop runtime (continuous command + dual memory)

Identical control path to before — the VLM's continuous `(v_x,v_y,w_z)` is clamped, low-pass
filtered, and written straight into `env.commands`; the 50 Hz PPO policy tracks it. The only
difference is the planner is now `MEMPlanner`, so each query also (a) sees the short-term frame
window and (b) rewrites the long-term language memory. We log the evolving memory per query.

Carried-over fixes: command set **before** `env.step()` and re-set **after** (step resamples), 4-value
unpack, dual onboard/iso render, MP4 output, `planner.reset()` per trial so memory starts empty.

In [ ]:
# §11  Continuous closed-loop runtime with MEM hybrid memory.
import numpy as np
from PIL import Image as PILImage
import imageio.v2 as imageio

# drive one trial: VLM plans, PPO controls, camera + memory update every query
def run_closed_loop(env, policy, planner, left, right, instruction,
                    temperature=0.7, seed=0, max_seconds=8.0, commit_radius=None,
                    record_path=None, want_attention=False, log_every_query=True, record_fps=50):
    commit_radius = CFG.ARENA.commit_radius if commit_radius is None else commit_radius
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(max_seconds / env.dt)

    planner.reset()                          # MEM memory starts empty each trial
    obs, _ = env.reset()
    cmd = np.zeros(3, dtype=float)
    queries = []; chosen = None; commit_step = None; rec_frames = []

    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, _ = U.render_camera(env.camera); last_rgb = rgb

    with torch.no_grad():
        for t in range(max_steps):
            if t % query_every == 0:
                frame = PILImage.fromarray(last_rgb[..., :3].astype('uint8')).convert('RGB')
                r = planner.act(frame, instruction, temperature=temperature,
                                seed=seed + t, want_attention=want_attention)
                new = np.array([r['v_x'], r['v_y'], r['w_z']], float)   # CONTINUOUS, direct
                cmd = CFG.LOWPASS_ALPHA * new + (1 - CFG.LOWPASS_ALPHA) * cmd
                if log_every_query:
                    sem = bucketize(r['v_x'], r['v_y'], r['w_z'])
                    queries.append(dict(step=t, parse_status=r['parse_status'],
                                        v_x=r['v_x'], v_y=r['v_y'], w_z=r['w_z'],
                                        semantic=sem['label'], choice=r.get('choice'),
                                        reasoning=r.get('reasoning'),
                                        language_memory=r.get('language_memory'),
                                        n_short_frames=r.get('n_short_frames'),
                                        used_video=r.get('used_video')))
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)

            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            act = policy(obs); obs, _, done, _ = env.step(act)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)

            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, _ = U.render_camera(env.camera); last_rgb = rgb

            if record_path:
                pr, lr = iso_pose(env); set_level_pose(env.camera, pr, lr)
                iso_rgb, _ = U.render_camera(env.camera)
                rec_frames.append(iso_rgb[..., :3].astype('uint8'))

            p2 = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0][:2]
            for obj in (left, right):
                if np.linalg.norm(p2 - np.array(obj['pos'][:2])) < commit_radius:
                    chosen = obj; commit_step = t; break
            if chosen is not None: break

    if record_path and rec_frames:
        imageio.mimwrite(record_path, rec_frames, fps=record_fps, macro_block_size=None)

    statuses = [q['parse_status'] for q in queries]
    parse_rate = sum(s != 'parse_failed' for s in statuses) / max(1, len(statuses))
    return dict(chosen_side=None if chosen is None else chosen['side'],
                chosen_color=None if chosen is None else chosen['color'],
                commit_step=commit_step, queries=queries, parse_rate=parse_rate,
                n_queries=len(queries), final_memory=planner.language_memory)

## §12 — Comprehensive evaluation (mirror + control, grouped detailed logs)

No videos here — just **trial outcomes + detailed per-query logs** written to disk,
grouped by category, so the behaviour can be audited later.

**Design (answers to the open questions):**

* **Mirror test.** Each semantic condition is run in both arrangements — `L=red,R=blue`
  and its mirror `L=blue,R=red`. Pooling the two lets us separate *colour-tracking*
  (does it pick **red** regardless of side?) from *side-tracking* (does it pick a fixed
  **side** regardless of colour?). Without the mirror those two are confounded.
* **Control (identical targets, both red).** With no colour cue, choice can only come
  from positional/visual bias, so the control measures the **baseline side bias**.
  *Do I need both prompts?* If you only care about positional bias, **one prompt is
  enough**. But running **both** ("friendly"/"hostile") on identical targets is cheap and
  additionally tests whether the *word itself* shifts behaviour (e.g. "hostile" →
  approach/avoid) even with no colour to act on. We run **both** — each control prompt is its own
  category cell (§12c-5 / §12c-6); skip one of them for a single neutral run.
* **How many trials?** Each trial is a Bernoulli outcome, so the 95% CI half-width at the
  worst case (p=0.5) is ≈ `1.96·√(0.25/N)`: N=20 → ±0.22, N=30 → ±0.18, N=50 → ±0.14,
  N≈96 → ±0.10, N≈196 → ±0.07. These results were collected with **N=50/condition**
  (`EVAL.N_TRIALS=50` → ±0.14 worst case); ~100 would be needed for ±10%. Cost scales
  linearly (~30–60 s/trial on A100); `EVAL.QUICK=True` gives a 3-trial smoke test
  (the recorded runs used `QUICK=False`).

**Drift handling.** If the agent never enters `commit_radius`, the trial is labelled
**`drift`** but still **committed to the target nearest its final onboard-camera position**
(its last focus-of-attention), so every trial contributes to the choice statistics.

**Frame memory (V2).** Short-term visual memory is the **last 4 recent frames sampled
~0.33 s apart** (≈1 s window; one frame per VLM decision at 3 Hz = 1/`VLM_RATE_HZ`; time-strided, adjustable) — because at this walking speed consecutive
control frames are near-identical, so spacing the window gives real motion. There is **no
initial-anchor frame**; long-term memory stays the running text summary. The contract gets a
short note describing the recent clip.


In [ ]:
# §12a  MEM planner V2: short-term visual memory = the last `recent_frames` onboard frames
#       sampled at a fixed TIME interval (NOT every query). The agent walks slowly, so
#       consecutive control frames look identical; sampling the recent window ~0.33 s apart
#       gives real motion. There is NO initial-anchor frame anymore — just the recent clip.
#       Long-term language memory + parsing/clamping/rewrite are inherited from MEMPlanner.
import collections
import mem_planner as MP   # reuse parse_continuous / vx_token_attention / MEMPlanner

class MEMPlannerV2(MP.MEMPlanner):
    """Short-term visual memory = last `recent_frames` onboard frames sampled every
    `short_interval_s` seconds (time-strided), fed as a native Qwen video. No anchor frame.
    Long-term language memory + parsing/clamping/rewrite inherited from MEMPlanner."""
    # same as the base MEMPlanner, but keyed on wall-clock time instead of step count
    def __init__(self, *args, recent_frames=4, short_interval_s=0.3333, **kw):
        kw.setdefault("short_frames", recent_frames)   # bound the recent deque
        # set these BEFORE super().__init__: the base constructor calls self.reset()
        # -> MEMPlannerV2.reset(), which reads self.recent_frames.
        self.recent_frames    = recent_frames
        self.short_interval_s = short_interval_s
        super().__init__(*args, **kw)   # base __init__ already calls self.reset()

    # clear the recent-frame buffer, memory string and internal clock
    def reset(self):
        super().reset()
        self._recent      = collections.deque(maxlen=max(1, self.recent_frames))
        self._last_push_t = -1e9
        self._now_t       = 0.0

    # like the base planner, but only pushes a frame once per short_interval_s
    def _build_messages(self, frame, instruction):
        # time-strided push; between pushes keep the newest slot fresh so the latest view
        # is always current even when we have not advanced a full short_interval_s yet.
        if len(self._recent) == 0 or (self._now_t - self._last_push_t) >= self.short_interval_s - 1e-6:
            self._recent.append(frame); self._last_push_t = self._now_t
        else:
            self._recent[-1] = frame

        content, recent, used_video = [], list(self._recent), False
        if self.short_enabled:
            if len(recent) > 1 and self.as_video:
                content.append({"type": "text",
                                "text": f"[recent motion - last {len(recent)} frames, "
                                        f"~{self.short_interval_s:g}s apart, oldest to newest]"})
                content.append({"type": "video", "video": recent})
                used_video = True
            else:
                for j, fr in enumerate(recent):
                    tag = "now" if j == len(recent) - 1 else f"t-{len(recent)-1-j}"
                    content.append({"type": "text", "text": f"[recent {tag}]"})
                    content.append({"type": "image", "image": fr})
        else:
            content.append({"type": "image", "image": frame})

        mem_block = ""
        if self.short_enabled:
            mem_block += (f"\nShort-term visual memory: the clip above is your last {len(recent)} "
                          f"onboard frames ~{self.short_interval_s:g}s apart (oldest to newest); "
                          "use them to judge your current motion and which way you are drifting.")
        if self.lang_enabled:
            mem_block += ("\nLong-term memory so far: "
                          + (self._language_memory if self._language_memory else "(empty)"))
        content.append({"type": "text",
                        "text": f"{self.contract}{mem_block}\n\nInstruction: {instruction}"})
        return [{"role": "user", "content": content}], used_video

    # advance the internal clock, then delegate to the base planner
    def act(self, frame, instruction, sim_time=0.0, **kw):
        self._now_t = float(sim_time)
        out = super().act(frame, instruction, **kw)
        out["n_short_frames"] = len(self._recent)
        return out

# Contract: describe the recent-frame clip (NO anchor frame anymore).
# Short-term stride is LOCKED to the VLM query period (1 / VLM_RATE_HZ): the planner only
# acts at the VLM rate, so recent frames can't physically be spaced any finer than that.
# Deriving it keeps the labelled spacing honest and auto-tracks CFG.VLM_RATE_HZ.
#   4 frames @ ~0.333 s  ->  ~1.0 s between oldest/newest (clean 1 s window; even temporal
#   grouping so Qwen does NOT pad an odd frame count).
CFG.MEM2 = SimpleNamespace(
    RECENT_FRAMES    = 4,
    SHORT_INTERVAL_S = round(1.0 / CFG.VLM_RATE_HZ, 4),   # 3 Hz -> 0.3333 s
)
CFG.CONTRACT_MEM2 = CFG.CONTRACT + (
    "\nNote on your visual memory: the frames are your most recent onboard views a fraction "
    "of a second apart (oldest to newest); use them to judge your current motion and which "
    "way you are drifting, and keep your committed target stable across them."
)

planner_v2 = MEMPlannerV2(
    model, processor, CFG.CONTRACT_MEM2,
    CFG.CMD.lin_vel_x_range, CFG.CMD.lin_vel_y_range, CFG.CMD.ang_vel_range,
    recent_frames=CFG.MEM2.RECENT_FRAMES, short_interval_s=CFG.MEM2.SHORT_INTERVAL_S,
    as_video=CFG.MEM_AS_VIDEO, lang_max_chars=CFG.MEM_LANG_MAX_CHARS,
    short_enabled=CFG.MEM_SHORT_ENABLED, lang_enabled=CFG.MEM_LANG_ENABLED,
    max_new_tokens=CFG.VLM_MAX_NEW_TOKENS)
print("MEM V2 planner ready (no anchor; recent-only):", CFG.MEM2)

In [ ]:
# §12b  One trial: NO video render, just outcomes + per-query logs. Drift-aware: if the
#       agent never enters commit_radius we still COMMIT it to the target nearest its final
#       onboard-camera (x,y) position, and label the trial 'drift'.
import numpy as np
from PIL import Image as PILImage

def color_name(c):
    """Map an RGB tuple to 'red'/'blue' by nearest reference colour."""
    r = np.array(CFG.COLOR_RED, float); b = np.array(CFG.COLOR_BLUE, float); c = np.array(c[:3], float)
    return "red" if np.linalg.norm(c - r) <= np.linalg.norm(c - b) else "blue"

# drive one full trial to a commit radius or timeout, logging every VLM query
def run_trial(env, policy, planner, left, right, instruction,
              temperature=0.7, seed=0, max_seconds=6.0, commit_radius=None):
    commit_radius = CFG.ARENA.commit_radius if commit_radius is None else commit_radius
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(max_seconds / env.dt)
    planner.reset(); obs, _ = env.reset()
    cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, _ = U.render_camera(env.camera); last_rgb = rgb
    queries = []; reached = None; commit_step = None
    with torch.no_grad():
        for t in range(max_steps):
            if t % query_every == 0:
                frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
                r = planner.act(frame, instruction, sim_time=t * env.dt,
                                temperature=temperature, seed=seed + t)
                new = np.array([r["v_x"], r["v_y"], r["w_z"]], float)
                cmd = CFG.LOWPASS_ALPHA * new + (1 - CFG.LOWPASS_ALPHA) * cmd
                sem = bucketize(r["v_x"], r["v_y"], r["w_z"])
                queries.append(dict(step=t, parse_status=r["parse_status"],
                                    v_x=r["v_x"], v_y=r["v_y"], w_z=r["w_z"],
                                    semantic=sem["label"], choice=r.get("choice"),
                                    reasoning=r.get("reasoning"),
                                    language_memory=r.get("language_memory"),
                                    n_short_frames=r.get("n_short_frames"),
                                    used_video=r.get("used_video")))
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            act = policy(obs); obs, _, done, _ = env.step(act)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, _ = U.render_camera(env.camera); last_rgb = rgb
            p2 = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0][:2]
            for obj in (left, right):
                if np.linalg.norm(p2 - np.array(obj["pos"][:2])) < commit_radius:
                    reached = obj; commit_step = t; break
            if reached is not None:
                break
    cam_xy = np.array(onboard_pose(env)[0][:2])
    if reached is not None:
        outcome, chosen = "reached", reached
    else:
        outcome = "drift"
        dL = np.linalg.norm(cam_xy - np.array(left["pos"][:2]))
        dR = np.linalg.norm(cam_xy - np.array(right["pos"][:2]))
        chosen = left if dL <= dR else right
    statuses = [q["parse_status"] for q in queries]
    parse_rate = sum(s != "parse_failed" for s in statuses) / max(1, len(statuses))
    return dict(outcome=outcome, chosen_side=chosen["side"], chosen_color=chosen["color"],
                commit_step=commit_step, parse_rate=parse_rate, n_queries=len(queries),
                final_memory=planner.language_memory, queries=queries, cam_xy=cam_xy.tolist())

print("run_trial / color_name ready.")


In [ ]:
# §12c  Evaluation SETUP — run this ONCE per session (together with the §12e batched-runner
#       cells), then run any of the per-category cells below. Defines EVAL (N_TRIALS=50 per
#       category), the output paths, and the SEQUENTIAL `run_category` runner; the §12c-x
#       category cells actually call the BATCHED runner (§12e `run_category_batched`, identical
#       logs/format), with `run_category` kept as a drop-in sequential fallback. Each category
#       writes its own logs under OUT_DIR/<slug>/, so categories can be run one at a time
#       across different sessions. §12d aggregates whatever category logs exist on disk.
import os, json, gc, copy

EVAL = SimpleNamespace(
    N_TRIALS=50,                 # per category
    QUICK=False,                 # True -> N_TRIALS=3 smoke test
    TEMP=CFG.VLM_TEMPERATURE,    # >0 so choices have variance
    MAX_SECONDS=6.0,             # timeout per trial (lower = shorter; see §12 notes)
    SEED_BASE=1000,
    OUT_DIR=f"{CFG.TRIAL_DIR}/mem_eval",
)
if EVAL.QUICK:
    EVAL.N_TRIALS = 3
os.makedirs(EVAL.OUT_DIR, exist_ok=True)

RED  = CFG.COLOR_RED
BLUE = U.luminance_match(CFG.COLOR_RED, CFG.COLOR_BLUE)   # luminance-matched blue

# turn a category string into a filesystem-safe slug
def _slug(s):
    return (s.replace(" ", "").replace("|", "__").replace(",", "-")
             .replace("=", "").replace(".", "").replace("'", ""))

def load_policy(env):    # rsl-rl pops class_name from BOTH policy+algorithm cfgs each build
    tc = copy.deepcopy(train_cfg)
    tc["policy"].setdefault("class_name", "ActorCritic")
    tc["algorithm"].setdefault("class_name", "PPO")
    ck = [f for f in os.listdir(log_dir) if f.startswith("model_")]
    latest = max(int(f[6:-3]) for f in ck)
    runner = OnPolicyRunner(env, tc, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f"model_{latest}.pt"))
    print("Loaded checkpoint", latest)
    return runner.get_inference_policy(device=gs.device)

def run_category(arr_name, lc, rc, kind, prompt, n_trials=None):
    """Run ONE category (arrangement x prompt). Builds + tears down its own arena and writes
    its own logs under OUT_DIR/<slug>/ (trials.jsonl, queries.jsonl, log.txt, summary.json)."""
    n_trials = EVAL.N_TRIALS if n_trials is None else n_trials
    cat  = f"{arr_name} | {prompt}"
    cdir = f"{EVAL.OUT_DIR}/{_slug(cat)}"; os.makedirs(cdir, exist_ok=True)
    t_path, q_path, txt_path = f"{cdir}/trials.jsonl", f"{cdir}/queries.jsonl", f"{cdir}/log.txt"
    open(t_path, "w").close(); open(q_path, "w").close()
    print(f"\n######## {cat}  ({kind})  n={n_trials} ########")
    env, left, right = build_arena(left_color=lc, right_color=rc)
    policy = load_policy(env)
    recs, blocks = [], []
    for i in range(n_trials):
        seed = EVAL.SEED_BASE + 13 * i
        res = run_trial(env, policy, planner_v2, left, right, prompt,
                        temperature=EVAL.TEMP, seed=seed, max_seconds=EVAL.MAX_SECONDS)
        cname = color_name(res["chosen_color"])
        rec = dict(category=cat, arrangement=arr_name, kind=kind, prompt=prompt,
                   trial=i, seed=seed, outcome=res["outcome"],
                   chosen_side=res["chosen_side"], chosen_color=cname,
                   commit_step=res["commit_step"], parse_rate=res["parse_rate"],
                   n_queries=res["n_queries"], final_memory=res["final_memory"])
        recs.append(rec)
        with open(t_path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        with open(q_path, "a") as f:
            for q in res["queries"]:
                f.write(json.dumps(dict(category=cat, trial=i, seed=seed, **q)) + "\n")
        block = [f"--- trial {i} (seed {seed}) -> {res['outcome'].upper()} "
                 f"chose {cname}/{res['chosen_side']} | parse={res['parse_rate']:.2f} "
                 f"| final_mem: {res['final_memory']!r}"]
        for q in res["queries"]:
            block.append(f"  step {q['step']:>4} | ({q['v_x']:+.2f},{q['v_y']:+.2f},"
                         f"{q['w_z']:+.2f}) -> {q['semantic']:<13} choice={q['choice']}")
            if q["reasoning"]:       block.append(f"        reason: {q['reasoning']}")
            if q["language_memory"]: block.append(f"        mem   : {q['language_memory']}")
        blocks.append("\n".join(block))
        print(f"  {i+1}/{n_trials}: {res['outcome']:7s} -> {cname}/{res['chosen_side']}")
    with open(txt_path, "w") as f:
        f.write("=" * 92 + f"\nCATEGORY: {cat}  (n={len(blocks)}, kind={kind})\n" + "=" * 92 + "\n")
        f.write("\n\n".join(blocks) + "\n")
    json.dump(dict(category=cat, kind=kind, n=len(recs),
                   reached=sum(r["outcome"] == "reached" for r in recs),
                   red=sum(r["chosen_color"] == "red" for r in recs),
                   left=sum(r["chosen_side"] == "left" for r in recs)),
              open(f"{cdir}/summary.json", "w"), indent=2)
    del env, policy; gc.collect(); torch.cuda.empty_cache()
    print(f"  wrote logs -> {cdir}")
    return recs

print("Eval setup ready. Run a per-category cell below (running just one per session is fine).")
print("Per-category logs will land under:", EVAL.OUT_DIR)

### §12e — parallel (batched) evaluation  *(optional, faster)*
Runs **B trials of one category at once** — B parallel Genesis envs + one batched Qwen `generate()` per tick. Same logs/drift-handling as §12c, just faster. **Batch within a category only** (box colour is scene-global). Dependencies: §12a (CFG.MEM2/CONTRACT_MEM2), §12b (`run_trial`/`color_name`), §12c setup (`EVAL`, `_slug`, `load_policy`).

**Run order:** §12e setup → §12e-run → then the §12c category cells, which call `run_category_batched(...)` (same args as `run_category`; N=50 trials in batches of `EVALB.BATCH=6`). §12e-validate is kept **commented out** — the per-env render check passed; un-comment it to re-verify if the Genesis version changes. Tune `EVALB.BATCH` to VRAM (start 6) and `EVALB.ENV_SPACING` if neighbour cubes appear in the validation views.

In [ ]:
# §12e  PARALLEL evaluation (batched). Runs B trials of ONE category at once: B parallel
#       Genesis envs (same colours/prompt, different seeds) + ONE batched VLM generate per
#       tick. Drop-in for §12c: same logs, same drift handling. The VLM batch IS the speedup.
#       NOTE: batch within a category only (box colour is scene-global). Tune EVALB.BATCH to
#       VRAM. RUN THE §12e-validate CELL FIRST to confirm the per-env views look right.
#       Short-term memory = recent frames only (no anchor), matching §12a.
import os, json, gc, collections
import numpy as np
from PIL import Image as PILImage
import mem_planner as MP

EVALB = SimpleNamespace(BATCH=6, ENV_SPACING=(30.0, 30.0))   # BATCH ~ VLM batch; raise if VRAM allows

# ---- batched MEM-V2 planner: B independent memories, ONE batched generate -----------------
class MEMPlannerBatch:
    # store config shared by all B independent short/long-term memories
    def __init__(self, model, processor, contract, vx_range, vy_range, wz_range,
                 recent_frames=CFG.MEM2.RECENT_FRAMES, short_interval_s=CFG.MEM2.SHORT_INTERVAL_S, as_video=True,
                 lang_max_chars=600, short_enabled=True, lang_enabled=True, max_new_tokens=192):
        self.model = model; self.processor = processor; self.contract = contract
        self.vx_range, self.vy_range, self.wz_range = vx_range, vy_range, wz_range
        self.recent_frames = recent_frames; self.short_interval_s = short_interval_s
        self.as_video = as_video
        self.lang_max_chars = lang_max_chars; self.short_enabled = short_enabled
        self.lang_enabled = lang_enabled; self.max_new_tokens = max_new_tokens

    # allocate B independent per-env memory buffers
    def reset(self, B):
        self.B = B
        self._recent = [collections.deque(maxlen=max(1, self.recent_frames)) for _ in range(B)]
        self._lang   = [""] * B
        self._last_push = [-1e9] * B

    # push a frame into env i's short-term buffer if its sample interval elapsed
    def _update_i(self, i, frame, now_t):
        if len(self._recent[i]) == 0 or (now_t - self._last_push[i]) >= self.short_interval_s - 1e-6:
            self._recent[i].append(frame); self._last_push[i] = now_t
        else:
            self._recent[i][-1] = frame

    # assemble env i's chat content (recent frames + memory text)
    def _content_i(self, i, frame, instruction):
        content, recent = [], list(self._recent[i])
        if self.short_enabled:
            if len(recent) > 1 and self.as_video:
                content.append({"type": "text",
                                "text": f"[recent motion - last {len(recent)} frames, "
                                        f"~{self.short_interval_s:g}s apart, oldest to newest]"})
                content.append({"type": "video", "video": recent})
            else:
                for j, fr in enumerate(recent):
                    tag = "now" if j == len(recent) - 1 else f"t-{len(recent)-1-j}"
                    content.append({"type": "text", "text": f"[recent {tag}]"})
                    content.append({"type": "image", "image": fr})
        else:
            content.append({"type": "image", "image": frame})
        mem = ""
        if self.short_enabled:
            mem += (f"\nShort-term visual memory: the clip is your last {len(recent)} most recent "
                    f"onboard views ~{self.short_interval_s:g}s apart (oldest to newest).")
        if self.lang_enabled:
            mem += "\nLong-term memory so far: " + (self._lang[i] if self._lang[i] else "(empty)")
        content.append({"type": "text", "text": f"{self.contract}{mem}\n\nInstruction: {instruction}"})
        return [{"role": "user", "content": content}]

    # one batched slow-planner step: all B envs planned in a single generate() call
    def act_batch(self, frames, instructions, sim_time=0.0, temperature=0.7, seed=0):
        import torch
        from qwen_vl_utils import process_vision_info
        torch.manual_seed(seed)
        B = len(frames)
        for i in range(B):
            self._update_i(i, frames[i], float(sim_time))     # advance buffers ONCE
        # assemble prompts + vision inputs for the whole batch
        def build():
            msgs = [self._content_i(i, frames[i], instructions[i]) for i in range(B)]
            texts = [self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
            img, vid = process_vision_info(msgs)
            return texts, img, vid
        try:
            texts, img, vid = build()
        except Exception:
            self.as_video = False
            texts, img, vid = build()
        old_side = self.processor.tokenizer.padding_side
        self.processor.tokenizer.padding_side = "left"          # left-pad for batched generation
        inputs = self.processor(text=texts, images=img, videos=vid, padding=True, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens,
                                      do_sample=temperature > 0, temperature=max(1e-4, temperature),
                                      return_dict_in_generate=True)
        self.processor.tokenizer.padding_side = old_side
        seqs = out.sequences; prompt_len = inputs.input_ids.shape[1]
        results = []
        for i in range(B):
            raw = self.processor.decode(seqs[i][prompt_len:], skip_special_tokens=True)
            parsed = MP.parse_continuous(raw, self.vx_range, self.vy_range, self.wz_range)
            if self.lang_enabled and parsed.get("memory"):
                self._lang[i] = str(parsed["memory"])[: self.lang_max_chars]
            parsed["language_memory"] = self._lang[i]
            parsed["n_short_frames"] = len(self._recent[i])
            parsed["used_video"] = bool(self.as_video and self.short_enabled and len(self._recent[i]) > 1)
            results.append(parsed)
        return results

planner_batch = MEMPlannerBatch(
    model, processor, CFG.CONTRACT_MEM2,
    CFG.CMD.lin_vel_x_range, CFG.CMD.lin_vel_y_range, CFG.CMD.ang_vel_range,
    recent_frames=CFG.MEM2.RECENT_FRAMES, short_interval_s=CFG.MEM2.SHORT_INTERVAL_S,
    as_video=CFG.MEM_AS_VIDEO, lang_max_chars=CFG.MEM_LANG_MAX_CHARS,
    short_enabled=CFG.MEM_SHORT_ENABLED, lang_enabled=CFG.MEM_LANG_ENABLED,
    max_new_tokens=CFG.VLM_MAX_NEW_TOKENS)

# ---- batched arena + per-env onboard pose -------------------------------------------------
def build_arena_batched(lc, rc, num_envs, env_spacing=(30.0, 30.0)):
    left, right = U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                                   left_color=lc, right_color=rc)
    env = Go2Env(num_envs=num_envs, env_cfg=env_cfg, obs_cfg=obs_cfg, reward_cfg=reward_cfg,
                 command_cfg=command_cfg, show_viewer=False, attach_camera=True,
                 camera_res=CFG.CAM.res, camera_pos=CFG.CAM.arena_pos, camera_lookat=CFG.CAM.arena_lookat,
                 camera_fov=CFG.CAM.fov, hiq=True, grid_texture=GRID, extra_objects=[left, right],
                 env_spacing=env_spacing, rendered_envs_idx=list(range(num_envs)))
    return env, left, right

def _discover_env_offsets(env, B):
    """Genesis get_pos() is per-env LOCAL; find each env's GLOBAL tiling offset for rendering."""
    owners = [env, getattr(env, "scene", None)]
    names = ["envs_offset", "_envs_offset", "env_offset", "_env_offset", "envs_origin", "_envs_origin"]
    for owner in owners:
        if owner is None:
            continue
        for nm in names:
            v = getattr(owner, nm, None)
            if v is None:
                continue
            try:
                arr = (v.detach().cpu().numpy() if hasattr(v, "detach")
                       else (v.cpu().numpy() if hasattr(v, "cpu") else np.asarray(v)))
                arr = arr.reshape(-1, 3)
                if arr.shape[0] >= B:
                    return arr[:B], f"{type(owner).__name__}.{nm}"
            except Exception:
                continue
    return None, None

# onboard camera pose for env i, offset into its own tile
def onboard_pose_i(env, i):
    off = getattr(env, "_render_offsets", None)
    if off is None:
        off, srcname = _discover_env_offsets(env, env.num_envs)
        if off is None:
            off = np.zeros((env.num_envs, 3))
            print("WARN: per-env global offsets NOT found; batched onboard render is likely WRONG "
                  "for env>0. Run the decisive §12e-validate cell; if panels are identical, use §12c instead.")
        else:
            print("batched render using env offsets from", srcname)
        env._render_offsets = off
    base = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[i] + off[i]
    q = np.asarray(env.robot.get_quat().cpu()).reshape(-1, 4)[i]
    w, x, y, z = q
    yaw = float(np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z)))
    R = _yaw_R(yaw)
    pos  = base + R @ np.array([0.30, 0.0, 0.15])
    look = base + R @ np.array([3.0, 0.0, 0.0])
    return tuple(pos.tolist()), tuple(look.tolist())

print("Batched eval ready (no anchor). RUN §12e-validate FIRST, then run_category_batched(...).")

In [ ]:
# §12e-run  Batched per-category runner. Same signature/logs/drift-handling as run_category,
#           but runs `batch` trials at once. Robust to Genesis' env-frame convention: it
#           measures each env's world offset at reset and does all commit math in world frame.
import math

# drop-in batched replacement for run_category (§12c)
def run_category_batched(arr_name, lc, rc, kind, prompt, n_trials=None, batch=None):
    n_trials = EVAL.N_TRIALS if n_trials is None else n_trials
    batch = EVALB.BATCH if batch is None else batch
    batch = max(1, min(batch, n_trials))
    cat  = f"{arr_name} | {prompt}"
    cdir = f"{EVAL.OUT_DIR}/{_slug(cat)}"; os.makedirs(cdir, exist_ok=True)
    t_path, q_path, txt_path = f"{cdir}/trials.jsonl", f"{cdir}/queries.jsonl", f"{cdir}/log.txt"
    open(t_path, "w").close(); open(q_path, "w").close()
    print(f"\n######## {cat}  ({kind})  n={n_trials}  batch={batch} ########")
    env, left, right = build_arena_batched(lc, rc, num_envs=batch, env_spacing=EVALB.ENV_SPACING)
    policy = load_policy(env)
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(EVAL.MAX_SECONDS / env.dt)
    base_xy = np.array(CFG.SIM.base_init_pos[:2])
    Lxy, Rxy = np.array(left["pos"][:2]), np.array(right["pos"][:2])
    recs, blocks, done_trials = [], [], 0

    # write env i's low-pass-filtered command into the shared commands tensor
    def _set_cmd(c):
        env.commands[:] = torch.as_tensor(c, device=env.commands.device, dtype=env.commands.dtype)

    while done_trials < n_trials:
        cur = min(batch, n_trials - done_trials)            # how many of this chunk to KEEP
        planner_batch.reset(batch)
        obs, _ = env.reset()
        offs = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[:, :2] - base_xy   # per-env world offset
        tgtL, tgtR = Lxy[None, :] + offs, Rxy[None, :] + offs
        cmd = np.zeros((batch, 3)); reached = [None]*batch; commit_step = [None]*batch
        seeds = [EVAL.SEED_BASE + 13*(done_trials + i) for i in range(batch)]
        qlogs = [[] for _ in range(batch)]
        with torch.no_grad():
            for t in range(max_steps):
                if t % query_every == 0:
                    frames = []
                    for i in range(batch):
                        set_level_pose(env.camera, *onboard_pose_i(env, i))
                        rgb, _ = U.render_camera(env.camera)
                        frames.append(PILImage.fromarray(rgb[..., :3].astype("uint8")).convert("RGB"))
                    res = planner_batch.act_batch(frames, [prompt]*batch, sim_time=t*env.dt,
                                                  temperature=EVAL.TEMP, seed=seeds[0] + t)
                    for i in range(batch):
                        r = res[i]
                        cmd[i] = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]]) + (1-CFG.LOWPASS_ALPHA)*cmd[i]
                        if reached[i] is None:
                            sem = bucketize(r["v_x"], r["v_y"], r["w_z"])
                            qlogs[i].append(dict(step=t, parse_status=r["parse_status"],
                                v_x=r["v_x"], v_y=r["v_y"], w_z=r["w_z"], semantic=sem["label"],
                                choice=r.get("choice"), reasoning=r.get("reasoning"),
                                language_memory=r.get("language_memory"),
                                n_short_frames=r.get("n_short_frames"), used_video=r.get("used_video")))
                c = cmd.copy()
                c[:, 0] = np.clip(c[:, 0], *CFG.CMD.lin_vel_x_range)
                c[:, 1] = np.clip(c[:, 1], *CFG.CMD.lin_vel_y_range)
                c[:, 2] = np.clip(c[:, 2], *CFG.CMD.ang_vel_range)
                for i in range(batch):
                    if reached[i] is not None: c[i] = 0.0          # park committed envs
                _set_cmd(c)
                act = policy(obs); obs, _, done, _ = env.step(act)
                _set_cmd(c)                                          # step resamples -> re-set
                rpos = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[:, :2]
                for i in range(batch):
                    if reached[i] is None:
                        if np.linalg.norm(rpos[i] - tgtL[i]) < CFG.ARENA.commit_radius:
                            reached[i], commit_step[i] = "left", t
                        elif np.linalg.norm(rpos[i] - tgtR[i]) < CFG.ARENA.commit_radius:
                            reached[i], commit_step[i] = "right", t
                if all(reached[i] is not None for i in range(cur)):
                    break
        rpos = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[:, :2]
        for i in range(cur):
            if reached[i] is not None:
                outcome, side = "reached", reached[i]
            else:
                outcome = "drift"
                side = "left" if np.linalg.norm(rpos[i]-tgtL[i]) <= np.linalg.norm(rpos[i]-tgtR[i]) else "right"
            chosen = left if side == "left" else right
            cname = color_name(chosen["color"]); gi = done_trials + i
            pr = sum(q["parse_status"] != "parse_failed" for q in qlogs[i]) / max(1, len(qlogs[i]))
            rec = dict(category=cat, arrangement=arr_name, kind=kind, prompt=prompt, trial=gi,
                       seed=seeds[i], outcome=outcome, chosen_side=side, chosen_color=cname,
                       commit_step=commit_step[i], parse_rate=pr, n_queries=len(qlogs[i]),
                       final_memory=planner_batch._lang[i])
            recs.append(rec)
            with open(t_path, "a") as f: f.write(json.dumps(rec) + "\n")
            with open(q_path, "a") as f:
                for q in qlogs[i]: f.write(json.dumps(dict(category=cat, trial=gi, seed=seeds[i], **q)) + "\n")
            block = [f"--- trial {gi} (seed {seeds[i]}) -> {outcome.upper()} chose {cname}/{side} "
                     f"| parse={pr:.2f} | final_mem: {planner_batch._lang[i]!r}"]
            for q in qlogs[i]:
                block.append(f"  step {q['step']:>4} | ({q['v_x']:+.2f},{q['v_y']:+.2f},"
                             f"{q['w_z']:+.2f}) -> {q['semantic']:<13} choice={q['choice']}")
                if q["reasoning"]:       block.append(f"        reason: {q['reasoning']}")
                if q["language_memory"]: block.append(f"        mem   : {q['language_memory']}")
            blocks.append("\n".join(block))
            print(f"  trial {gi+1}/{n_trials}: {outcome:7s} -> {cname}/{side}")
        done_trials += cur
    with open(txt_path, "w") as f:
        f.write("=" * 92 + f"\nCATEGORY: {cat}  (n={len(blocks)}, kind={kind})\n" + "=" * 92 + "\n")
        f.write("\n\n".join(blocks) + "\n")
    json.dump(dict(category=cat, kind=kind, n=len(recs),
                   reached=sum(r["outcome"] == "reached" for r in recs),
                   red=sum(r["chosen_color"] == "red" for r in recs),
                   left=sum(r["chosen_side"] == "left" for r in recs)),
              open(f"{cdir}/summary.json", "w"), indent=2)
    del env, policy; gc.collect(); torch.cuda.empty_cache()
    print(f"  wrote logs -> {cdir}")
    return recs

print("run_category_batched ready. Swap run_category -> run_category_batched in §12c-1..6 cells.")

In [ ]:
# §12e-validate — kept commented out: the per-env render check passed on this setup.
#                 Un-comment the block to re-verify if Genesis / the env layout changes.
# # §12e-validate  DECISIVE check — RUN FIRST. Drives each env DIFFERENTLY for ~1.5 s so they
# #                diverge, THEN renders each env's onboard view. The panels MUST now look
# #                DIFFERENT and match each env's motion. The reset-time view is NOT enough: at
# #                reset all envs are identical, so it can't reveal whether rendering tracks each
# #                env. If the pixel-diffs below are ~0, the render is showing ONE env for all of
# #                them -> do NOT use the batched runner; use §12c (sequential) instead.
# import gc, itertools
# import numpy as np, torch
# import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
# from IPython.display import Image as IPImage, display

# _env, _l, _r = build_arena_batched(RED, BLUE, num_envs=4, env_spacing=EVALB.ENV_SPACING)
# _pol = load_policy(_env); _obs, _ = _env.reset()
# _cmds = np.array([[1.5, 0.0, 0.0],     # env0: straight in
#                   [1.0, 0.0, 0.8],     # env1: curve left
#                   [1.0, 0.0, -0.8],    # env2: curve right
#                   [0.0, 0.0, 1.2]], float)  # env3: spin in place
# for _ in range(75):                    # ~1.5 s at 50 Hz
#     _env.commands[:] = torch.as_tensor(_cmds, device=_env.commands.device, dtype=_env.commands.dtype)
#     _obs, _, _, _ = _env.step(_pol(_obs))
#     _env.commands[:] = torch.as_tensor(_cmds, device=_env.commands.device, dtype=_env.commands.dtype)
# print("per-env LOCAL pos after diverging (should now DIFFER per env):")
# print(np.asarray(_env.robot.get_pos().cpu()).reshape(-1, 3).round(2).tolist())

# imgs = []
# fig, ax = plt.subplots(1, 4, figsize=(16, 3))
# for i in range(4):
#     set_level_pose(_env.camera, *onboard_pose_i(_env, i))
#     rgb, _ = U.render_camera(_env.camera)
#     imgs.append(rgb[..., :3].astype("uint8"))
#     ax[i].imshow(imgs[-1]); ax[i].set_title(f"env {i}"); ax[i].axis("off")
# plt.tight_layout(); plt.savefig("/content/batch_views_decisive.png", dpi=90); plt.close()
# display(IPImage("/content/batch_views_decisive.png"))

# diffs = [float(np.mean(np.abs(imgs[a].astype(int) - imgs[b].astype(int))))
#          for a, b in itertools.combinations(range(4), 2)]
# print("mean abs pixel diff between env pairs:", [round(d, 1) for d in diffs])
# print(">> ~0 everywhere  -> render is NOT per-env (all showing one env): use §12c, not §12e.")
# print(">> clearly >0 and each panel matches its env's motion -> batched render is GOOD.")
# del _env, _pol; gc.collect(); torch.cuda.empty_cache()

### §12c — per-category runs
Run the **§12c setup** and **§12e batched-runner** cells above once per session, then run any category cell below — each calls `run_category_batched` (N=50 trials, batches of 6) and writes its own logs under `…/mem_eval/<slug>/`. Re-run §12d to aggregate.

In [ ]:
# §12c-1  L=red,R=blue | friendly  (semantic)
_ = run_category_batched("L=red,R=blue", RED, BLUE, "semantic", CFG.AMBIGUOUS_PROMPTS[0])


In [ ]:
# §12c-2  L=red,R=blue | hostile   (semantic)
_ = run_category_batched("L=red,R=blue", RED, BLUE, "semantic", CFG.AMBIGUOUS_PROMPTS[1])


In [ ]:
# §12c-3  L=blue,R=red | friendly  (semantic)
_ = run_category_batched("L=blue,R=red", BLUE, RED, "semantic", CFG.AMBIGUOUS_PROMPTS[0])


In [ ]:
# §12c-4  L=blue,R=red | hostile   (semantic)
_ = run_category_batched("L=blue,R=red", BLUE, RED, "semantic", CFG.AMBIGUOUS_PROMPTS[1])


In [ ]:
# §12c-5  L=red,R=red  | friendly  (CONTROL)
_ = run_category_batched("L=red,R=red", RED, RED, "control", CFG.AMBIGUOUS_PROMPTS[0])


In [ ]:
# §12c-6  L=red,R=red  | hostile   (CONTROL)
_ = run_category_batched("L=red,R=red", RED, RED, "control", CFG.AMBIGUOUS_PROMPTS[1])


In [ ]:
# §12d  Statistics. AGGREGATES whatever per-category logs exist under OUT_DIR/*/trials.jsonl,
#       so it works after running one category or all of them. Proportion + Wilson 95% CI +
#       binomial test; mirror decomposition (colour vs side); control = baseline side bias.
import os, json, glob, math, csv
from collections import defaultdict
try:
    from scipy.stats import binomtest, chi2_contingency
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

OUT_DIR = f"{CFG.TRIAL_DIR}/mem_eval"
PROMPTS = list(CFG.AMBIGUOUS_PROMPTS)
trials = []
for tf in sorted(glob.glob(f"{OUT_DIR}/*/trials.jsonl")):
    trials += [json.loads(l) for l in open(tf) if l.strip()]

# Wilson score confidence interval for a binomial proportion
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

# two-sided binomial test p-value against 0.5
def binom_p(k, n):
    if n == 0:
        return float("nan")
    if HAVE_SCIPY:
        return float(binomtest(k, n, 0.5).pvalue)
    z = abs(k - n / 2) / math.sqrt(n * 0.25 + 1e-9)
    return float(2 * (1 - 0.5 * (1 + math.erf(z / math.sqrt(2)))))

if not trials:
    print(f"No trial logs found under {OUT_DIR}. Run §12c setup + at least one category cell first.")
else:
    by_cat = defaultdict(list)
    for t in trials:
        by_cat[t["category"]].append(t)
    N = max(len(v) for v in by_cat.values())
    half = 1.96 * math.sqrt(0.25 / max(1, N))
    print(f"Loaded {len(trials)} trials across {len(by_cat)} categories. "
          f"Largest N={N} -> worst-case 95% CI half-width +/-{half:.3f} (p=0.5).\n")

    print(f"{'CATEGORY':40s} {'n':>3} {'reach':>6} {'red% [95% CI]':>18} {'left% [95% CI]':>18}")
    rows = []
    for cat, ts in by_cat.items():
        n = len(ts)
        reach = sum(t["outcome"] == "reached" for t in ts)
        kred  = sum(t["chosen_color"] == "red" for t in ts)
        kleft = sum(t["chosen_side"] == "left" for t in ts)
        pr, _, _ = wilson(reach, n); rp, rlo, rhi = wilson(kred, n); lp, llo, lhi = wilson(kleft, n)
        print(f"{cat:40s} {n:>3} {100*pr:>5.0f}% "
              f"{100*rp:>4.0f}%[{100*rlo:>3.0f},{100*rhi:>3.0f}] "
              f"{100*lp:>5.0f}%[{100*llo:>3.0f},{100*lhi:>3.0f}]")
        rows.append(dict(category=cat, n=n, reach_rate=round(pr, 3),
                         red_rate=round(rp, 3), red_lo=round(rlo, 3), red_hi=round(rhi, 3),
                         left_rate=round(lp, 3), left_lo=round(llo, 3), left_hi=round(lhi, 3),
                         p_red_vs_chance=round(binom_p(kred, n), 4),
                         p_left_vs_chance=round(binom_p(kleft, n), 4)))

    print("\n--- Mirror decomposition (semantic conditions pooled across L/R arrangements) ---")
    print("    RED% high & LEFT% ~50% => colour-driven;  LEFT% extreme & RED% ~50% => side-driven.")
    for prompt in PROMPTS:
        ts = [t for t in trials if t["kind"] == "semantic" and t["prompt"] == prompt]
        n = len(ts)
        if n == 0:
            print(f"  {prompt:26s} (no semantic trials yet)"); continue
        kred = sum(t["chosen_color"] == "red" for t in ts)
        kleft = sum(t["chosen_side"] == "left" for t in ts)
        rp, rlo, rhi = wilson(kred, n); lp, llo, lhi = wilson(kleft, n)
        print(f"  {prompt:26s} n={n:>3} | RED {100*rp:>4.0f}%[{100*rlo:>3.0f},{100*rhi:>3.0f}] "
              f"p={binom_p(kred,n):.3g} | LEFT {100*lp:>4.0f}%[{100*llo:>3.0f},{100*lhi:>3.0f}] "
              f"p={binom_p(kleft,n):.3g}")

    if HAVE_SCIPY and len(PROMPTS) >= 2:
        fr = [t for t in trials if t["kind"] == "semantic" and t["prompt"] == PROMPTS[0]]
        ho = [t for t in trials if t["kind"] == "semantic" and t["prompt"] == PROMPTS[1]]
        if fr and ho:
            table = [[sum(t["chosen_color"] == "red" for t in fr), sum(t["chosen_color"] == "blue" for t in fr)],
                     [sum(t["chosen_color"] == "red" for t in ho), sum(t["chosen_color"] == "blue" for t in ho)]]
            try:
                chi2, p, _, _ = chi2_contingency(table)
                print(f"\nPrompt-word effect (friendly vs hostile colour choice): "
                      f"chi2={chi2:.2f} p={p:.3g}  table[red,blue]={table}")
            except Exception as e:
                print("chi-square skipped:", e)

    ctrl = [t for t in trials if t["kind"] == "control"]
    if ctrl:
        n = len(ctrl); kleft = sum(t["chosen_side"] == "left" for t in ctrl)
        lp, llo, lhi = wilson(kleft, n)
        print("\n--- Control (identical targets) baseline SIDE bias ---")
        print(f"  LEFT-rate {100*lp:.0f}%[{100*llo:.0f},{100*lhi:.0f}] "
              f"p(vs 0.5)={binom_p(kleft,n):.3g}  (n={n}; deviation from 50% = positional bias)")

    if rows:
        with open(f"{OUT_DIR}/mem_eval_summary.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
        print(f"\nsummary -> {OUT_DIR}/mem_eval_summary.csv")

### §12f — showcase videos (MEM-V2 planner)
Two example MP4s (one per ambiguous prompt) driven by the **MEM-V2** planner — recent frames ~0.33 s apart (no anchor) + compressed long-term language memory — with the evolving reasoning/memory printed per query. Self-contained; needs §12a run first.

In [ ]:
# §12f  Showcase videos with the MEM-V2 planner (recent frames ~0.33s apart, no anchor, +
#       compressed long-term LANGUAGE memory) — one MP4 per ambiguous prompt, with the
#       evolving reasoning/memory printed per query. Self-contained: builds its own arena.
#       Deps: §10a (poses), §10b (build_arena/train_cfg), §8 (bucketize), §12a (planner_v2).
import os, shutil, copy as _copy, gc
import numpy as np, imageio.v2 as imageio
from PIL import Image as PILImage
import random

def _load_policy(env):                 # robust to repeated builds (rsl-rl pops class_name)
    tc = _copy.deepcopy(train_cfg)
    tc["policy"].setdefault("class_name", "ActorCritic")
    tc["algorithm"].setdefault("class_name", "PPO")
    ck = [f for f in os.listdir(log_dir) if f.startswith("model_")]
    latest = max(int(f[6:-3]) for f in ck)
    runner = OnPolicyRunner(env, tc, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f"model_{latest}.pt"))
    print("Loaded checkpoint", latest)
    return runner.get_inference_policy(device=gs.device)

# human-readable red/blue label for an RGB colour
def _color_label(c):
    r = np.array(CFG.COLOR_RED); b = np.array(CFG.COLOR_BLUE); c = np.array(c[:3])
    return "red" if np.linalg.norm(c - r) <= np.linalg.norm(c - b) else "blue"

# run and record one narrated demo trial for the showcase video
def run_showcase(env, policy, planner, left, right, instruction,
                 temperature=CFG.VLM_TEMPERATURE, seed=10, max_seconds=6.0,
                 record_path=None, record_fps=50):
    commit_radius = CFG.ARENA.commit_radius
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(max_seconds / env.dt)
    planner.reset(); obs, _ = env.reset()
    cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, _ = U.render_camera(env.camera); last_rgb = rgb
    queries = []; reached = None; commit_step = None; rec_frames = []
    with torch.no_grad():
        for t in range(max_steps):
            if t % query_every == 0:
                frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
                r = planner.act(frame, instruction, sim_time=t * env.dt,    # V2 needs sim_time
                                temperature=temperature, seed=seed + t)
                cmd = CFG.LOWPASS_ALPHA * np.array([r["v_x"], r["v_y"], r["w_z"]]) + (1 - CFG.LOWPASS_ALPHA) * cmd
                sem = bucketize(r["v_x"], r["v_y"], r["w_z"])
                queries.append(dict(step=t, parse_status=r["parse_status"],
                    v_x=r["v_x"], v_y=r["v_y"], w_z=r["w_z"], semantic=sem["label"],
                    choice=r.get("choice"), reasoning=r.get("reasoning"),
                    language_memory=r.get("language_memory"),
                    n_short_frames=r.get("n_short_frames"), used_video=r.get("used_video")))
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            act = policy(obs); obs, _, done, _ = env.step(act)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)      # onboard for next query
            rgb, _ = U.render_camera(env.camera); last_rgb = rgb
            if record_path:                                                  # iso frame for the video
                pr, lr = iso_pose(env); set_level_pose(env.camera, pr, lr)
                iso_rgb, _ = U.render_camera(env.camera)
                rec_frames.append(iso_rgb[..., :3].astype("uint8"))
            p2 = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0][:2]
            for obj in (left, right):
                if np.linalg.norm(p2 - np.array(obj["pos"][:2])) < commit_radius:
                    reached, commit_step = obj, t; break
            if reached is not None:
                break
    if record_path and rec_frames:
        imageio.mimwrite(record_path, rec_frames, fps=record_fps, macro_block_size=None)
    cam_xy = np.array(onboard_pose(env)[0][:2])
    if reached is not None:
        outcome, chosen = "reached", reached
    else:
        outcome = "drift"
        chosen = left if np.linalg.norm(cam_xy - np.array(left["pos"][:2])) <= \
                         np.linalg.norm(cam_xy - np.array(right["pos"][:2])) else right
    return dict(outcome=outcome, chosen_side=chosen["side"], chosen_color=chosen["color"],
                commit_step=commit_step, queries=queries, final_memory=planner.language_memory)

# ---- run one showcase video per ambiguous prompt -----------------------------------------
red  = CFG.COLOR_RED
blue = U.luminance_match(CFG.COLOR_RED, CFG.COLOR_BLUE)
env, left, right = build_arena(left_color=red, right_color=blue)
policy = _load_policy(env)

demos = []
for i, instr in enumerate(CFG.AMBIGUOUS_PROMPTS):
    demo_seed = random.randint(0, 10**6)              # <-- capture so it can be printed/reused
    out = f"/content/mem2_demo_{i}_seed{demo_seed}.mp4"
    print(f"\n=== Trial [{instr}] | seed={demo_seed} (running...) ===")
    res = run_showcase(env, policy, planner_v2, left, right, instr,
                       temperature=CFG.VLM_TEMPERATURE, seed=demo_seed, max_seconds=6.0,
                       record_path=out, record_fps=50)
    shutil.copy(out, f"{CFG.VIDEO_DIR}/{os.path.basename(out)}")    # Drive copy carries the seed
    cname = _color_label(res["chosen_color"])
    print(f"=== Trial [{instr}] | seed={demo_seed} -> {res['outcome']} chose {cname}/{res['chosen_side']} ===")
    for q in res["queries"]:
        vid = "video" if q["used_video"] else "imgs"
        print(f"  step {q['step']:>4} | {q['parse_status']:<15} | "
              f"(v_x={q['v_x']:+.2f},v_y={q['v_y']:+.2f},w_z={q['w_z']:+.2f}) -> {q['semantic']:<13} "
              f"| short:{q['n_short_frames']}({vid})")
        if q["reasoning"]:       print(f"        reasoning: {q['reasoning']}")
        if q["language_memory"]: print(f"        memory   : {q['language_memory']}")
    print(f"  >> final long-term memory: {res['final_memory']!r}")
    demos.append((instr, out, res, demo_seed))         # <-- seed stored alongside the demo

del env, policy; gc.collect(); torch.cuda.empty_cache()

# ---- recap: prompt -> seed (copy these if you want to reproduce a run) --------------------
print("\n================ demo seeds ================")
for instr, out, res, demo_seed in demos:
    print(f"  seed={demo_seed:<7} | {instr}  -> {_color_label(res['chosen_color'])}/{res['chosen_side']} ({res['outcome']})")

import mediapy as media
for instr, out, res, demo_seed in demos:
    print(f"\n{instr} | seed={demo_seed} -> {_color_label(res['chosen_color'])}/{res['chosen_side']} ({res['outcome']})")
    media.show_video(media.read_video(out), fps=50)

## §13 — Per-category 3D saliency heatmaps (attention/IG projected into the arena)

For each trial category we run a few **IG-instrumented** rollouts, take the IG
saliency of `w_z` on the current onboard frame at sampled steps, and **unproject every
salient pixel through its depth** into world coordinates using the §5 camera geometry.
Pixels that land on the **floor or target faces** become weighted 3D points; **sky/far
pixels** have no finite depth and are **projected onto the back wall** instead of being
discarded. Accumulated over the rollouts this yields, per category, both a 3D scatter on the
real surfaces and a top-down accumulated "terrain" heatmap.

> This part **does render** (it needs depth + IG), so it is intentionally slow. Keep
> `HEAT.TRIALS`, `HEAT.IG_STEPS`, and `HEAT.SAMPLE_EVERY_Q` small. A **projection sanity
> check** prints where the top-decile saliency centroid lands relative to each target; if
> it comes out mirrored, flip `PROJ.FLIP_U` / `PROJ.FLIP_V` in §13a and re-run.


In [ ]:
# §13a  Project per-pixel saliency into the 3D arena using DEPTH + camera geometry.
#       Each pixel that hits a surface (floor / target faces) becomes a 3D point weighted
#       by its saliency; sky / far pixels (no finite depth < MAX_DEPTH) are intersected
#       with the back wall (PROJ.HORIZON_TO_WALL) so horizon attention is preserved
#       rather than dropped.
import numpy as np
import mem_planner as MP
import torch
from PIL import Image as PILImage
from qwen_vl_utils import process_vision_info

# IG-on-command (defined here so §13 is fully self-contained)
def integrated_gradients_on_command(planner, frame, instruction, component="w_z",
                                    steps=16, temperature=0.0, seed=0):
    """IG of the first numeric token's logit (for `component`) w.r.t. input image pixels.
    Returns (saliency_HxW float32 in [0,1], decoded_text, semantic_label)."""
    model, processor = planner.model, planner.processor
    torch.manual_seed(seed)

    # 1) Greedy-decode once to locate the numeric token for `component` and its target id.
    messages, _used_video = planner._build_messages(frame, instruction)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    base_inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                            padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(**base_inputs, max_new_tokens=planner.max_new_tokens,
                             do_sample=False, return_dict_in_generate=True)
    gen_ids = gen.sequences[0][base_inputs.input_ids.shape[1]:]
    dec = [processor.tokenizer.decode([t]) for t in gen_ids.tolist()]
    decoded_text = processor.decode(gen_ids, skip_special_tokens=True)

    key = "v_x" if component == "v_x" else ("v_y" if component == "v_y" else "w_z")
    tgt_pos = None
    for i, tok in enumerate(dec):
        if any(ch.isdigit() for ch in tok):
            ctx = "".join(dec[max(0, i - 8):i]).lower().replace("_", "")
            if key.replace("_", "") in ctx:
                tgt_pos = i; break
    used_fallback = tgt_pos is None
    if used_fallback:                         # fallback: first numeric token anywhere
        for i, tok in enumerate(dec):
            if any(ch.isdigit() for ch in tok):
                tgt_pos = i; break
    if tgt_pos is None:
        raise RuntimeError("no numeric output token found for IG target")

    # Expand to the FULL signed number: optional leading sign token + digits + "." + digits.
    # Qwen splits "-0.5" as [" -", "0", ".", "5"]; the bare sign token holds no digit, so the
    # old code dropped it and attributed only the leading "0" (identical for +0.5 and -0.5).
    _SIGN, _NUM = set("+-"), set("0123456789.")
    start = tgt_pos
    if tgt_pos > 0:
        prev = dec[tgt_pos - 1].strip()
        if prev and all(c in _SIGN for c in prev):
            start = tgt_pos - 1               # include the sign token (" -", "-")
    end = tgt_pos
    while end + 1 < len(dec) and dec[end + 1].strip() and all(c in _NUM for c in dec[end + 1].strip()):
        end += 1                              # include trailing ".", digits
    span = list(range(start, end + 1))
    span_str = "".join(dec[p] for p in span).strip()

    # Teacher-forced sequence through the END of the numeric span.
    full_ids = torch.cat([base_inputs.input_ids[0], gen_ids[:end + 1]]).unsqueeze(0)
    prompt_len = base_inputs.input_ids.shape[1]
    span_li  = [prompt_len + p - 1 for p in span]   # logit row that predicts each span token
    span_ids = [int(gen_ids[p]) for p in span]
    _flag = f"  !! FALLBACK: '{component}' label not found in output -> attributing first number instead" if used_fallback else ""
    print(f"   IG target [{component}] = '{span_str}'  ({len(span)} tok, sign-inclusive){_flag}")

    pv = base_inputs["pixel_values"]           # (num_patches, feat) flattened patch pixels
    baseline = torch.zeros_like(pv)
    grid_thw = base_inputs["image_grid_thw"] if "image_grid_thw" in base_inputs else base_inputs["video_grid_thw"]

    total_grad = torch.zeros_like(pv, dtype=torch.float32)
    # transformers v5 only computes the correct 2D image m-rope when `mm_token_type_ids`
    # (0=text, 1=image, 2=video) is supplied AND matches the sequence length. We teacher-force
    # the answer onto `full_ids`, so extend the prompt-length copy with text(0) for the appended
    # tokens and pass it -> attribution then uses the SAME positions as generation. Omitting it
    # silently falls back to flat positions; a length-mismatched copy raises the get_rope_index
    # mask IndexError. (transformers 4.x omits this key, so behaviour there is unchanged.)
    fwd_extra = {}
    if "mm_token_type_ids" in base_inputs:
        _mtt  = base_inputs["mm_token_type_ids"]
        _npad = full_ids.shape[1] - _mtt.shape[1]
        fwd_extra["mm_token_type_ids"] = (
            torch.cat([_mtt, _mtt.new_zeros((_mtt.shape[0], _npad))], dim=1) if _npad > 0 else _mtt)
    with torch.enable_grad():                  # IG needs grad even if caller is under no_grad
        for s in range(1, steps + 1):
            alpha = s / steps
            pv_s = (baseline + alpha * (pv - baseline)).clone().requires_grad_(True)
            out = model(input_ids=full_ids,
                        attention_mask=torch.ones_like(full_ids),
                        pixel_values=pv_s, image_grid_thw=grid_thw, **fwd_extra)
            rows = out.logits[0][span_li].float()                 # (n_span, vocab)
            tids = torch.tensor(span_ids, device=rows.device)
            logp = rows.gather(1, tids[:, None]).squeeze(1) - torch.logsumexp(rows, dim=-1)
            scalar = logp.sum()                                   # log P(full signed number | image)
            grad, = torch.autograd.grad(scalar, pv_s, retain_graph=False)
            total_grad += grad.float()

    ig = ((pv - baseline).float() * total_grad / steps)   # (num_patches, feat)
    patch_sal = ig.abs().sum(dim=-1).detach().cpu().numpy()

    # reshape patches -> 2D grid using grid_thw (last frame), then upsample to image size
    t, h, w = [int(x) for x in grid_thw[0].tolist()]  # last frame used below
    per = h * w
    patch_sal = patch_sal[-per:] if patch_sal.shape[0] >= per else np.pad(patch_sal, (0, per - patch_sal.shape[0]))
    sal = patch_sal.reshape(h, w)
    sal = (sal - sal.min()) / (np.ptp(sal) + 1e-9)
    sal_img = np.asarray(PILImage.fromarray((sal * 255).astype('uint8')).resize(frame.size))

    sem = bucketize(*[_grab_float(decoded_text, k) for k in ("v_x", "v_y", "w_z")])["label"]
    return sal_img.astype(np.float32) / 255.0, decoded_text, sem

# regex-extract a named float out of generated text
def _grab_float(text, name):
    import re
    m = re.search(rf'{name}"?\s*[:=]\s*([-+]?\d*\.?\d+)', text, flags=re.IGNORECASE)
    return float(m.group(1)) if m else 0.0

# Shared scene constants (back wall = far +x plane; two viewing angles).
X_WALL = CFG.ARENA.distance + 1.3
Z_TOP  = 1.05
Y_HALF = CFG.ARENA.lateral + 1.4
VIEWS  = ((30, 130), (30, 250))          # sketch angle + complementary angle

PROJ = SimpleNamespace(
    MAX_DEPTH=CFG.ARENA.distance + CFG.ARENA.lateral + 3.0,  # beyond this = sky/horizon
    WEIGHT_THRESH=0.0,            # keep ALL hits -> full coverage; saliency = colour
    MAX_POINTS=60000,            # cap projected points per frame (memory / speed)
    HORIZON_TO_WALL=True,        # project sky/horizon saliency onto the back wall (keep data)
    FLIP_U=False, FLIP_V=False)

# onboard camera intrinsics matrix
def onboard_K():
    return U.camera_intrinsics(CFG.CAM.res, CFG.CAM.fov)

def project_saliency(sal, depth, cam_pos, lookat, K, max_depth=None, wthresh=None):
    """Unproject per-pixel saliency. Surface pixels -> their 3D hit point. Sky/horizon
    pixels (no finite depth < MAX_DEPTH) are intersected with the back wall (x=X_WALL)
    instead of being discarded, so horizon attention is preserved. Returns (XYZ Nx3, w N)."""
    from PIL import Image as PILImage
    max_depth = PROJ.MAX_DEPTH if max_depth is None else max_depth
    wthresh = PROJ.WEIGHT_THRESH if wthresh is None else wthresh
    d = np.asarray(depth, float)
    if d.ndim == 3: d = d[..., 0]
    H, W = d.shape[:2]
    sal = np.asarray(sal, float)
    if sal.shape[:2] != (H, W):
        sal = np.asarray(PILImage.fromarray((np.clip(sal, 0, 1)*255).astype("uint8")).resize((W, H)))/255.0
    if sal.ndim == 3: sal = sal[..., 0]
    _, T_wc = U.look_at_extrinsics(cam_pos, lookat, up=(0, 0, 1))
    R = T_wc[:3, :3]; cam = T_wc[:3, 3]
    vv, uu = np.mgrid[0:H, 0:W].astype(float)
    if PROJ.FLIP_U: uu = (W - 1) - uu
    if PROJ.FLIP_V: vv = (H - 1) - vv
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    w = sal.reshape(-1); dd = d.reshape(-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        x = (uu - cx)/fx*d; y = (vv - cy)/fy*d; z = d
        world = (R @ np.stack([x, y, z], -1).reshape(-1, 3).T).T + cam
    surf = np.isfinite(dd) & (dd > 1e-3) & (dd < max_depth) & (w > wthresh)
    outP, outW = [world[surf]], [w[surf]]
    if getattr(PROJ, "HORIZON_TO_WALL", True):                 # paint sky/horizon on the back wall
        dirw = (R @ np.stack([(uu - cx)/fx, (vv - cy)/fy, np.ones_like(uu)], -1).reshape(-1, 3).T).T
        fwd = dirw[:, 0] > 1e-3
        sky = ((~np.isfinite(dd)) | (dd >= max_depth)) & (w > wthresh) & fwd
        with np.errstate(invalid="ignore", divide="ignore"):
            t = (X_WALL - cam[0]) / np.where(fwd, dirw[:, 0], 1.0)
            wp = cam + t[:, None]*dirw
        good = sky & (t > 0) & (np.abs(wp[:, 1]) <= Y_HALF) & (wp[:, 2] >= -0.2) & (wp[:, 2] <= Z_TOP + 0.4)
        wp = wp[good]
        if len(wp):
            wp[:, 0] = X_WALL; wp[:, 2] = np.clip(wp[:, 2], 0.05, Z_TOP)
        outP.append(wp); outW.append(w[good])
    Wm = np.concatenate(outP); wm = np.concatenate(outW)
    cap = getattr(PROJ, "MAX_POINTS", None)
    if cap and len(Wm) > cap:
        idx = np.random.default_rng(0).choice(len(Wm), cap, replace=False); Wm, wm = Wm[idx], wm[idx]
    return Wm, wm

def make_single_frame_planner():
    """Memory-free single-image planner -> clean IG (one image => image_grid_thw)."""
    return MP.MEMPlanner(model, processor, CFG.CONTRACT,
                         CFG.CMD.lin_vel_x_range, CFG.CMD.lin_vel_y_range, CFG.CMD.ang_vel_range,
                         short_frames=1, short_stride=1, as_video=False,
                         short_enabled=False, lang_enabled=False,
                         max_new_tokens=CFG.VLM_MAX_NEW_TOKENS)

def rollout_project(env, policy, drive_planner, sf_planner, left, right, instruction,
                    component="w_z", ig_steps=8, sample_every_q=4, temperature=0.7,
                    seed=0, max_seconds=10.0, saliency_fn=None, return_path=False):
    """Drive with the MEM-V2 planner; at sampled queries compute an attribution map on
    the CURRENT frame (IG by default, or any `saliency_fn(planner, frame, instruction,
    component=, steps=)`) and project it with that frame's depth. Returns (XYZ, W)."""
    from PIL import Image as PILImage
    sal_fn = saliency_fn or integrated_gradients_on_command
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(max_seconds / env.dt); K = onboard_K()
    drive_planner.reset(); obs, _ = env.reset(); cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
    pts, ws, qn, path = [], [], 0, []
    with torch.no_grad():
        for t in range(max_steps):
            if t % query_every == 0:
                frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
                r = drive_planner.act(frame, instruction, sim_time=t*env.dt,
                                      temperature=temperature, seed=seed + t)
                cmd = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]], float) + (1-CFG.LOWPASS_ALPHA)*cmd
                if depth is not None and qn % sample_every_q == 0:
                    sf_planner.reset()
                    try:
                        sal, _, _ = sal_fn(sf_planner, frame, instruction, component=component, steps=ig_steps)
                        cp, cl = onboard_pose(env)
                        P, Wt = project_saliency(sal, depth, cp, cl, K)
                        if len(P): pts.append(P); ws.append(Wt)
                    except Exception as e:
                        print("   proj skip:", repr(e))
                qn += 1
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            a = policy(obs); obs, _, done, _ = env.step(a)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
            p2 = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0][:2]
            path.append(p2.copy())
            if any(np.linalg.norm(p2 - np.array(o["pos"][:2])) < CFG.ARENA.commit_radius for o in (left, right)):
                break
    P = np.concatenate(pts, 0) if pts else np.zeros((0, 3))
    Wt = np.concatenate(ws, 0) if ws else np.zeros((0,))
    pa = np.asarray(path, float) if path else np.zeros((0, 2))
    if return_path:
        return P, Wt, pa
    return P, Wt

# ---- Two-angle painted render: floor + cube faces + BACK WALL (horizon) -------
def _smooth(a, sigma):
    if sigma <= 0:
        return a
    try:
        from scipy.ndimage import gaussian_filter
        return gaussian_filter(a, sigma=sigma, mode="constant")
    except Exception:
        k = max(1, int(round(sigma))); ker = np.ones(2 * k + 1) / (2 * k + 1)
        out = a.astype(float).copy()
        for ax_ in (0, 1):
            out = np.apply_along_axis(lambda m: np.convolve(m, ker, mode="same"), ax_, out)
        return out


def render_heatmap_3d(XYZ, W, left, right, title, save_prefix,
                      grid_bins=72, face_bins=7, wall_bins=(60, 26), cmap_name="turbo",
                      views=VIEWS, label_height=0.95, smooth_sigma=1.1,
                      slab=0.06, robust_q=0.99, paths=None):
    """Static render of the scene with accumulated saliency painted on the floor, the
    target-cube faces (wrapped volume, black edges), AND the far back wall (where the
    VLM's horizon/sky attention is projected, so no data is silently discarded).
    Two side-by-side viewing angles capture the whole scene. White = genuine blind spot."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    from mpl_toolkits.mplot3d import proj3d                       # noqa: F401
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection

    try:
        cmap = matplotlib.colormaps[cmap_name]
    except Exception:
        cmap = plt.cm.get_cmap(cmap_name)
    norm = mcolors.Normalize(0.0, 1.0)
    WHITE = np.array([1.0, 1.0, 1.0, 1.0])

    # clip a goal cube's colour to [0,1] and label it Red/Blue
    def _truecolor(o):
        c = np.clip(np.array(o["color"][:3], float), 0, 1)
        name = ("Red" if np.linalg.norm(c - np.array(CFG.COLOR_RED))
                <= np.linalg.norm(c - np.array(CFG.COLOR_BLUE)) else "Blue")
        return tuple(c), name

    ax0 = np.array(CFG.SIM.base_init_pos, float)
    xr = [-0.6, X_WALL]; yr = [-Y_HALF, Y_HALF]

    XYZ = np.asarray(XYZ, float).reshape(-1, 3); W = np.asarray(W, float).reshape(-1)

    # ---- classify: cube faces / back-wall (horizon) / floor ----------------
    cubes = []
    for o in (left, right):
        col, name = _truecolor(o)
        cx, cy, _ = o["pos"]; sx, sy, sz = o["size"]
        cubes.append(dict(cx=cx, cy=cy, hx=sx/2, hy=sy/2, sz=sz, color=col, name=name))

    rest = np.ones(len(XYZ), bool)
    for cb in cubes:
        inb = ((np.abs(XYZ[:, 0]-cb["cx"]) <= cb["hx"]+slab) &
               (np.abs(XYZ[:, 1]-cb["cy"]) <= cb["hy"]+slab) &
               (XYZ[:, 2] <= cb["sz"]+slab) & (XYZ[:, 2] >= -slab) & (XYZ[:, 2] > 0.03))
        cb["pts"], cb["w"] = XYZ[inb], W[inb]; rest &= ~inb
    wall_m = rest & (np.abs(XYZ[:, 0] - X_WALL) < 1e-3) & (XYZ[:, 2] > 0.03)
    wP, wW = XYZ[wall_m], W[wall_m]
    floor_m = rest & ~wall_m
    fXYZ, fW = XYZ[floor_m], W[floor_m]

    # ---- floor density ------------------------------------------------------
    if len(fXYZ):
        Hsum, xe, ye = np.histogram2d(fXYZ[:, 0], fXYZ[:, 1], bins=grid_bins, range=[xr, yr], weights=fW)
        Hcnt, _, _ = np.histogram2d(fXYZ[:, 0], fXYZ[:, 1], bins=grid_bins, range=[xr, yr])
    else:
        xe = np.linspace(*xr, grid_bins+1); ye = np.linspace(*yr, grid_bins+1)
        Hsum = np.zeros((grid_bins, grid_bins)); Hcnt = np.zeros((grid_bins, grid_bins))
    cnt_s = _smooth(Hcnt.astype(float), smooth_sigma); sum_s = _smooth(Hsum.astype(float), smooth_sigma)
    floor_cov = cnt_s > (0.04 * cnt_s.max() if cnt_s.max() > 0 else 1.0)
    floor_dens = np.zeros_like(Hsum); floor_dens[floor_cov] = sum_s[floor_cov] / (cnt_s[floor_cov] + 1e-9)

    # ---- back-wall density (y, z) ------------------------------------------
    wb_y, wb_z = wall_bins
    if len(wP):
        Wsum, wye, wze = np.histogram2d(wP[:, 1], wP[:, 2], bins=[wb_y, wb_z],
                                        range=[yr, [0.0, Z_TOP]], weights=wW)
        Wcnt, _, _ = np.histogram2d(wP[:, 1], wP[:, 2], bins=[wb_y, wb_z], range=[yr, [0.0, Z_TOP]])
    else:
        wye = np.linspace(*yr, wb_y+1); wze = np.linspace(0, Z_TOP, wb_z+1)
        Wsum = np.zeros((wb_y, wb_z)); Wcnt = np.zeros((wb_y, wb_z))
    wcnt_s = _smooth(Wcnt.astype(float), smooth_sigma); wsum_s = _smooth(Wsum.astype(float), smooth_sigma)
    wall_cov = wcnt_s > (0.04 * wcnt_s.max() if wcnt_s.max() > 0 else 1.0)
    wall_dens = np.zeros_like(Wsum); wall_dens[wall_cov] = wsum_s[wall_cov] / (wcnt_s[wall_cov] + 1e-9)

    # ---- cube face densities ------------------------------------------------
    def _faces(cb):
        cx, cy, hx, hy, sz = cb["cx"], cb["cy"], cb["hx"], cb["hy"], cb["sz"]
        return [(2, sz, (0, cx-hx, cx+hx), (1, cy-hy, cy+hy)),
                (0, cx-hx, (1, cy-hy, cy+hy), (2, 0, sz)), (0, cx+hx, (1, cy-hy, cy+hy), (2, 0, sz)),
                (1, cy-hy, (0, cx-hx, cx+hx), (2, 0, sz)), (1, cy+hy, (0, cx-hx, cx+hx), (2, 0, sz))]

    all_dens = []
    if floor_cov.any(): all_dens.append(floor_dens[floor_cov])
    if wall_cov.any(): all_dens.append(wall_dens[wall_cov])
    for cb in cubes:
        cb["faces"] = []; P, Wt = cb["pts"], cb["w"]
        for (ca, pl, (ia, alo, ahi), (ib, blo, bhi)) in _faces(cb):
            ae = np.linspace(alo, ahi, face_bins+1); be = np.linspace(blo, bhi, face_bins+1)
            if len(P):
                near = np.abs(P[:, ca]-pl) < slab; Pn, Wn = P[near], Wt[near]
            else:
                Pn, Wn = P, Wt
            if len(Pn):
                fs, _, _ = np.histogram2d(Pn[:, ia], Pn[:, ib], bins=[ae, be], weights=Wn)
                fc, _, _ = np.histogram2d(Pn[:, ia], Pn[:, ib], bins=[ae, be])
            else:
                fs = np.zeros((face_bins, face_bins)); fc = np.zeros((face_bins, face_bins))
            cov = fc > 0; dn = np.zeros_like(fs); dn[cov] = fs[cov] / (fc[cov] + 1e-9)
            cb["faces"].append(dict(ca=ca, plane=pl, ia=ia, ib=ib, ae=ae, be=be, dens=dn, cov=cov))
            if cov.any(): all_dens.append(dn[cov])

    pool = np.concatenate(all_dens) if all_dens else np.array([1.0])
    vmax = np.quantile(pool, robust_q) if len(pool) > 5 else (pool.max() if len(pool) else 1.0)
    vmax = vmax if vmax > 1e-9 else 1.0

    # map a density grid to RGBA via the colormap, transparent where uncovered
    def _rgba(dens, cov, blind_transparent):
        rgba = cmap(norm(np.clip(dens / vmax, 0, 1)))
        if blind_transparent: rgba[~cov, 3] = 0.0
        else: rgba[~cov] = WHITE
        return rgba

    # precompute grids / colours (built once, re-added per axis)
    xc = 0.5*(xe[:-1]+xe[1:]); yc = 0.5*(ye[:-1]+ye[1:]); Xg, Yg = np.meshgrid(xc, yc, indexing="ij")
    floor_rgba = _rgba(floor_dens, floor_cov, True)
    wyc = 0.5*(wye[:-1]+wye[1:]); wzc = 0.5*(wze[:-1]+wze[1:]); Yw, Zw = np.meshgrid(wyc, wzc, indexing="ij")
    Xw = np.full_like(Yw, X_WALL); wall_rgba = _rgba(wall_dens, wall_cov, True)

    # 3D corner point of a face's axis-aligned rectangle
    def _corner(ca, pl, ia, av, ib, bv):
        p = [0.0, 0.0, 0.0]; p[ca] = pl; p[ia] = av; p[ib] = bv; return p

    cube_art = []
    for cb in cubes:
        quads, qcols = [], []
        for f in cb["faces"]:
            fr = _rgba(f["dens"], f["cov"], False); ae, be = f["ae"], f["be"]
            for i in range(len(ae)-1):
                for j in range(len(be)-1):
                    quads.append([_corner(f["ca"], f["plane"], f["ia"], ae[i], f["ib"], be[j]),
                                  _corner(f["ca"], f["plane"], f["ia"], ae[i+1], f["ib"], be[j]),
                                  _corner(f["ca"], f["plane"], f["ia"], ae[i+1], f["ib"], be[j+1]),
                                  _corner(f["ca"], f["plane"], f["ia"], ae[i], f["ib"], be[j+1])])
                    qcols.append(fr[i, j])
        cx, cy, hx, hy, sz = cb["cx"], cb["cy"], cb["hx"], cb["hy"], cb["sz"]
        c = np.array([[cx-hx,cy-hy,0],[cx+hx,cy-hy,0],[cx+hx,cy+hy,0],[cx-hx,cy+hy,0],
                      [cx-hx,cy-hy,sz],[cx+hx,cy-hy,sz],[cx+hx,cy+hy,sz],[cx-hx,cy+hy,sz]])
        E = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
        cube_art.append((quads, qcols, [[c[a], c[b]] for a, b in E]))

    objs = [(cb["cx"], cb["cy"], cb["sz"], cb["name"]) for cb in cubes]
    objs.append((ax0[0], ax0[1], 0.0, "Agent"))

    # draw the arena, cubes, floor/wall heatmaps and travelled paths on one 3D axis
    def _add_scene(ax):
        ax.plot_surface(Xg, Yg, np.zeros_like(Xg), facecolors=floor_rgba, rcount=grid_bins,
                        ccount=grid_bins, shade=False, linewidth=0, antialiased=False, zorder=1)
        ax.plot_surface(Xw, Yw, Zw, facecolors=wall_rgba, rcount=wb_y, ccount=wb_z,
                        shade=False, linewidth=0, antialiased=False, zorder=1)
        for (quads, qcols, edges) in cube_art:
            pc = Poly3DCollection(quads, facecolors=qcols, edgecolors="none", shade=False)
            pc.set_zorder(3); ax.add_collection3d(pc)
            lc = Line3DCollection(edges, colors="k", linewidths=1.4); lc.set_zorder(5)
            ax.add_collection3d(lc)
        ax.scatter([ax0[0]], [ax0[1]], [0.02], c="k", marker="o", s=55, depthshade=False, zorder=4)
        if paths:
            from mpl_toolkits.mplot3d.art3d import Line3DCollection as _L3
            _a = float(np.clip(1.2/np.sqrt(max(1, len(paths))), 0.12, 0.85))
            _segs = []
            for _p in paths:
                _p = np.asarray(_p, float).reshape(-1, 2)
                for _i in range(len(_p)-1):
                    _segs.append([(_p[_i,0],_p[_i,1],0.012),(_p[_i+1,0],_p[_i+1,1],0.012)])
            if _segs:
                _lc = _L3(_segs, colors=[(0.10,0.10,0.10,_a)]*len(_segs), linewidths=1.3, linestyles="--")
                _lc.set_zorder(6); ax.add_collection3d(_lc)
        for (x, y, ztop, _) in objs:
            ax.plot([x, x], [y, y], [ztop, label_height], color="k", lw=1.0, ls=(0, (2, 2)), zorder=6)
        ax.set_xlim(*xr); ax.set_ylim(*yr); ax.set_zlim(0, Z_TOP)
        ax.set_xlabel("x (fwd)"); ax.set_ylabel("y (left)"); ax.set_zticks([])
        try: ax.set_box_aspect((xr[1]-xr[0], yr[1]-yr[0], 1.3))
        except Exception: pass

    fig = plt.figure(figsize=(15, 6.6))
    axes = []
    for n, view in enumerate(views):
        try:
            ax = fig.add_subplot(1, len(views), n+1, projection="3d", computed_zorder=False)
        except Exception:
            ax = fig.add_subplot(1, len(views), n+1, projection="3d")
            try: ax.computed_zorder = False
            except Exception: pass
        _add_scene(ax); ax.view_init(elev=view[0], azim=view[1])
        ax.set_title(f"view {n+1}  (elev={view[0]}, azim={view[1]})", fontsize=10)
        axes.append(ax)

    fig.suptitle(title, color="k", fontsize=14, fontweight="bold", y=0.98)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cb_ = fig.colorbar(sm, ax=axes, shrink=0.62, pad=0.02)
    cb_.set_label("accumulated saliency (norm.)", fontsize=10)
    cb_.ax.text(0.5, -0.05, "white = VLM blind spot", transform=cb_.ax.transAxes,
                ha="center", va="top", fontsize=8, color="0.35")

    fig.canvas.draw()
    for ax in axes:
        for (x, y, ztop, name) in objs:
            xp, yp, _ = proj3d.proj_transform(x, y, label_height, ax.get_proj())
            ax.annotate(name, xy=(xp, yp), xytext=(0, 5), textcoords="offset points", ha="center",
                        va="bottom", color="k", fontsize=11, fontweight="bold", zorder=20,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))

    out = save_prefix + "_painted.png"
    fig.savefig(out, dpi=125, bbox_inches="tight"); plt.close(fig)
    return out

print("3D projection + 2-view/back-wall painted-surface utilities ready:", PROJ)


In [ ]:
# §13  Heatmap SETUP — run ONCE per session, then run any per-category cell below. This is
#      the SLOWEST part (IG per sampled frame), so per-session/per-category running matters
#      most here. Each category writes its own .npz + _scatter/_surface PNGs under HEAT.OUT.
#      Self-contained: redefines _slug / load_policy / RED / BLUE so §13 can run without §12.
import os, gc, glob, copy, json
import numpy as np

HEAT = SimpleNamespace(TRIALS=2, IG_STEPS=8, SAMPLE_EVERY_Q=4, COMPONENT="w_z",
                       MAX_SECONDS=6.0, OUT=f"{CFG.SALIENCY_DIR}/mem_3d")
os.makedirs(HEAT.OUT, exist_ok=True)

RED  = CFG.COLOR_RED
BLUE = U.luminance_match(CFG.COLOR_RED, CFG.COLOR_BLUE)

# filesystem-safe slug for a category string
def _slug(s):
    return (s.replace(" ", "").replace("|", "__").replace(",", "-")
             .replace("=", "").replace(".", "").replace("'", ""))

def load_policy(env):    # robust to repeated builds (rsl-rl pops class_name)
    tc = copy.deepcopy(train_cfg)
    tc["policy"].setdefault("class_name", "ActorCritic")
    tc["algorithm"].setdefault("class_name", "PPO")
    ck = [f for f in os.listdir(log_dir) if f.startswith("model_")]
    latest = max(int(f[6:-3]) for f in ck)
    runner = OnPolicyRunner(env, tc, log_dir, device=gs.device)
    runner.load(os.path.join(log_dir, f"model_{latest}.pt"))
    print("Loaded checkpoint", latest)
    return runner.get_inference_policy(device=gs.device)

sf_planner = make_single_frame_planner()   # memory-free single-image planner for clean IG

def run_heatmap_category(arr_name, lc, rc, prompt):
    """One category: drive HEAT.TRIALS rollouts, project IG saliency into 3D, save + show."""
    from IPython.display import Image as IPImage, display
    cat = f"{arr_name} | {prompt}"
    print(f"\n######## HEATMAP {cat} ########")
    env, left, right = build_arena(left_color=lc, right_color=rc)
    policy = load_policy(env)
    allP, allW = [], []
    for i in range(HEAT.TRIALS):
        P, Wt = rollout_project(env, policy, planner_v2, sf_planner, left, right, prompt,
                                component=HEAT.COMPONENT, ig_steps=HEAT.IG_STEPS,
                                sample_every_q=HEAT.SAMPLE_EVERY_Q,
                                temperature=CFG.VLM_TEMPERATURE, seed=2000 + i,
                                max_seconds=HEAT.MAX_SECONDS)
        if len(P):
            allP.append(P); allW.append(Wt)
        print(f"  [{cat}] trial {i+1}/{HEAT.TRIALS}: {len(P)} saliency points")
    if not allP:
        print(f"  [{cat}] no points (depth missing?) — skipping")
        del env, policy; gc.collect(); torch.cuda.empty_cache(); return
    XYZ = np.concatenate(allP, 0); Wt = np.concatenate(allW, 0)
    np.savez(f"{HEAT.OUT}/{_slug(cat)}.npz", XYZ=XYZ, W=Wt)
    thr = np.quantile(Wt, 0.9) if len(Wt) > 10 else Wt.min()
    top = XYZ[Wt >= thr]
    cen = top[:, :2].mean(0) if len(top) else np.array([np.nan, np.nan])
    dL = np.linalg.norm(cen - np.array(left["pos"][:2])); dR = np.linalg.norm(cen - np.array(right["pos"][:2]))
    nearer = color_name(left["color"]) if dL <= dR else color_name(right["color"])
    png = render_heatmap_3d(XYZ, Wt, left, right, cat, f"{HEAT.OUT}/{_slug(cat)}")
    del env, policy; gc.collect(); torch.cuda.empty_cache()
    print(f"  saved -> {png}")
    print(f"  projection sanity: top-decile saliency centroid {cen.round(2).tolist()} nearer-target={nearer}")
    display(IPImage(png))

print("Heatmap setup ready. Run a per-category cell below (one per session is fine).")

## §13-ABL — MEM-architecture saliency **ablation engine** (shared trial cache)

A novel ablation of the **MEM hybrid-memory architecture**: how does each memory component shape *which environmental details* drive the agent's steering ($w_z$)? Every mode teacher-forces the $w_z$ the agent **actually chose** while driving (no greedy re-decode), and the four modes form a 2×2 grid:

| IG input | no text memory | + long-term text memory |
|---|---|---|
| **single onboard frame** | §13c / §13d (`single_nomem`) | §13g / §13h (`single_mem`) |
| **recent-frame video** | §13e / §13f (`video_nomem`) | §13i / §13j (`video_mem`) ⭐ |

**Caching / speed-up.** The 20-trial **drive** simulation for a category is run **once** and cached (`TRIAL_CACHE`): trajectory + per-query frames/depth/pose + the chosen-$w_z$ tokens + the text-memory snapshot. Each mode then only adds its **own** attribution pass (`SAL_CACHE`) — no re-driving, no env. So running §13c drives the trials; §13d reuses them; §13e/§13g/§13i reuse the **same** trials (new IG pass only); and §13f/§13h/§13j reuse their mode's saliency. Run this cell **once** (after re-running the planner cell so it returns `gen_ids`). `clear_trial_cache(cat)` frees the heavy cached frames if RAM is tight.

In [ ]:
# §13-ABL  UNIFIED MEM-ABLATION SALIENCY ENGINE (shared trial cache + chosen-w_z IG).
#   One subsystem powering §13c..§13j and §14. It performs a 2x2 ablation of the MEM memory
#   architecture's influence on the agent's steering decision (w_z):
#
#         INPUT TO THE IG ATTRIBUTION            no text-memory      + long-term text-memory
#         single onboard frame                   §13c / §13d         §13g / §13h
#         recent-frame video window              §13e / §13f         §13i / §13j  (most faithful)
#
#   ALL modes teacher-force the w_z token the agent ACTUALLY chose while driving (the sampled
#   drive output), so every map explains the real decision rather than a fresh greedy re-decode.
#
#   CACHING (the speed-up): the 20-trial DRIVE simulation for a category is run ONCE and cached in
#   TRIAL_CACHE (trajectory + per-query frames/depth/pose + chosen-w_z tokens + text-memory snapshot).
#   Each saliency mode then only adds its OWN attribution pass over those cached trials (SAL_CACHE),
#   with NO re-driving and NO env. So running §13c builds the trials; §13d reuses them; §13e/§13g/§13i
#   reuse the SAME trials (only a new IG pass); and §13f/§13h/§13j reuse their mode's saliency. Drive
#   the four arena categories once, then every ablation is just attribution.
import os, gc, collections
import numpy as np, torch
from PIL import Image as PILImage
from qwen_vl_utils import process_vision_info

ABL = SimpleNamespace(
    CHECKPOINTS  = (1, 5, 10, 20),
    N            = 20,
    COMPONENT    = "w_z",          # which command token to attribute
    IG_STEPS     = 8,              # Riemann steps black -> real
    SAMPLE_EVERY_Q = 3,            # attribute every k-th VLM query along the path
    MAX_SECONDS  = 6.0,
    MAX_SIDE     = 392,            # cap each IG frame's long side (divisible by 28) -> bounds VRAM/tokens
    DUPLICATE    = True,           # duplicate each video frame -> 1 clean saliency map per real frame
    SAL_NORM     = "per_query",    # cross-frame/-time scaling: "per_frame" | "per_query" | "global"
    SAL_ROBUST_Q = 0.99,
    WINDOW_FRAMES   = CFG.MEM2.RECENT_FRAMES,    # = 4: the SAME recent clip the planner drove on
    FRAME_DT        = CFG.MEM2.SHORT_INTERVAL_S, # = ~0.333 s stride (matches short-term memory)
    OUT_CONV     = f"{CFG.SALIENCY_DIR}/mem_3d_abl_conv",
    OUT_SPLIT    = f"{CFG.SALIENCY_DIR}/mem_3d_abl_split",
    SEED         = None)
for _d in (ABL.OUT_CONV, ABL.OUT_SPLIT): os.makedirs(_d, exist_ok=True)

# (use_video, use_text) per mode -> the 2x2 ablation grid.
MODES = {
    "single_nomem": (False, False),   # §13c / §13d   single onboard frame, no text memory
    "video_nomem":  (True,  False),   # §13e / §13f   recent-frame video,   no text memory
    "single_mem":   (False, True),    # §13g / §13h   single onboard frame, + long-term text memory
    "video_mem":    (True,  True),    # §13i / §13j   recent-frame video,   + long-term text memory (MOST FAITHFUL)
}
MODE_LABEL = {"single_nomem": "single-frame, no-mem", "video_nomem": "recent-video, no-mem",
              "single_mem": "single-frame, +text-mem", "video_mem": "recent-video, +text-mem (faithful)"}

_TPS_A   = int(getattr(processor.image_processor, "temporal_patch_size", 2))
_MERGE_A = int(getattr(processor.image_processor, "merge_size", 2))
_PATCH_A = int(getattr(processor.image_processor, "patch_size", 14))

def _abl_resize(img):
    """Downsize a PIL frame so its long side <= ABL.MAX_SIDE and both sides divisible by 28."""
    step = _PATCH_A * _MERGE_A
    w, h = img.size
    s = min(1.0, ABL.MAX_SIDE / float(max(w, h)))
    nw = max(step, int(round(w * s)) // step * step)
    nh = max(step, int(round(h * s)) // step * step)
    return img.resize((nw, nh), PILImage.BILINEAR)

def _abl_patch_grid(block, gh, gw):
    """Un-scramble one frame's flat patch-saliency block into a proper (gh, gw) patch image."""
    m = _MERGE_A
    return (block.reshape(gh // m, gw // m, m, m).transpose(0, 2, 1, 3).reshape(gh, gw))

def _abl_normalize(grids, mode):
    """Cross-frame/-time scaling of a query's per-frame RAW |IG| grids (see ABL.SAL_NORM)."""
    if mode == "per_frame":
        return [(g - g.min()) / (np.ptp(g) + 1e-9) for g in grids]
    if mode == "global":
        return [np.maximum(g, 0.0) for g in grids]
    stack = np.concatenate([g.ravel() for g in grids])              # "per_query" (default)
    lo = float(stack.min()); hi = float(np.quantile(stack - lo, ABL.SAL_ROBUST_Q))
    hi = hi if hi > 1e-12 else float(stack.max() - lo + 1e-9)
    return [np.clip((g - lo) / hi, 0.0, 1.0) for g in grids]

def _abl_messages(frames, instruction, contract, use_video, summary):
    """Build the IG prompt. use_video -> native video clip (duplicated) else single newest image.
    summary is None -> NO long-term text memory; a string ('' allowed) -> include it (the EXACT text
    that conditioned the drive at this query)."""
    content = []
    if use_video:
        reps = _TPS_A if ABL.DUPLICATE else 1
        vid = [f for f in frames for _ in range(reps)]
        content.append({"type": "video", "video": vid})
        vis = (f"\nShort-term visual memory: the clip is your last {len(frames)} onboard frames "
               f"~{ABL.FRAME_DT:g}s apart (oldest to newest).")
    else:
        content.append({"type": "image", "image": frames[-1]})
        vis = ""
    mem = ("\nLong-term memory so far: " + (summary if summary else "(empty)")) if summary is not None else ""
    content.append({"type": "text", "text": f"{contract}{vis}{mem}\n\nInstruction: {instruction}"})
    return [{"role": "user", "content": content}]

_ABL_SIGN, _ABL_NUM = set("+-"), set("0123456789.")
def _abl_locate(gen_ids, component):
    """Find the chosen-component numeric token span (sign-inclusive) inside the drive's gen_ids."""
    dec = [processor.tokenizer.decode([x]) for x in gen_ids]
    key = component.replace("_", ""); tgt = None
    for i, tok in enumerate(dec):
        if any(c.isdigit() for c in tok):
            ctx = "".join(dec[max(0, i - 8):i]).lower().replace("_", "")
            if key in ctx: tgt = i; break
    used_fallback = tgt is None
    if used_fallback:
        for i, tok in enumerate(dec):
            if any(c.isdigit() for c in tok): tgt = i; break
    if tgt is None:
        raise RuntimeError("no numeric token in cached drive output for IG target")
    start = tgt
    if tgt > 0 and dec[tgt - 1].strip() and all(c in _ABL_SIGN for c in dec[tgt - 1].strip()):
        start = tgt - 1
    end = tgt
    while end + 1 < len(dec) and dec[end + 1].strip() and all(c in _ABL_NUM for c in dec[end + 1].strip()):
        end += 1
    span = list(range(start, end + 1))
    return span, "".join(dec[p] for p in span).strip(), end, used_fallback

def integrated_gradients_chosen(frames, gen_ids, summary, use_video, instruction,
                                contract=None, component=None, steps=None, out_size=None):
    """Chosen-w_z IG. Teacher-forces the agent's ACTUAL sampled answer tokens (gen_ids) and attributes
    the chosen component's logit to the pixels of the given input (single frame or video window, with
    or without the long-term text memory). Returns (maps, span_str): one HxW saliency map per input
    frame (one for single-frame; WINDOW_FRAMES for a duplicated video)."""
    component = ABL.COMPONENT if component is None else component
    contract  = CFG.CONTRACT_MEM2 if contract is None else contract
    steps     = ABL.IG_STEPS if steps is None else steps
    out_size  = out_size or frames[-1].size

    small = [_abl_resize(f) for f in frames] if use_video else [_abl_resize(frames[-1])]
    messages = _abl_messages(small, instruction, contract, use_video, summary)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    _img, _vid = process_vision_info(messages)
    base = processor(text=[text], images=_img, videos=_vid, padding=True, return_tensors="pt").to(model.device)

    is_video = "pixel_values_videos" in base
    pv_key   = "pixel_values_videos" if is_video else "pixel_values"
    grid_key = "video_grid_thw"      if is_video else "image_grid_thw"
    if pv_key not in base or grid_key not in base:
        raise RuntimeError(f"Qwen did not ingest the expected modality (video={use_video})")
    t, gh, gw = [int(x) for x in base[grid_key][0].tolist()]

    # Target tokens: prefer the w_z the agent ACTUALLY emitted while driving (gen_ids). If those were
    # not cached (e.g. the planner cell wasn't re-run, so act() didn't return gen_ids), fall back to a
    # deterministic greedy re-decode from THIS prompt so IG still runs and the target is ALWAYS reported.
    # Re-run the planner cell + clear_trial_cache() to restore the true CHOSEN target.
    gen_ids = list(gen_ids) if gen_ids else []
    _src = "CHOSEN"
    if not gen_ids:
        _src = "RE-DECODED"
        with torch.no_grad():
            _g = model.generate(**base, max_new_tokens=int(getattr(CFG, "VLM_MAX_NEW_TOKENS", 64)), do_sample=False)
        gen_ids = _g[0][base.input_ids.shape[1]:].tolist()

    gen_t = torch.as_tensor(gen_ids, device=model.device, dtype=base.input_ids.dtype)
    span, span_str, end, used_fallback = _abl_locate(gen_ids, component)
    full_ids   = torch.cat([base.input_ids[0], gen_t[:end + 1]]).unsqueeze(0)
    prompt_len = base.input_ids.shape[1]
    span_li    = [prompt_len + p - 1 for p in span]
    span_ids   = gen_t[span]
    _flag = f"  !! FALLBACK: '{component}' label not found -> first number" if used_fallback else ""
    print(f"   IG target [{component}] = '{span_str}'  ({len(span)} tok, sign-inclusive, {_src}){_flag}")

    pv = base[pv_key]; baseline = torch.zeros_like(pv); total_grad = torch.zeros_like(pv, dtype=torch.float32)
    extra = {k: v for k, v in base.items() if k not in ("input_ids", "attention_mask", pv_key)}
    if "mm_token_type_ids" in extra:                               # v5: extend prompt-length copy
        _mtt = extra["mm_token_type_ids"]; _npad = full_ids.shape[1] - _mtt.shape[1]
        if _npad > 0:
            extra["mm_token_type_ids"] = torch.cat([_mtt, _mtt.new_zeros((_mtt.shape[0], _npad))], dim=1)
    with torch.enable_grad():
        for sidx in range(1, steps + 1):
            a = sidx / steps
            pv_s = (baseline + a * (pv - baseline)).clone().requires_grad_(True)
            out = model(input_ids=full_ids, attention_mask=torch.ones_like(full_ids),
                        **{pv_key: pv_s}, **extra)
            rows = out.logits[0][span_li].float()
            logp = rows.gather(1, span_ids[:, None]).squeeze(1) - torch.logsumexp(rows, dim=-1)
            grad, = torch.autograd.grad(logp.sum(), pv_s, retain_graph=False)
            total_grad += grad.float()
            del out, grad, pv_s
    ig = ((pv - baseline).float() * total_grad / steps)
    patch_sal = ig.abs().sum(dim=-1).detach().cpu().numpy()

    per = gh * gw
    grids = [_abl_patch_grid(patch_sal[k * per:(k + 1) * per], gh, gw).astype(np.float32) for k in range(t)]
    grids = _abl_normalize(grids, ABL.SAL_NORM)
    maps = [np.asarray(PILImage.fromarray(np.asarray(g, np.float32)).resize(out_size, PILImage.BILINEAR), dtype=np.float32)
            for g in grids]
    del base, ig, total_grad, patch_sal, pv, baseline
    gc.collect(); torch.cuda.empty_cache()
    return maps, span_str

# ============================ shared 20-trial DRIVE cache =====================================
TRIAL_CACHE = {}     # cat -> list of trial dicts (timeline + per-query chosen-w_z + text snapshot + path)
SAL_CACHE   = {}     # (cat, mode) -> list of (P, W, path) per trial
ABL_SEEDS   = {}     # cat -> base seed actually used (printed; reproducible)
_ABL_RNG    = np.random.default_rng()

# resolve (and cache) a category's base trial seed
def _abl_seed_base(cat, seed=None):
    if seed is None: seed = ABL.SEED
    if seed is None: seed = ABL_SEEDS.get(cat)
    if seed is None: seed = int(_ABL_RNG.integers(0, 2**31 - 1))
    seed = int(seed); prev = ABL_SEEDS.get(cat)
    if prev is not None and prev != seed and TRIAL_CACHE.get(cat):
        print(f"  \u26a0 seed changed {prev} -> {seed}; clearing cached drive trials + saliency for this category")
        TRIAL_CACHE[cat] = []
        for k in [k for k in SAL_CACHE if k[0] == cat]: SAL_CACHE.pop(k, None)
    ABL_SEEDS[cat] = seed
    return seed

def clear_trial_cache(cat=None):
    """Free the (heavy) cached drive frames. Pass a category string or None to clear all.
    SAL_CACHE (light projected points) is kept so renders still work; re-running a mode after this
    would need the trials re-driven."""
    global TRIAL_CACHE
    if cat is None: TRIAL_CACHE = {}
    else: TRIAL_CACHE.pop(cat, None)
    gc.collect()
    print(f"  cleared drive-frame cache for {'ALL categories' if cat is None else cat}")

def _drive_record_trial(env, policy, left, right, instruction, seed):
    """Drive ONE trial with planner_v2 (its full memory, temperature sampling) and record everything
    later IG passes need: a de-duplicated frame/depth/pose timeline (~ABL.FRAME_DT apart), and per
    sampled query the chosen-w_z tokens + the long-term text-memory snapshot that conditioned them."""
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(ABL.MAX_SECONDS / env.dt)
    planner_v2.reset(); obs, _ = env.reset(); cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
    timeline, queries, path = [], [], []
    last_push, qn = -1e9, 0
    with torch.no_grad():
        for tstep in range(max_steps):
            now = tstep * env.dt
            frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
            if (now - last_push) >= ABL.FRAME_DT - 1e-6:           # de-dup timeline of recent frames
                cp, cl = onboard_pose(env)
                timeline.append(dict(frame=_abl_resize(frame),
                                     depth=np.asarray(depth, np.float32).astype(np.float16),
                                     pos=np.asarray(cp, float), look=np.asarray(cl, float)))
                last_push = now
            if tstep % query_every == 0:
                summary_before = planner_v2._language_memory       # the text in THIS prompt (pre-update)
                r = planner_v2.act(frame, instruction, sim_time=now,
                                   temperature=CFG.VLM_TEMPERATURE, seed=seed + tstep)
                cmd = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]], float) + (1-CFG.LOWPASS_ALPHA)*cmd
                if depth is not None and len(timeline) >= ABL.WINDOW_FRAMES and qn % ABL.SAMPLE_EVERY_Q == 0:
                    queries.append(dict(step=tstep, tl_idx=len(timeline) - 1,
                                        gen_ids=list(r.get("gen_ids") or []), summary=summary_before))
                qn += 1
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            a = policy(obs); obs, _, done, _ = env.step(a)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
            p2 = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0][:2]; path.append(p2.copy())
            if any(np.linalg.norm(p2 - np.array(o["pos"][:2])) < CFG.ARENA.commit_radius for o in (left, right)):
                break
    return dict(timeline=timeline, queries=queries, path=np.asarray(path, float), instruction=instruction)

def _saliency_trial(trial, mode):
    """Run mode's IG over one cached drive trial; project each attributed frame at its own pose+depth."""
    use_video, use_text = MODES[mode]
    K = onboard_K(); out_size = tuple(CFG.CAM.res); pts, ws = [], []
    tl = trial["timeline"]
    for q in trial["queries"]:
        idx = q["tl_idx"]
        win = tl[max(0, idx - ABL.WINDOW_FRAMES + 1): idx + 1] if use_video else [tl[idx]]
        frames  = [w["frame"] for w in win]
        summary = (q["summary"] if use_text else None)
        try:
            maps, _ = integrated_gradients_chosen(frames, q["gen_ids"], summary, use_video,
                                                  instruction=trial["instruction"],
                                                  component=ABL.COMPONENT, steps=ABL.IG_STEPS,
                                                  out_size=out_size)
        except Exception as e:
            print("   [abl-ig] skip:", repr(e)); continue
        for w, sal in zip(win, maps):
            P, Wt = project_saliency(sal, np.asarray(w["depth"], float), w["pos"], w["look"], K)
            if len(P): pts.append(P); ws.append(Wt)
    P  = np.concatenate(pts, 0) if pts else np.zeros((0, 3))
    Wt = np.concatenate(ws, 0)  if ws  else np.zeros((0,))
    return P, Wt, trial["path"]

# (re)use the cached drive trials for a category, running any that are missing
def _ensure_trials(cat, lc, rc, prompt, n, seed):
    base = _abl_seed_base(cat, seed)
    trials = TRIAL_CACHE.setdefault(cat, [])
    if len(trials) < n:
        env, left, right = build_arena(left_color=lc, right_color=rc); policy = load_policy(env)
        for i in range(len(trials), n):
            tr = _drive_record_trial(env, policy, left, right, prompt, base + i)
            trials.append(tr)
            print(f"  drive trial {i+1}/{n} (seed {base + i}): {len(tr['queries'])} IG queries, path len {len(tr['path'])}")
        del env, policy; gc.collect(); torch.cuda.empty_cache()
    else:
        print(f"  (reusing {len(trials)} cached DRIVE trials @ seed base {base})")
    return base

# (re)use the cached saliency for a category/mode, computing any that are missing
def _ensure_saliency(cat, mode, n):
    sc = SAL_CACHE.setdefault((cat, mode), [])
    trials = TRIAL_CACHE[cat]
    if len(sc) < n:
        print(f"  [{MODE_LABEL[mode]}] computing saliency for trials {len(sc)+1}..{n} ...")
        for i in range(len(sc), n):
            sc.append(_saliency_trial(trials[i], mode))
            P = sc[-1][0]; print(f"    trial {i+1}/{n}: {len(P)} pts")
    else:
        print(f"  [{MODE_LABEL[mode]}] reusing {len(sc)} cached saliency trials")
    return sc

# goal box positions for a left/right colour arrangement
def _abl_goals(lc, rc):
    return U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                            left_color=lc, right_color=rc)

# did the path reach a goal, and which colour is the final position nearer to?
def _abl_classify(path, red_xy, blue_xy, cr):
    pa = np.asarray(path, float).reshape(-1, 2)
    if not len(pa): return False, "nearer_red"
    final = pa[-1]
    dmin = min(np.min(np.linalg.norm(pa - red_xy, axis=1)), np.min(np.linalg.norm(pa - blue_xy, axis=1)))
    nearer = "nearer_red" if np.linalg.norm(final - red_xy) <= np.linalg.norm(final - blue_xy) else "nearer_blue"
    return bool(dmin < cr), nearer

def run_ablation_convergence(arr_name, lc, rc, prompt, mode, checkpoints=None, seed=None):
    """Convergence sweep (n=1,5,10,20) for one ablation `mode`. Drives the trials once (shared),
    then renders cumulative saliency at each checkpoint."""
    from IPython.display import Image as IPImage, display
    assert mode in MODES, f"mode must be one of {list(MODES)}"
    cks = sorted(set(checkpoints or ABL.CHECKPOINTS)); nmax = cks[-1]
    cat = f"{arr_name} | {prompt}"
    print(f"\n######## ABLATION CONVERGENCE [{MODE_LABEL[mode]}]  {cat}  |  n in {cks} ########")
    print(f"  chosen-w_z IG  |  window={'video x'+str(ABL.WINDOW_FRAMES) if MODES[mode][0] else 'single frame'}"
          f"  text-memory={'ON' if MODES[mode][1] else 'off'}  IG_STEPS={ABL.IG_STEPS}  sal_norm={ABL.SAL_NORM}")
    base = _ensure_trials(cat, lc, rc, prompt, nmax, seed)
    print(f"  seed base = {base}   (reproduce: seed={base})")
    sc = _ensure_saliency(cat, mode, nmax)
    left, right = _abl_goals(lc, rc)
    pngs = []
    for n in cks:
        sub = sc[:n]
        XYZ = np.concatenate([p for p, _, _ in sub if len(p)]) if any(len(p) for p, _, _ in sub) else np.zeros((0, 3))
        Wt  = np.concatenate([w for _, w, _ in sub if len(w)]) if any(len(w) for _, w, _ in sub) else np.zeros((0,))
        paths = [pa for _, _, pa in sub if len(pa)]
        png = render_heatmap_3d(XYZ, Wt, left, right,
                                f"{cat}   |   [{MODE_LABEL[mode]}]   n = {n} trials",
                                f"{ABL.OUT_CONV}/{_slug(cat)}_{mode}_n{n:02d}", paths=paths)
        pngs.append(png); print(f"  n={n}: {len(XYZ)} pts, {len(paths)} trails -> {png}")
        display(IPImage(png))
    return pngs

def run_ablation_split(arr_name, lc, rc, prompt, mode, n_trials=None, seed=None):
    """Outcome split (reached/drifted/nearer-red/nearer-blue) for one ablation `mode`, reusing the
    same cached drive trials AND the same per-mode saliency as the matching convergence section."""
    from IPython.display import Image as IPImage, display
    assert mode in MODES, f"mode must be one of {list(MODES)}"
    n = int(n_trials or ABL.N); cat = f"{arr_name} | {prompt}"
    print(f"\n######## ABLATION OUTCOME-SPLIT [{MODE_LABEL[mode]}]  {cat}  |  N = {n} ########")
    base = _ensure_trials(cat, lc, rc, prompt, n, seed)
    print(f"  seed base = {base}   (reproduce: seed={base})")
    sc = _ensure_saliency(cat, mode, n)
    left, right = _abl_goals(lc, rc)
    red_xy = blue_xy = None
    for o in (left, right):
        if color_name(o["color"]) == "red": red_xy = np.array(o["pos"][:2], float)
        else: blue_xy = np.array(o["pos"][:2], float)
    cr = CFG.ARENA.commit_radius
    groups = {"reached": [], "drifted": [], "nearer_red": [], "nearer_blue": []}
    for (P, Wt, path) in sc[:n]:
        reached, nearer = _abl_classify(path, red_xy, blue_xy, cr)
        groups["reached" if reached else "drifted"].append((P, Wt, path))
        groups[nearer].append((P, Wt, path))
    print("  groups:", {k: len(v) for k, v in groups.items()})
    pngs = []
    for key, label in [("reached", "reached"), ("drifted", "drifted"),
                       ("nearer_red", "nearer \u2192 RED"), ("nearer_blue", "nearer \u2192 BLUE")]:
        sub = groups[key]
        if not sub:
            print(f"  [{label}] no trials \u2014 skipped"); continue
        XYZ = np.concatenate([p for p, _, _ in sub if len(p)]) if any(len(p) for p, _, _ in sub) else np.zeros((0, 3))
        Wt  = np.concatenate([w for _, w, _ in sub if len(w)]) if any(len(w) for _, w, _ in sub) else np.zeros((0,))
        paths = [pa for _, _, pa in sub if len(pa)]
        png = render_heatmap_3d(XYZ, Wt, left, right,
                                f"{cat}   |   [{MODE_LABEL[mode]}]   {label}   (n = {len(sub)})",
                                f"{ABL.OUT_SPLIT}/{_slug(cat)}_{mode}_{key}", paths=paths)
        pngs.append(png); print(f"  [{label}] {len(sub)} trials, {len(XYZ)} pts -> {png}")
        display(IPImage(png))
    return pngs

print("MEM-ablation engine ready. Drive once per category; every §13c..§13j mode reuses the trials.")
print("  modes:", {k: MODE_LABEL[k] for k in MODES})


In [ ]:
# check
# §13-flipcheck  Project the cubes' OWN colours (ground truth) to test for a left/right mirror.
import numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display
_env, _left, _right = build_arena(left_color=RED, right_color=BLUE)   # left=red, right=blue
_ = load_policy(_env); _env.reset()
cp, cl = onboard_pose(_env); set_level_pose(_env.camera, cp, cl)
rgb, depth = U.render_camera(_env.camera, want_depth=True)
r,g,b = (rgb[...,i].astype(float) for i in range(3)); bright = (r+g+b) > 120
red_mask  = (bright & (r > g+30) & (r > b+30)).astype(float)
blue_mask = (bright & (b > r+30) & (b > g+30)).astype(float)
K = onboard_K()
PR,_ = project_saliency(red_mask,  depth, cp, cl, K, wthresh=0.5)
PB,_ = project_saliency(blue_mask, depth, cp, cl, K, wthresh=0.5)
red_pos, blue_pos = np.array(_left["pos"][:2]), np.array(_right["pos"][:2])
near = lambda c: "red" if np.linalg.norm(c-red_pos) <= np.linalg.norm(c-blue_pos) else "blue"
print(f"red mask {int(red_mask.sum())} px | blue mask {int(blue_mask.sum())} px")
if len(PR) and len(PB):
    cR, cB = PR[:,:2].mean(0), PB[:,:2].mean(0)
    print(f"RED  pixels -> (x,y)={cR.round(2).tolist()} nearest={near(cR)}")
    print(f"BLUE pixels -> (x,y)={cB.round(2).tolist()} nearest={near(cB)}")
    ok = near(cR)=="red" and near(cB)=="blue"
    print(f"\nPROJ.FLIP_U is currently {PROJ.FLIP_U}.")
    print(">> CORRECT: leave PROJ.FLIP_U as is." if ok
          else f">> MIRRORED: set PROJ.FLIP_U = {not PROJ.FLIP_U} in §13a and re-run §13a + the heatmap.")
else:
    print("masks empty - re-run; at reset both cubes should be visible.")
plt.figure(figsize=(5,3)); plt.imshow(rgb[...,:3].astype("uint8")); plt.axis("off")
plt.title("onboard frame at reset (red should be LEFT, blue RIGHT)")
plt.tight_layout(); plt.savefig("/content/flipcheck.png", dpi=100); plt.close()
display(IPImage("/content/flipcheck.png")); import gc; del _env; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# making sure the VLM is processing input as video
# §CHECK-A  Does Qwen ingest a PIL-frame list as a NATIVE VIDEO?  (no generate, no rollout)
import torch, numpy as np
from PIL import Image
from qwen_vl_utils import process_vision_info

assert 'model' in dir() and 'processor' in dir(), "Run §10b first (loads model+processor)."

# --- make a few dummy PIL frames with motion (so it's a legit clip) ---
W, H = CFG.CAM.res
# synthetic frame with a block sliding left-to-right, for probing video vs. image ingestion
def _dummy_frame(k, n):
    a = np.zeros((H, W, 3), np.uint8)
    x = int((k / max(1, n - 1)) * (W - 80))      # a block that slides L->R across frames
    a[H//2 - 40:H//2 + 40, x:x + 80] = (255, 60, 60)
    a[:, :, 2] = 30 + 20 * k                      # global tint drift
    return Image.fromarray(a)

def probe_vision(frames_or_msg, label, as_video=True):
    """Run process_vision_info + processor on either a frame list (builds a video msg)
    or a ready-made messages list; report whether Qwen got it as video."""
    if isinstance(frames_or_msg, list) and frames_or_msg and isinstance(frames_or_msg[0], dict):
        messages = frames_or_msg                                  # already a messages list
    else:
        frames = list(frames_or_msg)
        vis = ({"type": "video", "video": frames} if as_video
               else [{"type": "image", "image": f} for f in frames])
        content = ([vis] if as_video else vis) + [{"type": "text", "text": "test"}]
        messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    fell_back = False
    try:
        image_inputs, video_inputs = process_vision_info(messages)
    except Exception as e:
        fell_back = True
        print(f"[{label}] ❌ process_vision_info REJECTED it -> planner.act() would fall back to images.\n        {type(e).__name__}: {e}")
        return None

    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt")
    ids = inputs.input_ids[0]
    has_vid = "video_grid_thw" in inputs and inputs.get("pixel_values_videos") is not None
    has_img = "image_grid_thw" in inputs and inputs.get("pixel_values") is not None
    vid_tok = int((ids == model.config.video_token_id).sum())
    img_tok = int((ids == model.config.image_token_id).sum())

    print(f"\n=== {label} ===")
    print(f"  qwen_vl_utils -> video_inputs: {None if video_inputs is None else len(video_inputs)} clip(s),"
          f" image_inputs: {None if image_inputs is None else len(image_inputs)}")
    print(f"  inputs keys: {[k for k in inputs.keys()]}")
    if has_vid:
        t, h, w = [int(v) for v in inputs['video_grid_thw'][0].tolist()]
        ips = getattr(processor.image_processor, 'temporal_patch_size', 2)
        print(f"  video_grid_thw = (t={t}, h={h}, w={w})  [t is in temporal-patch units; ~t*{ips} raw frames]")
    if has_img:
        print(f"  image_grid_thw = {inputs['image_grid_thw'].tolist()}")
    print(f"  <|video|> tokens: {vid_tok}   <|image|> tokens: {img_tok}")
    print(f"  VERDICT: {'✅ NATIVE VIDEO (temporal)' if has_vid else '❌ ingested as IMAGES, not video'}")
    return inputs

# run it on a clip of dummy frames
_ = probe_vision([_dummy_frame(k, 6) for k in range(6)], "raw PIL list as VIDEO", as_video=True)
# and, for contrast, the multi-image fallback path:
_ = probe_vision([_dummy_frame(k, 6) for k in range(6)], "same frames as IMAGES", as_video=False)

In [ ]:
# §CHECK-B  Confirm the REAL planner messages are video (exercises _build_messages + fallback)
frames = [_dummy_frame(k, 6) for k in range(6)]

# ---- base MEMPlanner ----
planner.reset()
msgs = None
for f in frames:                      # _step stays 0 here, so each call pushes -> fills the deque
    msgs, used_video = planner._build_messages(f, "go to the red box")
print(f"[base planner] used_video flag = {used_video}  | window = {len(planner._frames)} frames")
probe_vision(msgs, "base planner ACTUAL messages")
planner.reset()                       # restore clean state for real runs

# ---- MEMPlannerV2 (if §12a was run) — recent-only, NO anchor ----
if 'planner_v2' in dir():
    planner_v2.reset()
    msgs2 = None
    for i, f in enumerate(frames):
        planner_v2._now_t = i * planner_v2.short_interval_s   # advance sim time so recent deque fills
        msgs2, used_video2 = planner_v2._build_messages(f, "go to the red box")
    n = len(planner_v2._recent)
    print(f"\n[planner_v2] used_video flag = {used_video2}  | recent window = {n} frames (no anchor)")
    probe_vision(msgs2, "planner_v2 ACTUAL messages")
    planner_v2.reset()
else:
    print("\n(planner_v2 not in namespace — run §12a if you want to check the V2 path too)")

In [ ]:
# check token
tok = processor.tokenizer
for s in ["w_z: -0.5", "w_z: 0.5", "-0.5", "0.5", "-1.2"]:
    ids = tok(s, add_special_tokens=False).input_ids
    print(s, "->", [tok.decode([i]) for i in ids])

### §13c — Chosen-$w_z$ IG, **single onboard frame, no memory** — convergence (n = 1, 5, 10, 20)

Ablation baseline: attribute the agent's *actually chosen* $w_z$ to a **single** onboard frame, with **no** long-term text memory in the prompt. Uses the shared 20-trial drive cache (`run_ablation_convergence(..., mode="single_nomem")`).

In [ ]:
# §13c  LEGACY live-IG convergence runner (reuses §13a rollout/render; caches trials per
#       category). NOTE: the §13c-1..4 category cells below run through the §13-ABL cached
#       engine instead (`run_ablation_convergence(mode="single_nomem")`, one shared 20-trial
#       drive per category); `run_convergence_category` here is kept as a standalone runner
#       that drives + attributes live with the single-frame planner.
#       Seeds are RANDOM per category and REPRODUCIBLE: a base seed is drawn once, PRINTED to
#       the cell output (never baked into the render title), and stored in CONV_SEEDS so the
#       cache stays consistent and §13d's legacy runner reuses the SAME trials. To redo a past
#       run later, pass seed=<the printed base> to this cell.
import os, gc, torch
import numpy as np
CONV = SimpleNamespace(CHECKPOINTS=(1, 5, 10, 20), COMPONENT="w_z",
                       SAMPLE_EVERY_Q=3, IG_STEPS=8, MAX_SECONDS=6.0,
                       SEED=None,                               # None -> fresh random base per category
                       OUT=f"{CFG.SALIENCY_DIR}/mem_3d_conv")
os.makedirs(CONV.OUT, exist_ok=True)
CONV_CACHE = {}     # cat -> list of (P, W, path) per trial, reused across reruns
CONV_SEEDS = {}     # cat -> base seed actually used (printed; for reproducibility)
_CONV_RNG  = np.random.default_rng()


def _conv_seed_base(cat, seed=None):
    """Resolve the base seed for a category. Priority: explicit arg > CONV.SEED > cached > fresh random.
    Stored in CONV_SEEDS so the cache stays consistent and §13c/§13d agree on seeds. If an explicit
    seed differs from the one the cache was built with, that category's cache is cleared first."""
    if seed is None: seed = CONV.SEED
    if seed is None: seed = CONV_SEEDS.get(cat)
    if seed is None: seed = int(_CONV_RNG.integers(0, 2**31 - 1))
    seed = int(seed)
    prev = CONV_SEEDS.get(cat)
    if prev is not None and prev != seed and CONV_CACHE.get(cat):
        print(f"  \u26a0 seed changed {prev} -> {seed}; clearing {len(CONV_CACHE[cat])} cached trials for this category")
        CONV_CACHE[cat] = []
    CONV_SEEDS[cat] = seed
    return seed


# drive/cache trials for one category, render saliency at each checkpoint n
def run_convergence_category(arr_name, lc, rc, prompt, checkpoints=None, seed=None):
    from IPython.display import Image as IPImage, display
    cks = sorted(set(checkpoints or CONV.CHECKPOINTS)); nmax = cks[-1]
    cat = f"{arr_name} | {prompt}"
    base = _conv_seed_base(cat, seed)
    print(f"\n######## CONVERGENCE {cat}  |  n in {cks} ########")
    print(f"  seed base = {base}   trial seeds = {base}..{base + nmax - 1}")
    print(f"  reproduce this exact run later: pass seed={base} to this cell")
    trials = CONV_CACHE.setdefault(cat, [])
    if len(trials) < nmax:                       # only run the missing trials
        env, left, right = build_arena(left_color=lc, right_color=rc)
        policy = load_policy(env)
        for i in range(len(trials), nmax):
            P, Wt, path = rollout_project(env, policy, planner_v2, sf_planner, left, right, prompt,
                                          component=CONV.COMPONENT, sample_every_q=CONV.SAMPLE_EVERY_Q,
                                          ig_steps=CONV.IG_STEPS, temperature=CFG.VLM_TEMPERATURE,
                                          seed=base + i, max_seconds=CONV.MAX_SECONDS, return_path=True)
            trials.append((P, Wt, path))
            print(f"  trial {i+1}/{nmax} (seed {base + i}): {len(P)} pts, path len {len(path)}")
        del env, policy; gc.collect(); torch.cuda.empty_cache()
    else:
        left, right = U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                                       left_color=lc, right_color=rc)
        print(f"  (reusing {len(trials)} cached trials @ seed base {base})")

    pngs = []
    for n in cks:
        sub = trials[:n]
        XYZ = np.concatenate([p for p, _, _ in sub if len(p)]) if any(len(p) for p, _, _ in sub) else np.zeros((0, 3))
        Wt  = np.concatenate([w for _, w, _ in sub if len(w)]) if any(len(w) for _, w, _ in sub) else np.zeros((0,))
        paths = [pa for _, _, pa in sub if len(pa)]
        png = render_heatmap_3d(XYZ, Wt, left, right, f"{cat}   |   n = {n} trials",
                                f"{CONV.OUT}/{_slug(cat)}_n{n:02d}", paths=paths)
        pngs.append(png); print(f"  n={n}: {len(XYZ)} pts, {len(paths)} trails -> {png}")
        display(IPImage(png))
    return pngs

print("Convergence-sweep runner ready (random seeds, printed). Run a per-category §13c cell below.")


In [ ]:
# §13c-1  Convergence [single_nomem]: L=red,R=blue | friendly
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="single_nomem")

In [ ]:
# §13c-2  Convergence [single_nomem]: L=red,R=blue | hostile
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="single_nomem")

In [ ]:
# §13c-3  Convergence [single_nomem]: L=blue,R=red | friendly
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="single_nomem")

In [ ]:
# §13c-4  Convergence [single_nomem]: L=blue,R=red | hostile
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="single_nomem")

### §13d — Chosen-$w_z$ IG, **single onboard frame, no memory** — outcome split (N = 20)

Same trials/saliency as §13c, split by outcome (reached / drifted / nearer-red / nearer-blue).

In [ ]:
# §13d  LEGACY live-IG outcome-split runner (2 angles, one image per group). NOTE: the
#       §13d-1..4 category cells below run through the §13-ABL engine instead
#       (`run_ablation_split(mode="single_nomem")`, reusing the shared cached drive trials);
#       `run_split_category` here is the standalone live counterpart.
#       Shares CONV_CACHE + CONV_SEEDS with §13c's legacy runner, so the per-category RANDOM
#       seed (printed below) is identical and the same cached trials are reused. Pass
#       seed=<base> to reproduce a run.
import os, gc, torch
import numpy as np
try:
    CONV; CONV_CACHE; CONV_SEEDS; _CONV_RNG; _conv_seed_base
except NameError:
    CONV = SimpleNamespace(CHECKPOINTS=(1, 5, 10, 20), COMPONENT="w_z", SAMPLE_EVERY_Q=3,
                           IG_STEPS=8, MAX_SECONDS=6.0, SEED=None, OUT=f"{CFG.SALIENCY_DIR}/mem_3d_conv")
    os.makedirs(CONV.OUT, exist_ok=True)
    CONV_CACHE = {}; CONV_SEEDS = {}; _CONV_RNG = np.random.default_rng()
    # resolve (and cache) a category's base trial seed (same pattern as §12c)
    def _conv_seed_base(cat, seed=None):
        if seed is None: seed = CONV.SEED
        if seed is None: seed = CONV_SEEDS.get(cat)
        if seed is None: seed = int(_CONV_RNG.integers(0, 2**31 - 1))
        seed = int(seed)
        prev = CONV_SEEDS.get(cat)
        if prev is not None and prev != seed and CONV_CACHE.get(cat):
            print(f"  \u26a0 seed changed {prev} -> {seed}; clearing {len(CONV_CACHE[cat])} cached trials")
            CONV_CACHE[cat] = []
        CONV_SEEDS[cat] = seed
        return seed
SPLIT = SimpleNamespace(N=20, OUT=f"{CFG.SALIENCY_DIR}/mem_3d_split")
os.makedirs(SPLIT.OUT, exist_ok=True)


def _classify_trial(path, red_xy, blue_xy, commit_radius):
    """Return (reached_bool, 'nearer_red'|'nearer_blue') from a trial's ground path."""
    pa = np.asarray(path, float).reshape(-1, 2)
    if not len(pa):
        return False, "nearer_red"
    final = pa[-1]
    dmin = min(np.min(np.linalg.norm(pa - red_xy, axis=1)),
               np.min(np.linalg.norm(pa - blue_xy, axis=1)))
    reached = bool(dmin < commit_radius)
    nearer = "nearer_red" if np.linalg.norm(final - red_xy) <= np.linalg.norm(final - blue_xy) else "nearer_blue"
    return reached, nearer


# drive/cache trials for one category, then split the saliency by outcome
def run_split_category(arr_name, lc, rc, prompt, n_trials=None, seed=None):
    from IPython.display import Image as IPImage, display
    n_trials = int(n_trials or SPLIT.N)
    cat = f"{arr_name} | {prompt}"
    base = _conv_seed_base(cat, seed)
    print(f"\n######## OUTCOME-SPLIT {cat}  |  N = {n_trials} trials ########")
    print(f"  seed base = {base}   trial seeds = {base}..{base + n_trials - 1}")
    print(f"  reproduce this exact run later: pass seed={base} to this cell")
    trials = CONV_CACHE.setdefault(cat, [])               # shared with §13c (same seeds)
    if len(trials) < n_trials:
        env, left, right = build_arena(left_color=lc, right_color=rc)
        policy = load_policy(env)
        for i in range(len(trials), n_trials):
            P, Wt, path = rollout_project(env, policy, planner_v2, sf_planner, left, right, prompt,
                                          component=CONV.COMPONENT, sample_every_q=CONV.SAMPLE_EVERY_Q,
                                          ig_steps=CONV.IG_STEPS, temperature=CFG.VLM_TEMPERATURE,
                                          seed=base + i, max_seconds=CONV.MAX_SECONDS, return_path=True)
            trials.append((P, Wt, path))
            print(f"  trial {i+1}/{n_trials} (seed {base + i}): {len(P)} pts, path len {len(path)}")
        del env, policy; gc.collect(); torch.cuda.empty_cache()
    else:
        left, right = U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                                       left_color=lc, right_color=rc)
        print(f"  (reusing {len(trials)} cached trials @ seed base {base})")

    red_xy = blue_xy = None
    for o in (left, right):
        if color_name(o["color"]) == "red": red_xy = np.array(o["pos"][:2], float)
        else: blue_xy = np.array(o["pos"][:2], float)
    cr = CFG.ARENA.commit_radius

    groups = {"reached": [], "drifted": [], "nearer_red": [], "nearer_blue": []}
    for (P, Wt, path) in trials[:n_trials]:
        reached, nearer = _classify_trial(path, red_xy, blue_xy, cr)
        groups["reached" if reached else "drifted"].append((P, Wt, path))
        groups[nearer].append((P, Wt, path))
    print("  groups:", {k: len(v) for k, v in groups.items()})

    pngs = []
    for key, label in [("reached", "reached"), ("drifted", "drifted"),
                       ("nearer_red", "nearer \u2192 RED"), ("nearer_blue", "nearer \u2192 BLUE")]:
        sub = groups[key]
        if not sub:
            print(f"  [{label}] no trials — skipped"); continue
        XYZ = np.concatenate([p for p, _, _ in sub if len(p)]) if any(len(p) for p, _, _ in sub) else np.zeros((0, 3))
        Wt = np.concatenate([w for _, w, _ in sub if len(w)]) if any(len(w) for _, w, _ in sub) else np.zeros((0,))
        paths = [pa for _, _, pa in sub if len(pa)]
        png = render_heatmap_3d(XYZ, Wt, left, right, f"{cat}   |   {label}   (n = {len(sub)} trials)",
                                f"{SPLIT.OUT}/{_slug(cat)}_{key}", paths=paths)
        pngs.append(png); print(f"  [{label}] {len(sub)} trials, {len(XYZ)} pts -> {png}")
        display(IPImage(png))                              # each group is its OWN image
    return pngs

print("Outcome-split runner ready (shares §13c random seeds). Run a per-category §13d cell below.")


In [ ]:
# §13d-1  Outcome split [single_nomem]: L=red,R=blue | friendly
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="single_nomem")

In [ ]:
# §13d-2  Outcome split [single_nomem]: L=red,R=blue | hostile
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="single_nomem")

In [ ]:
# §13d-3  Outcome split [single_nomem]: L=blue,R=red | friendly
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="single_nomem")

In [ ]:
# §13d-4  Outcome split [single_nomem]: L=blue,R=red | hostile
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="single_nomem")

### §13e — Chosen-$w_z$ IG, **recent-frame video, no memory** — convergence (n = 1, 5, 10, 20)

Adds short-term visual memory: attribute the chosen $w_z$ over the **recent video window** (still no text memory). Reuses the SAME drive trials as §13c (`mode="video_nomem"`); only the attribution pass differs.

In [ ]:
# §13e-setup  FAITHFUL MULTI-FRAME (VIDEO) INTEGRATED GRADIENTS.
#   NOTE: the §13e-1..4 category cells below run through the §13-ABL engine
#   (`run_ablation_convergence(mode="video_nomem")`); this cell is still REQUIRED —
#   §14b reads MF.SAL_NORM — and `run_mf_convergence_category` remains available as a
#   standalone live MF-IG runner.
#   Extends the §13c/§13d single-image IG to the short-term VIDEO window. The recent N frames
#   (~0.33 s apart, real motion) are fed to Qwen as ONE native video; IG interpolates the whole
#   clip from an all-BLACK video back to the real clip and attributes the command logit to each
#   frame's pixels. Each frame's saliency is then projected into the arena with THAT frame's own
#   camera pose + depth (where/when it was taken).
#
#   Why VIDEO (not a list of images): passing pixel_values_videos + video_grid_thw straight
#   through the top-level model forward keeps the gradient path clean. The per-image approach
#   needs a manual vision-tower/embedding splice, which is what produced the IndexError
#   ('shape of the mask ...') and the OOM you hit. We never touch the vision tower by hand here.
#
#   Qwen2.5-VL groups frames in pairs (temporal_patch_size=2 -> grid_t = n_frames/2). So we
#   DUPLICATE each frame ([f0,f0,f1,f1,...]) -> each ORIGINAL frame occupies exactly one temporal
#   group -> exactly one saliency map per real frame (no 2-frames-per-token ambiguity, no need to
#   guess a middle depth). Set MF.DUPLICATE=False to instead pair raw frames (then each map covers
#   a pair and is projected with the pair's mid pose/depth).
#
#   Deps: §13a (rollout/render/projection), §13-setup (build_arena/load_policy/_slug/RED/BLUE),
#         §12a (planner_v2 drives the trials).
import os, gc, collections
import numpy as np, torch
from PIL import Image as PILImage
from qwen_vl_utils import process_vision_info
import mem_planner as MP

MF = SimpleNamespace(
    # MATCH the drive memory exactly: the saliency must attribute over the SAME clip the
    # planner actually saw (4 frames @ 1/VLM_RATE_HZ), else IG explains a clip it never used.
    WINDOW_FRAMES = CFG.MEM2.RECENT_FRAMES,     # = 4 (faithful to short-term memory)
    FRAME_DT      = CFG.MEM2.SHORT_INTERVAL_S,  # = ~0.333 s (same stride as short-term memory)
    DUPLICATE     = True,     # duplicate each frame -> 1 clean saliency map per real frame
    IG_STEPS      = 8,        # Riemann steps black-video -> real-video
    COMPONENT     = "w_z",    # which command token to attribute (v_x|v_y|w_z)
    SAMPLE_EVERY_Q= 3,        # run MF-IG every k-th VLM query along the path
    MAX_SECONDS   = 6.0,
    MAX_SIDE      = 392,      # cap each IG frame's long side (divisible by 28) -> bounds VRAM/tokens
    SAL_NORM      = "per_query", # cross-frame/-time scaling: "per_frame" | "per_query" | "global"
    SAL_ROBUST_Q  = 0.99,     # robust upper percentile used by the "per_query"/"global" scaling
    CHECKPOINTS   = (1, 5, 10, 20),
    SEED          = None,     # None -> fresh random base per category (printed, reproducible)
    OUT_CONV      = f"{CFG.SALIENCY_DIR}/mem_3d_mf_conv",
    OUT_SPLIT     = f"{CFG.SALIENCY_DIR}/mem_3d_mf_split",
)
os.makedirs(MF.OUT_CONV, exist_ok=True); os.makedirs(MF.OUT_SPLIT, exist_ok=True)

_TPS   = int(getattr(processor.image_processor, "temporal_patch_size", 2))
_MERGE = int(getattr(processor.image_processor, "merge_size", 2))
_PATCH = int(getattr(processor.image_processor, "patch_size", 14))

def _resize_for_ig(img, max_side=None):
    """Downsize a PIL frame so its long side <= max_side and both sides divisible by
    patch_size*merge_size (=28). Fewer vision tokens -> bounded VRAM. The coarse saliency is
    upsampled back to the depth-map resolution before projection, so 3D quality is unaffected."""
    step = _PATCH * _MERGE
    max_side = MF.MAX_SIDE if max_side is None else max_side
    w, h = img.size
    s = min(1.0, max_side / float(max(w, h)))
    nw = max(step, int(round(w * s)) // step * step)
    nh = max(step, int(round(h * s)) // step * step)
    return img.resize((nw, nh), PILImage.BILINEAR)

def _build_video_messages(frames, instruction, contract):
    """One memory-free user turn: a native VIDEO clip (duplicated frames) + the contract text,
    mirroring §13a's single-image IG so the attribution isolates the visual window."""
    reps = _TPS if MF.DUPLICATE else 1
    vid = [f for f in frames for _ in range(reps)]      # -> grid_t == len(frames) when DUPLICATE
    content = [{"type": "video", "video": vid},
               {"type": "text",
                "text": f"{contract}\nVisual memory: the clip is your last {len(frames)} onboard "
                        f"frames ~{MF.FRAME_DT:g}s apart (oldest to newest).\n\nInstruction: {instruction}"}]
    return [{"role": "user", "content": content}]

def _frame_patch_grid(block, gh, gw):
    """Un-scramble one frame's flat patch-saliency block. Qwen flattens patches in
    (grid_h/merge, grid_w/merge, merge, merge) order; undo it to a proper (gh, gw) patch image."""
    m = _MERGE
    return (block.reshape(gh // m, gw // m, m, m)
                 .transpose(0, 2, 1, 3)
                 .reshape(gh, gw))

def _normalize_frames(grids, mode):
    """Scale a query's per-frame RAW |IG| grids before projection. This is the *temporal*
    normalization choice — how the WINDOW_FRAMES maps relate to each other and to other queries:
      "per_frame": each frame min-max -> [0,1] on its own. Most uniform-looking, but DESTROYS
                   cross-frame magnitude (a barely-attended frame looks as hot as a decisive one).
      "per_query": ONE shared scale (robust max over the whole window) -> frames stay comparable
                   to each other (which frame mattered survives) and every query lands on a common
                   [0,1] scale, so the 3D accumulation isn't hijacked by a single high-gradient
                   query. Balanced + stable for cumulative heatmaps.  [default]
      "global":    keep RAW |IG| (no per-call rescale). Most faithful to IG completeness
                   (sum of attributions = f(clip) - f(black)); the render's robust-quantile then
                   does the ONE global normalization. Most accurate, least even-looking.
    The final per-surface display scaling (robust quantile + mean-per-bin) happens later in
    render_heatmap_3d, independently of this choice."""
    if mode == "per_frame":
        return [(g - g.min()) / (np.ptp(g) + 1e-9) for g in grids]
    if mode == "global":
        return [np.maximum(g, 0.0) for g in grids]
    stack = np.concatenate([g.ravel() for g in grids])          # "per_query" (default)
    lo = float(stack.min())
    hi = float(np.quantile(stack - lo, MF.SAL_ROBUST_Q))
    hi = hi if hi > 1e-12 else float(stack.max() - lo + 1e-9)
    return [np.clip((g - lo) / hi, 0.0, 1.0) for g in grids]

def integrated_gradients_video(frames, instruction, component="w_z", steps=None,
                               contract=None, out_size=None):
    """IG of the first numeric token's logit (for `component`) w.r.t. the VIDEO pixels,
    baseline = all-black video. Returns (maps, decoded_text) where `maps` is a list of HxW
    float32 saliency maps in [0,1], ONE per original frame (when MF.DUPLICATE), already
    upsampled to `out_size` (defaults to the frames' native size)."""
    steps    = MF.IG_STEPS if steps is None else steps
    contract = CFG.CONTRACT if contract is None else contract
    out_size = out_size or frames[0].size

    small = [_resize_for_ig(f) for f in frames]
    messages = _build_video_messages(small, instruction, contract)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    _img, _vid = process_vision_info(messages)
    base = processor(text=[text], images=_img, videos=_vid, padding=True,
                     return_tensors="pt").to(model.device)
    if "pixel_values_videos" not in base or "video_grid_thw" not in base:
        raise RuntimeError("Qwen did not ingest the clip as a video (no pixel_values_videos)")
    grid = base["video_grid_thw"]                       # (1,3) = (t,h,w) in patch units
    t, gh, gw = [int(x) for x in grid[0].tolist()]

    # 1) greedy-decode once to locate the numeric token span for `component`
    with torch.no_grad():
        gen = model.generate(**base, max_new_tokens=CFG.VLM_MAX_NEW_TOKENS,
                             do_sample=False, return_dict_in_generate=True)
    gen_ids = gen.sequences[0][base.input_ids.shape[1]:]
    dec = [processor.tokenizer.decode([x]) for x in gen_ids.tolist()]
    decoded_text = processor.decode(gen_ids, skip_special_tokens=True)
    key = component.replace("_", "")
    tgt = None
    for i, tok in enumerate(dec):
        if any(c.isdigit() for c in tok):
            ctx = "".join(dec[max(0, i - 8):i]).lower().replace("_", "")
            if key in ctx:
                tgt = i; break
    used_fallback = tgt is None
    if used_fallback:                                    # fallback: first numeric token anywhere
        for i, tok in enumerate(dec):
            if any(c.isdigit() for c in tok):
                tgt = i; break
    if tgt is None:
        raise RuntimeError("no numeric output token found for IG target")
    _SIGN, _NUM = set("+-"), set("0123456789.")
    start = tgt
    if tgt > 0 and dec[tgt - 1].strip() and all(c in _SIGN for c in dec[tgt - 1].strip()):
        start = tgt - 1                                  # include sign token (" -")
    end = tgt
    while end + 1 < len(dec) and dec[end + 1].strip() and all(c in _NUM for c in dec[end + 1].strip()):
        end += 1                                         # include trailing ".", digits
    span = list(range(start, end + 1))
    span_str = "".join(dec[p] for p in span).strip()
    full_ids = torch.cat([base.input_ids[0], gen_ids[:end + 1]]).unsqueeze(0)
    prompt_len = base.input_ids.shape[1]
    span_li  = [prompt_len + p - 1 for p in span]
    span_ids = torch.tensor([int(gen_ids[p]) for p in span], device=model.device)
    _flag = f"  !! FALLBACK: '{component}' label not found in output -> attributing first number instead" if used_fallback else ""
    print(f"   IG target [{component}] = '{span_str}'  ({len(span)} tok, sign-inclusive){_flag}")

    # 2) IG over pixel_values_videos (baseline = black). pixel routing stays inside the model
    #    forward via video_grid_thw (+ second_per_grid_ts) -> NO manual splice.
    pv = base["pixel_values_videos"]
    baseline = torch.zeros_like(pv)
    total_grad = torch.zeros_like(pv, dtype=torch.float32)
    extra = {k: v for k, v in base.items()
             if k not in ("input_ids", "attention_mask", "pixel_values_videos")}
    # transformers v5 returns a PROMPT-length `mm_token_type_ids` (0=text, 1=image, 2=video);
    # we teacher-force the answer tokens onto `full_ids`, so it must be extended to match or
    # get_rope_index masks it with the longer attention_mask -> "shape of the mask [...] does
    # not match the shape of the indexed tensor [...]". Appended answer tokens are text (==0).
    # (4.x transformers omit this key entirely, so this block is simply skipped there.)
    if "mm_token_type_ids" in extra:
        _mtt  = extra["mm_token_type_ids"]
        _npad = full_ids.shape[1] - _mtt.shape[1]
        if _npad > 0:
            extra["mm_token_type_ids"] = torch.cat(
                [_mtt, _mtt.new_zeros((_mtt.shape[0], _npad))], dim=1)
    with torch.enable_grad():
        for s in range(1, steps + 1):
            a = s / steps
            pv_s = (baseline + a * (pv - baseline)).clone().requires_grad_(True)
            out = model(input_ids=full_ids, attention_mask=torch.ones_like(full_ids),
                        pixel_values_videos=pv_s, **extra)
            rows = out.logits[0][span_li].float()
            logp = rows.gather(1, span_ids[:, None]).squeeze(1) - torch.logsumexp(rows, dim=-1)
            grad, = torch.autograd.grad(logp.sum(), pv_s, retain_graph=False)
            total_grad += grad.float()
            del out, grad, pv_s
    ig = ((pv - baseline).float() * total_grad / steps)
    patch_sal = ig.abs().sum(dim=-1).detach().cpu().numpy()     # (t*gh*gw,)

    per = gh * gw
    grids = [_frame_patch_grid(patch_sal[k * per:(k + 1) * per], gh, gw).astype(np.float32)
             for k in range(t)]                                 # t == n_frames when DUPLICATE
    grids = _normalize_frames(grids, MF.SAL_NORM)               # cross-frame/-time scaling (see MF.SAL_NORM)
    maps = [np.asarray(PILImage.fromarray(np.asarray(g, np.float32)).resize(out_size, PILImage.BILINEAR),
                       dtype=np.float32)                        # float resize: no [0,1] quantisation step
            for g in grids]

    del base, gen, ig, total_grad, patch_sal, pv, baseline
    gc.collect(); torch.cuda.empty_cache()
    return maps, decoded_text

# ---- multi-frame rollout: drive with planner_v2, MF-IG over the recent video window --------
def rollout_project_mf(env, policy, drive_planner, left, right, instruction,
                       component=None, ig_steps=None, sample_every_q=None,
                       temperature=0.7, seed=0, max_seconds=None, return_path=False):
    """Drive with the (no-anchor) MEM-V2 planner; at sampled queries run MULTI-FRAME video IG
    over the last MF.WINDOW_FRAMES onboard frames (captured ~MF.FRAME_DT apart, real motion) and
    project EACH frame's saliency with that frame's own pose+depth. Returns (XYZ, W[, path])."""
    component      = MF.COMPONENT       if component      is None else component
    ig_steps       = MF.IG_STEPS        if ig_steps       is None else ig_steps
    sample_every_q = MF.SAMPLE_EVERY_Q  if sample_every_q is None else sample_every_q
    max_seconds    = MF.MAX_SECONDS     if max_seconds    is None else max_seconds

    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(max_seconds / env.dt); K = onboard_K()
    drive_planner.reset(); obs, _ = env.reset(); cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
    win = collections.deque(maxlen=MF.WINDOW_FRAMES); last_push = -1e9
    pts, ws, qn, path = [], [], 0, []
    with torch.no_grad():
        for tstep in range(max_steps):
            now = tstep * env.dt
            frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
            # capture the window (count + stride matched to short-term memory) from per-step renders (frame+depth+pose @ capture)
            if (now - last_push) >= MF.FRAME_DT - 1e-6:
                cp, cl = onboard_pose(env)
                win.append(dict(frame=frame, depth=depth, pos=cp, look=cl)); last_push = now
            if tstep % query_every == 0:
                r = drive_planner.act(frame, instruction, sim_time=now,
                                      temperature=temperature, seed=seed + tstep)
                cmd = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]], float) + (1-CFG.LOWPASS_ALPHA)*cmd
                if depth is not None and len(win) == MF.WINDOW_FRAMES and qn % sample_every_q == 0:
                    try:
                        frames_w = [w["frame"] for w in win]
                        maps, _ = integrated_gradients_video(frames_w, instruction,
                                                             component=component, steps=ig_steps)
                        for wk, sal in zip(win, maps):     # project each frame at its own pose/depth
                            P, Wt = project_saliency(sal, wk["depth"], wk["pos"], wk["look"], K)
                            if len(P): pts.append(P); ws.append(Wt)
                    except Exception as e:
                        print("   [mf-ig] skip:", repr(e))
                qn += 1
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            a = policy(obs); obs, _, done, _ = env.step(a)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
            p2 = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0][:2]
            path.append(p2.copy())
            if any(np.linalg.norm(p2 - np.array(o["pos"][:2])) < CFG.ARENA.commit_radius for o in (left, right)):
                break
    P  = np.concatenate(pts, 0) if pts else np.zeros((0, 3))
    Wt = np.concatenate(ws, 0)  if ws  else np.zeros((0,))
    pa = np.asarray(path, float) if path else np.zeros((0, 2))
    return (P, Wt, pa) if return_path else (P, Wt)

# ---- convergence sweep (n = 1,5,10,20), mirrors §13c but with multi-frame video IG ---------
MF_CACHE = {}     # cat -> list of (P, W, path) per trial (shared by §13e and §13f)
MF_SEEDS = {}     # cat -> base seed actually used (printed; reproducible)
_MF_RNG  = np.random.default_rng()

# resolve (and cache) a category's base trial seed for multi-frame IG
def _mf_seed_base(cat, seed=None):
    if seed is None: seed = MF.SEED
    if seed is None: seed = MF_SEEDS.get(cat)
    if seed is None: seed = int(_MF_RNG.integers(0, 2**31 - 1))
    seed = int(seed)
    prev = MF_SEEDS.get(cat)
    if prev is not None and prev != seed and MF_CACHE.get(cat):
        print(f"  \u26a0 seed changed {prev} -> {seed}; clearing {len(MF_CACHE[cat])} cached MF trials")
        MF_CACHE[cat] = []
    MF_SEEDS[cat] = seed
    return seed

# drive/cache trials, render multi-frame-IG saliency at each checkpoint
def run_mf_convergence_category(arr_name, lc, rc, prompt, checkpoints=None, seed=None):
    from IPython.display import Image as IPImage, display
    cks = sorted(set(checkpoints or MF.CHECKPOINTS)); nmax = cks[-1]
    cat = f"{arr_name} | {prompt}"
    base = _mf_seed_base(cat, seed)
    print(f"\n######## FAITHFUL MULTI-FRAME IG CONVERGENCE  {cat}  |  n in {cks} ########")
    print(f"  window = {MF.WINDOW_FRAMES} frames @ {MF.FRAME_DT:g}s  duplicate={MF.DUPLICATE}  "
          f"IG_STEPS={MF.IG_STEPS}  component={MF.COMPONENT}  max_side={MF.MAX_SIDE}")
    print(f"  seed base = {base}   trial seeds = {base}..{base + nmax - 1}   (reproduce: seed={base})")
    trials = MF_CACHE.setdefault(cat, [])
    if len(trials) < nmax:
        env, left, right = build_arena(left_color=lc, right_color=rc)
        policy = load_policy(env)
        for i in range(len(trials), nmax):
            P, Wt, path = rollout_project_mf(env, policy, planner_v2, left, right, prompt,
                                             temperature=CFG.VLM_TEMPERATURE, seed=base + i,
                                             return_path=True)
            trials.append((P, Wt, path))
            print(f"  trial {i+1}/{nmax} (seed {base + i}): {len(P)} pts, path len {len(path)}")
        del env, policy; gc.collect(); torch.cuda.empty_cache()
    else:
        left, right = U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                                       left_color=lc, right_color=rc)
        print(f"  (reusing {len(trials)} cached MF trials @ seed base {base})")

    pngs = []
    for n in cks:
        sub = trials[:n]
        XYZ = np.concatenate([p for p, _, _ in sub if len(p)]) if any(len(p) for p, _, _ in sub) else np.zeros((0, 3))
        Wt  = np.concatenate([w for _, w, _ in sub if len(w)]) if any(len(w) for _, w, _ in sub) else np.zeros((0,))
        paths = [pa for _, _, pa in sub if len(pa)]
        png = render_heatmap_3d(XYZ, Wt, left, right, f"{cat}   |   MF-IG   |   n = {n} trials",
                                f"{MF.OUT_CONV}/{_slug(cat)}_n{n:02d}", paths=paths)
        pngs.append(png); print(f"  n={n}: {len(XYZ)} pts, {len(paths)} trails -> {png}")
        display(IPImage(png))
    return pngs

print("§13e multi-frame video-IG ready (temporal_patch_size=%d, merge_size=%d). "
      "Run a per-category §13e cell below." % (_TPS, _MERGE))

In [ ]:
# §13e-1  Convergence [video_nomem]: L=red,R=blue | friendly
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="video_nomem")

In [ ]:
# §13e-2  Convergence [video_nomem]: L=red,R=blue | hostile
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="video_nomem")

In [ ]:
# §13e-3  Convergence [video_nomem]: L=blue,R=red | friendly
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="video_nomem")

In [ ]:
# §13e-4  Convergence [video_nomem]: L=blue,R=red | hostile
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="video_nomem")

### §13f — Chosen-$w_z$ IG, **recent-frame video, no memory** — outcome split (N = 20)

Same trials/saliency as §13e, split by outcome.

In [ ]:
# §13f-setup  Multi-frame video-IG, SPLIT BY OUTCOME (mirrors §13d). NOTE: the §13f-1..4 cells
#   below run through the §13-ABL engine (`run_ablation_split(mode="video_nomem")`);
#   `run_mf_split_category` here is the standalone live counterpart. Shares MF_CACHE + MF_SEEDS
#   with §13e's legacy runner, so the per-category random seed is identical and the SAME cached
#   trials are reused.
#   Pass seed=<base> to reproduce a run. Paints each outcome group (reached/drifted/nearer-red/
#   nearer-blue) as its own 2-angle image.
import os, gc, torch
import numpy as np

MF_SPLIT = SimpleNamespace(N=20, OUT=MF.OUT_SPLIT)
os.makedirs(MF_SPLIT.OUT, exist_ok=True)

# did the trial reach a goal, and which colour is the final position nearer to?
def _mf_classify_trial(path, red_xy, blue_xy, commit_radius):
    pa = np.asarray(path, float).reshape(-1, 2)
    if not len(pa):
        return False, "nearer_red"
    final = pa[-1]
    dmin = min(np.min(np.linalg.norm(pa - red_xy, axis=1)),
               np.min(np.linalg.norm(pa - blue_xy, axis=1)))
    reached = bool(dmin < commit_radius)
    nearer = "nearer_red" if np.linalg.norm(final - red_xy) <= np.linalg.norm(final - blue_xy) else "nearer_blue"
    return reached, nearer

# drive/cache multi-frame-IG trials, then split the saliency by outcome
def run_mf_split_category(arr_name, lc, rc, prompt, n_trials=None, seed=None):
    from IPython.display import Image as IPImage, display
    n_trials = int(n_trials or MF_SPLIT.N)
    cat = f"{arr_name} | {prompt}"
    base = _mf_seed_base(cat, seed)
    print(f"\n######## MF-IG OUTCOME-SPLIT  {cat}  |  N = {n_trials} trials ########")
    print(f"  window = {MF.WINDOW_FRAMES} frames @ {MF.FRAME_DT:g}s  duplicate={MF.DUPLICATE}  "
          f"IG_STEPS={MF.IG_STEPS}  component={MF.COMPONENT}")
    print(f"  seed base = {base}   trial seeds = {base}..{base + n_trials - 1}   (reproduce: seed={base})")
    trials = MF_CACHE.setdefault(cat, [])                 # shared with §13e (same seeds)
    if len(trials) < n_trials:
        env, left, right = build_arena(left_color=lc, right_color=rc)
        policy = load_policy(env)
        for i in range(len(trials), n_trials):
            P, Wt, path = rollout_project_mf(env, policy, planner_v2, left, right, prompt,
                                             temperature=CFG.VLM_TEMPERATURE, seed=base + i,
                                             return_path=True)
            trials.append((P, Wt, path))
            print(f"  trial {i+1}/{n_trials} (seed {base + i}): {len(P)} pts, path len {len(path)}")
        del env, policy; gc.collect(); torch.cuda.empty_cache()
    else:
        left, right = U.goal_positions(CFG.ARENA.distance, CFG.ARENA.lateral, CFG.ARENA.box_size,
                                       left_color=lc, right_color=rc)
        print(f"  (reusing {len(trials)} cached MF trials @ seed base {base})")

    red_xy = blue_xy = None
    for o in (left, right):
        if color_name(o["color"]) == "red": red_xy = np.array(o["pos"][:2], float)
        else: blue_xy = np.array(o["pos"][:2], float)
    cr = CFG.ARENA.commit_radius

    groups = {"reached": [], "drifted": [], "nearer_red": [], "nearer_blue": []}
    for (P, Wt, path) in trials[:n_trials]:
        reached, nearer = _mf_classify_trial(path, red_xy, blue_xy, cr)
        groups["reached" if reached else "drifted"].append((P, Wt, path))
        groups[nearer].append((P, Wt, path))
    print("  groups:", {k: len(v) for k, v in groups.items()})

    pngs = []
    for key, label in [("reached", "reached"), ("drifted", "drifted"),
                       ("nearer_red", "nearer \u2192 RED"), ("nearer_blue", "nearer \u2192 BLUE")]:
        sub = groups[key]
        if not sub:
            print(f"  [{label}] no trials — skipped"); continue
        XYZ = np.concatenate([p for p, _, _ in sub if len(p)]) if any(len(p) for p, _, _ in sub) else np.zeros((0, 3))
        Wt = np.concatenate([w for _, w, _ in sub if len(w)]) if any(len(w) for _, w, _ in sub) else np.zeros((0,))
        paths = [pa for _, _, pa in sub if len(pa)]
        png = render_heatmap_3d(XYZ, Wt, left, right, f"{cat}   |   MF-IG {label}   (n = {len(sub)} trials)",
                                f"{MF_SPLIT.OUT}/{_slug(cat)}_{key}", paths=paths)
        pngs.append(png); print(f"  [{label}] {len(sub)} trials, {len(XYZ)} pts -> {png}")
        display(IPImage(png))
    return pngs

print("§13f MF-IG outcome-split runner ready (shares §13e seeds/cache). Run a per-category §13f cell below.")

In [ ]:
# §13f-1  Outcome split [video_nomem]: L=red,R=blue | friendly
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="video_nomem")

In [ ]:
# §13f-2  Outcome split [video_nomem]: L=red,R=blue | hostile
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="video_nomem")

In [ ]:
# §13f-3  Outcome split [video_nomem]: L=blue,R=red | friendly
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="video_nomem")

In [ ]:
# §13f-4  Outcome split [video_nomem]: L=blue,R=red | hostile
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="video_nomem")

### §13g — Chosen-$w_z$ IG, **single onboard frame + long-term text memory** — convergence (n = 1, 5, 10, 20)

Isolates the **long-term language memory** while holding vision at a single frame: attribute the chosen $w_z$ to one onboard frame, now **with** the running text summary the agent wrote (the snapshot that conditioned this very output). Reuses the SAME shared drive trials as §13c/§13e (`mode="single_mem"`).

In [ ]:
# §13g-1  Ablation [single_mem]: L=red,R=blue | friendly
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="single_mem")

In [ ]:
# §13g-2  Ablation [single_mem]: L=red,R=blue | hostile
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="single_mem")

In [ ]:
# §13g-3  Ablation [single_mem]: L=blue,R=red | friendly
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="single_mem")

In [ ]:
# §13g-4  Ablation [single_mem]: L=blue,R=red | hostile
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="single_mem")

### §13h — Chosen-$w_z$ IG, **single onboard frame + long-term text memory** — outcome split (N = 20)

Same trials/saliency as §13g, split by outcome (reached / drifted / nearer-red / nearer-blue).

In [ ]:
# §13h-1  Ablation [single_mem]: L=red,R=blue | friendly
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="single_mem")

In [ ]:
# §13h-2  Ablation [single_mem]: L=red,R=blue | hostile
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="single_mem")

In [ ]:
# §13h-3  Ablation [single_mem]: L=blue,R=red | friendly
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="single_mem")

In [ ]:
# §13h-4  Ablation [single_mem]: L=blue,R=red | hostile
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="single_mem")

### §13i — Chosen-$w_z$ IG, **recent-frame video + long-term text memory** — convergence (n = 1, 5, 10, 20)  ⭐ most faithful

The **full MEM input**: recent-frame short-term visual memory **and** long-term language memory, attributing the $w_z$ the agent actually committed to. This is the most faithful reconstruction of what truly drove the decision. Reuses the SAME shared drive trials (`mode="video_mem"`).

In [ ]:
# §13i-1  Ablation [video_mem]: L=red,R=blue | friendly
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="video_mem")

In [ ]:
# §13i-2  Ablation [video_mem]: L=red,R=blue | hostile
run_ablation_convergence("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="video_mem")

In [ ]:
# §13i-3  Ablation [video_mem]: L=blue,R=red | friendly
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="video_mem")

In [ ]:
# §13i-4  Ablation [video_mem]: L=blue,R=red | hostile
run_ablation_convergence("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="video_mem")

### §13j — Chosen-$w_z$ IG, **recent-frame video + long-term text memory** — outcome split (N = 20)  ⭐ most faithful

Same trials/saliency as §13i, split by outcome. Compare §13d→§13f→§13h→§13j to read off how each MEM component (short-term frames vs long-term text) shifts *which environmental details* drive the agent's steering.

In [ ]:
# §13j-1  Ablation [video_mem]: L=red,R=blue | friendly
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0], mode="video_mem")

In [ ]:
# §13j-2  Ablation [video_mem]: L=red,R=blue | hostile
run_ablation_split("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1], mode="video_mem")

In [ ]:
# §13j-3  Ablation [video_mem]: L=blue,R=red | friendly
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0], mode="video_mem")

In [ ]:
# §13j-4  Ablation [video_mem]: L=blue,R=red | hostile
run_ablation_split("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1], mode="video_mem")

## §14a — Saliency-accumulation **videos**: single onboard frame (**baseline**)

One trial per condition, the agent moving through the arena while saliency accumulates in 3D. This is the **naive single-frame baseline**: the agent still drives with its full MEM memory, but each map attributes the agent's *actually chosen* $w_z$ to **only the single newest onboard frame, with no memory** in the IG prompt. It exists to be compared against §14b's faithful 4D method — the difference shows what the spatio-temporal (recent-frame + memory) attribution adds. Each query's raw 2D map is also exported (pre-projection) to `saliency2d/baseline_single/`.

In [ ]:
# §14a  SINGLE-ONBOARD-FRAME baseline accumulation video (naive 2D saliency, chosen w_z). One MP4 per condition
#       showing saliency accumulating in time, from TWO viewing angles, with horizon
#       saliency painted on the back wall. Drives with planner_v2, teacher-forces the chosen
#       w_z via §13-ABL's integrated_gradients_chosen, and reuses §13a's projection machinery.
import os, gc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.cm as cm, matplotlib.colors as mcolors
from PIL import Image as PILImage
import imageio.v2 as imageio

VIDEO = SimpleNamespace(
    FPS=10, VIDEO_EVERY=8,           # snapshot a frame every N sim steps (0.02 s each)
    IG_STEPS=8, SAMPLE_EVERY_Q=3,    # attribution cost per sampled query
    COMPONENT="w_z", MAX_SECONDS=8.0, HOLD=10,
    VIEWS=VIEWS, GRID_BINS=60, FACE_BINS=7, WALL_BINS=(52, 22),
    SMOOTH_SIGMA=1.1, SLAB=0.06, FRUSTUM_MAXDIST=4.5,
    SHOW_SIM=True, SIM_RADIUS=5.4, SEPARATE=True, SEED=None,
    OUT=f"{CFG.SALIENCY_DIR}/mem_3d_video")
os.makedirs(VIDEO.OUT, exist_ok=True)
_C = matplotlib.colormaps["turbo"]; _N = mcolors.Normalize(0.0, 1.0); _WHITE = np.array([1.0, 1.0, 1.0, 1.0])


# goal-cube geometry (center / half-extent / colour) for the left and right cubes
def _cubes_from(left, right):
    out = []
    for o in (left, right):
        cx, cy, _ = o["pos"]; sx, sy, sz = o["size"]
        out.append(dict(cx=cx, cy=cy, hx=sx/2, hy=sy/2, sz=sz, name=color_name(o["color"]).capitalize()))
    return out

# a goal cube's five visible rectangular faces (local copy of the §13a helper)
def _faces(cb):
    cx, cy, hx, hy, sz = cb["cx"], cb["cy"], cb["hx"], cb["hy"], cb["sz"]
    return [(2, sz, (0, cx-hx, cx+hx), (1, cy-hy, cy+hy)),
            (0, cx-hx, (1, cy-hy, cy+hy), (2, 0, sz)), (0, cx+hx, (1, cy-hy, cy+hy), (2, 0, sz)),
            (1, cy-hy, (0, cx-hx, cx+hx), (2, 0, sz)), (1, cy+hy, (0, cx-hx, cx+hx), (2, 0, sz))]


# partition saliency points into per-cube and floor buckets
def _split(XYZ, W, cubes, slab):
    rest = np.ones(len(XYZ), bool); per = []
    for cb in cubes:
        inb = ((np.abs(XYZ[:, 0]-cb["cx"]) <= cb["hx"]+slab) & (np.abs(XYZ[:, 1]-cb["cy"]) <= cb["hy"]+slab) &
               (XYZ[:, 2] <= cb["sz"]+slab) & (XYZ[:, 2] >= -slab) & (XYZ[:, 2] > 0.03))
        per.append((XYZ[inb], W[inb])); rest &= ~inb
    wm = rest & (np.abs(XYZ[:, 0]-X_WALL) < 1e-3) & (XYZ[:, 2] > 0.03)
    fl = rest & ~wm
    return (XYZ[fl], W[fl]), (XYZ[wm], W[wm]), per


# 2D weighted density histogram, smoothed
def _dens(P, W, ia, ib, ae, be, ca=None, plane=None, slab=None, smooth=0.0):
    if ca is not None and len(P):
        near = np.abs(P[:, ca]-plane) < slab; P, W = P[near], W[near]
    if len(P):
        s, _, _ = np.histogram2d(P[:, ia], P[:, ib], bins=[ae, be], weights=W)
        c, _, _ = np.histogram2d(P[:, ia], P[:, ib], bins=[ae, be])
    else:
        s = np.zeros((len(ae)-1, len(be)-1)); c = np.zeros_like(s)
    if smooth: s = _smooth(s, smooth); c = _smooth(c, smooth)
    cov = c > (0.04*c.max() if (smooth and c.max() > 0) else 0)
    d = np.zeros_like(s); d[cov] = s[cov]/(c[cov]+1e-9)
    return d, cov


# shared colour-scale max across the floor and cube densities
def compute_vmax(XYZ, W, cubes, xr, yr):
    (fP, fW), (wP, wW), per = _split(np.asarray(XYZ).reshape(-1, 3), np.asarray(W).reshape(-1), cubes, VIDEO.SLAB)
    xe = np.linspace(*xr, VIDEO.GRID_BINS+1); ye = np.linspace(*yr, VIDEO.GRID_BINS+1)
    pool = []
    d, cov = _dens(fP, fW, 0, 1, xe, ye, smooth=VIDEO.SMOOTH_SIGMA)
    if cov.any(): pool.append(d[cov])
    wye = np.linspace(*yr, VIDEO.WALL_BINS[0]+1); wze = np.linspace(0, Z_TOP, VIDEO.WALL_BINS[1]+1)
    d, cov = _dens(wP, wW, 1, 2, wye, wze, smooth=VIDEO.SMOOTH_SIGMA)
    if cov.any(): pool.append(d[cov])
    for cb, (P, W_) in zip(cubes, per):
        for (ca, pl, (ia, alo, ahi), (ib, blo, bhi)) in _faces(cb):
            ae = np.linspace(alo, ahi, VIDEO.FACE_BINS+1); be = np.linspace(blo, bhi, VIDEO.FACE_BINS+1)
            d, cov = _dens(P, W_, ia, ib, ae, be, ca=ca, plane=pl, slab=VIDEO.SLAB)
            if cov.any(): pool.append(d[cov])
    pool = np.concatenate(pool) if pool else np.array([1.0])
    v = np.quantile(pool, 0.99) if len(pool) > 5 else (pool.max() if len(pool) else 1.0)
    return v if v > 1e-9 else 1.0


# draw the floor + cube heatmaps on one 3D axis
def _paint(ax, XYZ, W, cubes, xr, yr, vmax):
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
    XYZ = np.asarray(XYZ, float).reshape(-1, 3); W = np.asarray(W, float).reshape(-1)
    (fP, fW), (wP, wW), per = _split(XYZ, W, cubes, VIDEO.SLAB)
    xe = np.linspace(*xr, VIDEO.GRID_BINS+1); ye = np.linspace(*yr, VIDEO.GRID_BINS+1)
    d, cov = _dens(fP, fW, 0, 1, xe, ye, smooth=VIDEO.SMOOTH_SIGMA)
    rgba = _C(_N(np.clip(d/vmax, 0, 1))); rgba[~cov, 3] = 0.0
    xc = 0.5*(xe[:-1]+xe[1:]); yc = 0.5*(ye[:-1]+ye[1:]); Xg, Yg = np.meshgrid(xc, yc, indexing="ij")
    ax.plot_surface(Xg, Yg, np.zeros_like(Xg), facecolors=rgba, rcount=VIDEO.GRID_BINS, ccount=VIDEO.GRID_BINS,
                    shade=False, linewidth=0, antialiased=False, zorder=1)
    wye = np.linspace(*yr, VIDEO.WALL_BINS[0]+1); wze = np.linspace(0, Z_TOP, VIDEO.WALL_BINS[1]+1)
    d, cov = _dens(wP, wW, 1, 2, wye, wze, smooth=VIDEO.SMOOTH_SIGMA)
    wr = _C(_N(np.clip(d/vmax, 0, 1))); wr[~cov, 3] = 0.0
    wyc = 0.5*(wye[:-1]+wye[1:]); wzc = 0.5*(wze[:-1]+wze[1:]); Yw, Zw = np.meshgrid(wyc, wzc, indexing="ij")
    ax.plot_surface(np.full_like(Yw, X_WALL), Yw, Zw, facecolors=wr, rcount=VIDEO.WALL_BINS[0],
                    ccount=VIDEO.WALL_BINS[1], shade=False, linewidth=0, antialiased=False, zorder=1)

    # corner helper (local copy of the §13a helper)
    def _cn(ca, pl, ia, av, ib, bv):
        p = [0.0, 0.0, 0.0]; p[ca] = pl; p[ia] = av; p[ib] = bv; return p
    for cb, (P, W_) in zip(cubes, per):
        quads, qcols = [], []
        for (ca, pl, (ia, alo, ahi), (ib, blo, bhi)) in _faces(cb):
            ae = np.linspace(alo, ahi, VIDEO.FACE_BINS+1); be = np.linspace(blo, bhi, VIDEO.FACE_BINS+1)
            d, cov = _dens(P, W_, ia, ib, ae, be, ca=ca, plane=pl, slab=VIDEO.SLAB)
            fr = _C(_N(np.clip(d/vmax, 0, 1))); fr[~cov] = _WHITE
            for i in range(VIDEO.FACE_BINS):
                for j in range(VIDEO.FACE_BINS):
                    quads.append([_cn(ca, pl, ia, ae[i], ib, be[j]), _cn(ca, pl, ia, ae[i+1], ib, be[j]),
                                  _cn(ca, pl, ia, ae[i+1], ib, be[j+1]), _cn(ca, pl, ia, ae[i], ib, be[j+1])])
                    qcols.append(fr[i, j])
        pc = Poly3DCollection(quads, facecolors=qcols, edgecolors="none", shade=False); pc.set_zorder(3)
        ax.add_collection3d(pc)
        cx, cy, hx, hy, sz = cb["cx"], cb["cy"], cb["hx"], cb["hy"], cb["sz"]
        c = np.array([[cx-hx,cy-hy,0],[cx+hx,cy-hy,0],[cx+hx,cy+hy,0],[cx-hx,cy+hy,0],
                      [cx-hx,cy-hy,sz],[cx+hx,cy-hy,sz],[cx+hx,cy+hy,sz],[cx-hx,cy+hy,sz]])
        E = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
        lc = Line3DCollection([[c[a], c[b]] for a, b in E], colors="k", linewidths=1.3); lc.set_zorder(5)
        ax.add_collection3d(lc)


# the camera frustum's intersection with the ground, for the overlay
def frustum_ground_lines(cam_pos, cam_look, K, res, max_dist):
    W, Hh = res; _, T = U.look_at_extrinsics(cam_pos, cam_look, up=(0, 0, 1))
    R = T[:3, :3]; cam = np.asarray(cam_pos, float); fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]; segs = []
    for (u, v) in [(0, Hh), (W, Hh), (W, 0), (0, 0)]:
        d = R @ np.array([(u-cx)/fx, (v-cy)/fy, 1.0]); d /= np.linalg.norm(d)+1e-9
        t = min(-cam[2]/d[2], max_dist) if d[2] < -1e-3 else max_dist
        segs.append([cam, cam + t*d])
    return segs


# render one frame of the saliency-accumulation video (both viewing angles)
def render_video_frame(XYZ, W, cubes, path_xy, cam_pos, cam_look, K, title, vmax, sim_rgb=None, views=None, view_names=None):
    from mpl_toolkits.mplot3d import proj3d                       # noqa
    from mpl_toolkits.mplot3d.art3d import Line3DCollection
    xr = [-0.6, X_WALL]; yr = [-Y_HALF, Y_HALF]
    vws = views if views is not None else VIDEO.VIEWS
    vnames = view_names if view_names is not None else [f"view {n+1}" for n in range(len(vws))]
    nv = len(vws); ncol = nv + (1 if sim_rgb is not None else 0)
    fig = plt.figure(figsize=(7.0*ncol, 6.2))
    axes = []; off = 0
    if sim_rgb is not None:
        axsim = fig.add_subplot(1, ncol, 1); axsim.imshow(np.asarray(sim_rgb)); axsim.axis("off")
        axsim.set_title("Genesis sim (≈ view 1)", fontsize=10); off = 1
    for n, view in enumerate(vws):
        try:
            ax = fig.add_subplot(1, ncol, off+n+1, projection="3d", computed_zorder=False)
        except Exception:
            ax = fig.add_subplot(1, ncol, off+n+1, projection="3d")
            try: ax.computed_zorder = False
            except Exception: pass
        _paint(ax, XYZ, W, cubes, xr, yr, vmax)
        pth = np.asarray(path_xy, float).reshape(-1, 2)
        if len(pth) >= 2:
            segs = [[(pth[i,0], pth[i,1], 0.01), (pth[i+1,0], pth[i+1,1], 0.01)] for i in range(len(pth)-1)]
            cols = np.zeros((len(segs), 4)); cols[:, 3] = np.linspace(0.12, 0.9, len(segs))
            tlc = Line3DCollection(segs, colors=cols, linewidths=1.8, linestyles="--"); tlc.set_zorder(6)
            ax.add_collection3d(tlc)
        for s in frustum_ground_lines(cam_pos, cam_look, K, CFG.CAM.res, VIDEO.FRUSTUM_MAXDIST):
            s = np.array(s); ax.plot(s[:, 0], s[:, 1], s[:, 2], color="0.55", lw=0.9, alpha=0.5, zorder=7)
        ax.plot([cam_pos[0]]*2, [cam_pos[1]]*2, [0.0, cam_pos[2]], color="0.3", lw=0.8, zorder=7)
        ax.scatter([cam_pos[0]], [cam_pos[1]], [cam_pos[2]], c="k", marker="o", s=55, depthshade=False, zorder=8)
        objs = [(cb["cx"], cb["cy"], cb["sz"], cb["name"]) for cb in cubes] + [(cam_pos[0], cam_pos[1], cam_pos[2], "Agent")]
        for (x, y, ztop, _) in objs[:-1]:
            ax.plot([x, x], [y, y], [ztop, 0.95], color="k", lw=1.0, ls=(0, (2, 2)), zorder=6)
        ax.set_xlim(*xr); ax.set_ylim(*yr); ax.set_zlim(0, Z_TOP)
        ax.set_xlabel("x (fwd)"); ax.set_ylabel("y (left)"); ax.set_zticks([])
        try: ax.set_box_aspect((xr[1]-xr[0], yr[1]-yr[0], 1.3))
        except Exception: pass
        ax.view_init(elev=view[0], azim=view[1]); ax.set_title(vnames[n], fontsize=10)
        axes.append((ax, objs))

    fig.suptitle(title, color="k", fontsize=13, fontweight="bold", y=0.99)
    sm = cm.ScalarMappable(cmap=_C, norm=_N); sm.set_array([])
    cbar = fig.colorbar(sm, ax=[a for a, _ in axes], shrink=0.6, pad=0.02)
    cbar.set_label("accumulated saliency (norm.)", fontsize=10)
    cbar.ax.text(0.5, -0.05, "white = VLM blind spot", transform=cbar.ax.transAxes,
                 ha="center", va="top", fontsize=8, color="0.35")
    fig.canvas.draw()
    for ax, objs in axes:
        for (x, y, ztop, name) in objs:
            xp, yp, _ = proj3d.proj_transform(x, y, max(ztop, 0.95), ax.get_proj())
            ax.annotate(name, xy=(xp, yp), xytext=(0, 5), textcoords="offset points", ha="center", va="bottom",
                        color="k", fontsize=11, fontweight="bold", zorder=20,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))
    fig.canvas.draw()
    buf = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return buf


def _mpl_view_campose(center, radius, elev, azim):
    """Approximate the matplotlib 3D (elev, azim) framing with a fixed Genesis camera,
    so the real-sim panel roughly matches saliency view 1."""
    el = np.radians(elev); az = np.radians(azim)
    d = np.array([np.cos(el)*np.cos(az), np.cos(el)*np.sin(az), np.sin(el)])
    pos = np.asarray(center, float) + radius*d
    return tuple(pos.tolist()), tuple(np.asarray(center, float).tolist())


# NOTE: sf_planner / saliency_fn are accepted for signature compatibility but unused —
# attribution is always chosen-w_z integrated_gradients_chosen (§13-ABL).
def rollout_saliency_timeline(env, policy, drive_planner, sf_planner, left, right, instruction, saliency_fn=None, seed=0, cat=""):
    """One trial, SINGLE-ONBOARD-FRAME BASELINE (the naive 2D-saliency comparison for §14b's 4D method).
    The agent still DRIVES with its full MEM memory, but saliency is attributed to ONLY the single newest
    onboard frame, with NO memory in the IG prompt, teacher-forcing the w_z the agent actually chose.
    Projects that one frame; records path + poses + per-query saliency + chosen w_z, and exports the
    baseline 2D map (pre-projection) under saliency2d/baseline_single/."""
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(VIDEO.MAX_SECONDS / env.dt); K = onboard_K()
    drive_planner.reset(); obs, _ = env.reset(); cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
    torch.manual_seed(int(seed)); np.random.seed(int(seed) % (2**31 - 1))
    _cx = (-0.6 + X_WALL)/2.0
    sim_pos, sim_look = _mpl_view_campose((_cx, 0.0, 0.3), VIDEO.SIM_RADIUS, VIDEO.VIEWS[0][0], VIDEO.VIEWS[0][1])
    track, samples, qn, last_wz = [], [], 0, None
    with torch.no_grad():
        for t in range(max_steps):
            now = t * env.dt
            frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
            if t % query_every == 0:
                r = drive_planner.act(frame, instruction, sim_time=now,
                                      temperature=CFG.VLM_TEMPERATURE, seed=seed + t)
                cmd = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]], float) + (1-CFG.LOWPASS_ALPHA)*cmd
                if depth is not None and qn % VIDEO.SAMPLE_EVERY_Q == 0:
                    try:
                        maps, _ = integrated_gradients_chosen(
                            [frame], r.get("gen_ids") or [], None, False,
                            instruction=instruction, component=VIDEO.COMPONENT,
                            steps=VIDEO.IG_STEPS, out_size=tuple(CFG.CAM.res))
                        last_wz = r.get("w_z")
                        cp, cl = onboard_pose(env)
                        if SAL2D.ENABLE:
                            _save_sal2d(maps[0], cat, t, now, "baseline_single", rgb=frame, wz=last_wz, method="single_nomem")
                        P, Wt = project_saliency(maps[0], depth, cp, cl, K)
                        if len(P): samples.append((t, P, Wt))
                    except Exception as e:
                        print("   [single-ig] skip:", repr(e))
                qn += 1
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            a = policy(obs); obs, _, done, _ = env.step(a)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
            base = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0]
            if t % VIDEO.VIDEO_EVERY == 0:
                cp, cl = onboard_pose(env)
                sim_rgb = None
                if VIDEO.SHOW_SIM:
                    try:
                        set_level_pose(env.camera, sim_pos, sim_look)
                        srgb, _ = U.render_camera(env.camera, want_depth=True)
                        sim_rgb = np.asarray(srgb)[..., :3].copy()
                    except Exception:
                        sim_rgb = None
                track.append(dict(step=t, base_xy=base[:2].copy(),
                                  cam_pos=np.asarray(cp, float), cam_look=np.asarray(cl, float),
                                  sim_rgb=sim_rgb, wz=last_wz))
            if any(np.linalg.norm(base[:2]-np.array(o["pos"][:2])) < CFG.ARENA.commit_radius for o in (left, right)):
                break
    return track, samples


def run_saliency_video(arr_name, lc, rc, prompt, saliency_fn=None, seed=None):
    """Build arena, run ONE trial, render the accumulation video, save MP4, display."""
    from IPython.display import Video, display, HTML
    cat = f"{arr_name} | {prompt}"
    if seed is None:
        seed = VIDEO.SEED if getattr(VIDEO, "SEED", None) is not None else int(np.random.default_rng().integers(0, 2**31 - 1))
    print(f"\n######## VIDEO {cat} ########")
    print(f"  seed = {seed}   (reproduce with: run_saliency_video(..., seed={seed}))")
    env, left, right = build_arena(left_color=lc, right_color=rc)
    policy = load_policy(env)
    track, samples = rollout_saliency_timeline(env, policy, planner_v2, sf_planner, left, right, prompt,
                                               saliency_fn=saliency_fn, seed=seed, cat=cat)
    del env, policy; gc.collect(); torch.cuda.empty_cache()
    if not track:
        print("  no track recorded — skipping"); return
    cubes = _cubes_from(left, right); K = onboard_K()
    xr = [-0.6, CFG.ARENA.distance + 1.3]; yr = [-(CFG.ARENA.lateral + 1.4), CFG.ARENA.lateral + 1.4]
    if samples:
        allP = np.concatenate([P for _, P, _ in samples]); allW = np.concatenate([W for _, _, W in samples])
        vmax = compute_vmax(allP, allW, cubes, xr, yr)
    else:
        vmax = 1.0
    print(f"  {len(track)} frames | {len(samples)} IG samples | vmax={vmax:.4f}")

    # precompute per-frame specs once (cumulative saliency + path snapshot)
    specs, path = [], []
    for snap in track:
        path.append(snap["base_xy"])
        cumP = [P for (s, P, _) in samples if s <= snap["step"]]
        cumW = [W for (s, _, W) in samples if s <= snap["step"]]
        specs.append(dict(
            XYZ=np.concatenate(cumP) if cumP else np.zeros((0, 3)),
            W=np.concatenate(cumW) if cumW else np.zeros(0),
            path=list(path), cam_pos=snap["cam_pos"], cam_look=snap["cam_look"],
            title=f"{cat}   |   chosen w_z={('%+.2f'%snap['wz']) if snap.get('wz') is not None else '\u2026'}   (t={snap['step']*CFG.SIM.dt:.1f}s)", sim=snap.get("sim_rgb")))

    # write out one accumulation clip plus its still frames
    def _emit(frames, stem, caption):
        frames = list(frames) + [frames[-1]] * VIDEO.HOLD
        p = f"{VIDEO.OUT}/{_slug(cat)}_{stem}.mp4"
        try:
            imageio.mimsave(p, frames, fps=VIDEO.FPS, quality=8, macro_block_size=1)
            display(HTML(f"<b>{caption}</b>")); display(Video(p, embed=True, width=720))
            print(f"  saved -> {p}")
        except Exception as e:
            g = f"{VIDEO.OUT}/{_slug(cat)}_{stem}.gif"
            imageio.mimsave(g, frames, duration=1.0/VIDEO.FPS)
            from IPython.display import Image as IPImage
            display(HTML(f"<b>{caption}</b>")); display(IPImage(g))
            print(f"  mp4 failed ({e!r}); saved -> {g}")

    if VIDEO.SEPARATE:                                   # one player per angle + the sim, separately
        for i, view in enumerate(VIDEO.VIEWS):
            fr = [render_video_frame(sp["XYZ"], sp["W"], cubes, sp["path"], sp["cam_pos"], sp["cam_look"],
                                     K, sp["title"], vmax, views=[view], view_names=[f"view {i+1}"]) for sp in specs]
            _emit(fr, f"view{i+1}", f"saliency — view {i+1} (elev {view[0]}, azim {view[1]})")
        sim_fr = [np.asarray(sp["sim"]) for sp in specs if sp["sim"] is not None]
        if sim_fr:
            _emit(sim_fr, "sim", "Genesis sim (≈ view 1)")
    else:                                                # combined multi-panel video (legacy)
        fr = [render_video_frame(sp["XYZ"], sp["W"], cubes, sp["path"], sp["cam_pos"], sp["cam_look"],
                                 K, sp["title"], vmax, sim_rgb=sp["sim"]) for sp in specs]
        _emit(fr, "combined", cat)

print("Video utilities ready. Run a per-condition §14 cell below (one trial each).")


# ---- 2D saliency-frame EXPORT (raw onboard maps BEFORE 3D projection, for baseline-vs-4D comparison) ----
SAL2D = SimpleNamespace(
    ENABLE        = True,
    ROOT          = f"{CFG.SALIENCY_DIR}/saliency2d",
    CMAP          = "inferno",
    SAVE_NPY      = True,    # also dump the raw float map (.npy) for quantitative comparison
    OVERLAY       = True,    # also write "<stem>__overlay.png": saliency composited on the POV frame
    OVERLAY_ALPHA = 0.85,    # peak opacity of the heat mask in the overlay
    OVERLAY_GAMMA = 0.7)     # <1 lifts faint saliency; >1 suppresses it (alpha = nrm**gamma)

def _overlay_sal_on_rgb(sal, rgb, cmap_name="inferno", alpha_gamma=0.7, alpha_max=0.85):
    """Composite a saliency map (HxW) onto its onboard RGB frame as a translucent heat overlay
    (high saliency = opaque/bright, low = transparent), IG-attribution-mask style. The saliency is
    min-max normalized exactly like the stored heatmap PNG so the two stay consistent. Returns a
    uint8 HxWx3 array at the saliency-map resolution (the POV frame is resized to match if needed).
    Accepts `rgb` as either a PIL image or an HxW / HxWx3 / HxWx4 numpy array."""
    import numpy as np, matplotlib
    a = np.asarray(sal, np.float32); nrm = (a - a.min()) / (np.ptp(a) + 1e-9)
    h, w = nrm.shape
    if hasattr(rgb, "convert"):                                  # PIL image -> resize to the saliency grid
        img = rgb.convert("RGB")
        if img.size != (w, h): img = img.resize((w, h))
        base = np.asarray(img, np.float32) / 255.0
    else:                                                        # numpy array -> coerce to float HxWx3
        base = np.asarray(rgb, np.float32)
        if base.ndim == 2: base = np.repeat(base[..., None], 3, axis=2)
        base = base[..., :3]
        if base.max() > 1.5: base = base / 255.0
        if base.shape[:2] != (h, w):
            from PIL import Image as _PILImage
            src8 = (np.clip(base, 0, 1) * 255).astype("uint8")
            base = np.asarray(_PILImage.fromarray(src8).resize((w, h)), np.float32) / 255.0
    if base.ndim == 2: base = np.repeat(base[..., None], 3, axis=2)
    base = base[..., :3]
    heat = matplotlib.colormaps[cmap_name](nrm)[..., :3]         # colorize saliency with the stored cmap
    alpha = (np.clip(nrm, 0, 1) ** float(alpha_gamma) * float(alpha_max))[..., None]
    comp = base * (1.0 - alpha) + heat * alpha
    return (np.clip(comp, 0, 1) * 255).astype("uint8")


def _save_sal2d(sal, cat, step, t_sec, subdir, *, rgb=None, frame_idx=None, n_frames=None, wz=None, method=""):
    """Save ONE 2D onboard saliency map (PRE-3D-projection) as a colorized PNG (+ raw .npy) under
    SAL2D.ROOT/<subdir>/, with a human-sortable, self-describing filename:
        <category>__step####__t##.##s[__f<j>_recent<j>of<n>][__wz±#p##].png
    For multi-frame dumps, frame_idx is the recency rank (0 = most recent ... n-1 = oldest).
    If `rgb` (the onboard POV frame this map was attributed on) is supplied and SAL2D.OVERLAY is
    on, ALSO writes "<stem>__overlay.png": the saliency composited on that frame (interpretable)."""
    import os, numpy as np, matplotlib
    matplotlib.use("Agg"); import matplotlib.pyplot as plt
    if not SAL2D.ENABLE: return None
    d = f"{SAL2D.ROOT}/{subdir}"; os.makedirs(d, exist_ok=True)
    a = np.asarray(sal, np.float32); nrm = (a - a.min()) / (np.ptp(a) + 1e-9)
    parts = [_slug(cat), f"step{int(step):04d}", f"t{float(t_sec):05.2f}s"]
    if frame_idx is not None: parts.append(f"f{int(frame_idx)}_recent{int(frame_idx)}of{int(n_frames)}")
    if wz is not None:        parts.append(("wz%+.2f" % float(wz)).replace(".", "p"))
    if method:                parts.append(method)
    stem = "__".join(parts); png = f"{d}/{stem}.png"
    h, w = nrm.shape
    fig = plt.figure(figsize=(w/100.0, h/100.0), dpi=100); ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off")
    ax.imshow(nrm, cmap=SAL2D.CMAP, vmin=0.0, vmax=1.0); fig.savefig(png, dpi=100); plt.close(fig)
    if SAL2D.SAVE_NPY: np.save(f"{d}/{stem}.npy", a)
    if rgb is not None and getattr(SAL2D, "OVERLAY", True):      # interpretable overlay on the POV frame
        try:
            import imageio.v2 as _imageio
            ov = _overlay_sal_on_rgb(sal, rgb, cmap_name=SAL2D.CMAP,
                                     alpha_gamma=getattr(SAL2D, "OVERLAY_GAMMA", 0.7),
                                     alpha_max=getattr(SAL2D, "OVERLAY_ALPHA", 0.85))
            _imageio.imwrite(f"{d}/{stem}__overlay.png", ov)
        except Exception as _e:
            print("   [sal2d-overlay] skip:", repr(_e))
    return png
print("2D saliency export ready -> baseline_single/ (§14a single-frame baseline) and",
      "faithful_multi/ (§14b recent-frame 4D method, frames numbered recent->oldest).",
      "Toggle SAL2D.ENABLE; raw .npy saved alongside each PNG.")

In [ ]:
# §14a-tune  Denser IG along the trajectory -> smoother accumulation. RUN AFTER §14a, BEFORE §14-1..4.
#   SAMPLE_EVERY_Q : 3 -> 1   run IG on EVERY VLM query (≈3x more projected point clouds along the path)
#   IG_STEPS       : kept at 8 (bump to 16 for a finer Riemann sum / less per-frame noise)
#   VIDEO_EVERY    : 8 -> 4   snapshot the video twice as often — smoother playback, NOT more IG
VIDEO.SAMPLE_EVERY_Q = 1
VIDEO.IG_STEPS       = 8
VIDEO.VIDEO_EVERY  = 4
print(f"IG density bumped: SAMPLE_EVERY_Q={VIDEO.SAMPLE_EVERY_Q}, IG_STEPS={VIDEO.IG_STEPS}. "
      f"Expect ~3x more IG samples and a proportionally slower rollout. (§14 already prints + uses a random seed.)")


In [ ]:
# §14-1  Video: L=red,R=blue | friendly
run_saliency_video("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0])


In [ ]:
# §14-2  Video: L=red,R=blue | hostile
run_saliency_video("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1])


In [ ]:
# §14-3  Video: L=blue,R=red | friendly
run_saliency_video("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0])


In [ ]:
# §14-4  Video: L=blue,R=red | hostile
run_saliency_video("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1])


## §14b — Saliency-accumulation **videos**: fully faithful MEM (recent frames + text)  ⭐

The **most faithful** accumulation video: attributes the chosen $w_z$ over the **recent-frame video window AND the long-term text memory** (mode `video_mem`), projecting each recent frame at its own camera pose/depth. This is the 4D spatio-temporal method; compare it against §14a's single-frame baseline. Each query's per-frame 2D maps are exported (pre-projection, numbered most-recent→oldest) to `saliency2d/faithful_multi/`.

In [ ]:
# §14b-setup  FULLY FAITHFUL MEM saliency-accumulation VIDEOS: recent-frame window + long-term
#   text memory. Same accumulation engine as §14a, but every query attributes the CHOSEN w_z over
#   the recent VIDEO window PLUS the text-memory snapshot that conditioned it (§13-ABL's
#   integrated_gradients_chosen, mode video_mem) and projects EACH recent frame at its OWN
#   pose+depth, so the movie accumulates the whole clip the planner saw.
#   The IG target w_z value is printed per query AND drawn on each frame's title.
#   Deps: §14a (render_video_frame / compute_vmax / _cubes_from / _mpl_view_campose / SAL2D),
#         §13-ABL (integrated_gradients_chosen), §13e (MF.SAL_NORM), §13a (project_saliency),
#         §12a (planner_v2).
import os, gc, collections
import numpy as np, torch
from PIL import Image as PILImage

VIDEO_MF = SimpleNamespace(
    FPS=10, VIDEO_EVERY=8,            # snapshot a video frame every N sim steps (0.02 s each)
    IG_STEPS=8, SAMPLE_EVERY_Q=3,     # attribution cost per sampled query
    COMPONENT="w_z", MAX_SECONDS=8.0, HOLD=10,
    WINDOW_FRAMES=CFG.MEM2.RECENT_FRAMES,    # = 4: the SAME clip the planner drives on
    FRAME_DT=CFG.MEM2.SHORT_INTERVAL_S,      # = ~0.333 s stride (matches short-term memory)
    VIEWS=VIEWS, SIM_RADIUS=5.4, SHOW_SIM=True, SEPARATE=True, SEED=None,
    OUT=f"{CFG.SALIENCY_DIR}/mem_3d_video_mf")
# 3D binning / smoothing / frustum params for the render are inherited from §14a's VIDEO namespace
# (render_video_frame / compute_vmax read it directly) — kept identical so the two videos line up.
os.makedirs(VIDEO_MF.OUT, exist_ok=True)


def rollout_saliency_timeline_mf(env, policy, drive_planner, left, right, instruction, seed=0, cat=""):
    """One trial with MULTI-FRAME video IG. Drives with the (no-anchor) MEM-V2 planner; at sampled
    queries attributes over the recent video window and projects each frame at its own pose+depth.
    Records path + camera poses + per-query projected saliency for the cumulative video."""
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    max_steps = int(VIDEO_MF.MAX_SECONDS / env.dt); K = onboard_K()
    drive_planner.reset(); obs, _ = env.reset(); cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
    torch.manual_seed(int(seed)); np.random.seed(int(seed) % (2**31 - 1))
    _cx = (-0.6 + X_WALL) / 2.0
    sim_pos, sim_look = _mpl_view_campose((_cx, 0.0, 0.3), VIDEO_MF.SIM_RADIUS,
                                          VIDEO_MF.VIEWS[0][0], VIDEO_MF.VIEWS[0][1])
    win = collections.deque(maxlen=VIDEO_MF.WINDOW_FRAMES); last_push = -1e9
    track, samples, qn, last_wz = [], [], 0, None
    with torch.no_grad():
        for t in range(max_steps):
            now = t * env.dt
            frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
            if (now - last_push) >= VIDEO_MF.FRAME_DT - 1e-6:        # build the recent-frame window
                cp0, cl0 = onboard_pose(env)
                win.append(dict(frame=frame, depth=depth, pos=cp0, look=cl0)); last_push = now
            if t % query_every == 0:
                summary_before = drive_planner._language_memory   # text that conditioned THIS output
                r = drive_planner.act(frame, instruction, sim_time=now,
                                      temperature=CFG.VLM_TEMPERATURE, seed=seed + t)
                cmd = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]], float) + (1-CFG.LOWPASS_ALPHA)*cmd
                if depth is not None and len(win) >= 1 and qn % VIDEO_MF.SAMPLE_EVERY_Q == 0:
                    try:
                        # Attribute over WHATEVER recent frames already exist (1..WINDOW_FRAMES) instead of
                        # waiting for the window to fill — so heat appears from the very first query rather
                        # than ~1 s in. With a single frame this is exactly §14a's single-frame IG; it grows
                        # into the full faithful recent-frame video-IG as the window fills.
                        win_list = list(win)
                        frames_w = [w["frame"] for w in win_list]
                        _use_video = len(frames_w) > 1
                        maps, _ = integrated_gradients_chosen(
                            frames_w, r.get("gen_ids") or [], summary_before, _use_video,
                            instruction=instruction, component=VIDEO_MF.COMPONENT,
                            steps=VIDEO_MF.IG_STEPS, out_size=tuple(CFG.CAM.res))
                        last_wz = r.get("w_z")
                        if not _use_video:                    # single-frame IG attributes the NEWEST frame only
                            win_list = win_list[-1:]
                        if SAL2D.ENABLE:                      # 2D dump (+overlay): numbered recent -> oldest
                            _n = len(maps)
                            for _pos, _m in enumerate(maps):
                                _save_sal2d(_m, cat, t, now, "faithful_multi", rgb=win_list[_pos]["frame"],
                                            frame_idx=_n-1-_pos, n_frames=_n, wz=last_wz, method="video_mem")
                        Ps, Ws = [], []
                        for wk, sal in zip(win_list, maps):         # project each frame @ its own pose/depth
                            P, Wt = project_saliency(sal, wk["depth"], wk["pos"], wk["look"], K)
                            if len(P): Ps.append(P); Ws.append(Wt)
                        if Ps: samples.append((t, np.concatenate(Ps), np.concatenate(Ws)))
                    except Exception as e:
                        print("   [mf-ig] skip:", repr(e))
                qn += 1
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range)
            cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            a = policy(obs); obs, _, done, _ = env.step(a)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
            base = np.asarray(env.robot.get_pos().cpu()).reshape(-1, 3)[0]
            if t % VIDEO_MF.VIDEO_EVERY == 0:
                cp, cl = onboard_pose(env)
                sim_rgb = None
                if VIDEO_MF.SHOW_SIM:
                    try:
                        set_level_pose(env.camera, sim_pos, sim_look)
                        srgb, _ = U.render_camera(env.camera, want_depth=True)
                        sim_rgb = np.asarray(srgb)[..., :3].copy()
                    except Exception:
                        sim_rgb = None
                track.append(dict(step=t, base_xy=base[:2].copy(),
                                  cam_pos=np.asarray(cp, float), cam_look=np.asarray(cl, float),
                                  sim_rgb=sim_rgb, wz=last_wz))
            if any(np.linalg.norm(base[:2]-np.array(o["pos"][:2])) < CFG.ARENA.commit_radius for o in (left, right)):
                break
    return track, samples


def run_saliency_video_mf(arr_name, lc, rc, prompt, seed=None):
    """Build arena, run ONE trial with multi-frame video IG, render the accumulation video, save, display."""
    from IPython.display import Video, display, HTML
    cat = f"{arr_name} | {prompt}"
    if seed is None:
        seed = VIDEO_MF.SEED if getattr(VIDEO_MF, "SEED", None) is not None else int(np.random.default_rng().integers(0, 2**31 - 1))
    print(f"\n######## MULTI-FRAME VIDEO {cat} ########")
    print(f"  window = {VIDEO_MF.WINDOW_FRAMES} frames @ {VIDEO_MF.FRAME_DT:g}s  IG_STEPS={VIDEO_MF.IG_STEPS}  "
          f"component={VIDEO_MF.COMPONENT}  sal_norm={MF.SAL_NORM}")
    print(f"  seed = {seed}   (reproduce with: run_saliency_video_mf(..., seed={seed}))")
    env, left, right = build_arena(left_color=lc, right_color=rc)
    policy = load_policy(env)
    track, samples = rollout_saliency_timeline_mf(env, policy, planner_v2, left, right, prompt, seed=seed, cat=cat)
    del env, policy; gc.collect(); torch.cuda.empty_cache()
    if not track:
        print("  no track recorded — skipping"); return
    cubes = _cubes_from(left, right); K = onboard_K()
    xr = [-0.6, CFG.ARENA.distance + 1.3]; yr = [-(CFG.ARENA.lateral + 1.4), CFG.ARENA.lateral + 1.4]
    if samples:
        allP = np.concatenate([P for _, P, _ in samples]); allW = np.concatenate([W for _, _, W in samples])
        vmax = compute_vmax(allP, allW, cubes, xr, yr)
    else:
        vmax = 1.0
    print(f"  {len(track)} frames | {len(samples)} MF-IG samples | vmax={vmax:.4f}")

    specs, path = [], []
    for snap in track:
        path.append(snap["base_xy"])
        cumP = [P for (s, P, _) in samples if s <= snap["step"]]
        cumW = [W for (s, _, W) in samples if s <= snap["step"]]
        wz = snap.get("wz")
        wz_str = f"w_z={wz:+.2f}" if wz is not None else "w_z=…"
        specs.append(dict(
            XYZ=np.concatenate(cumP) if cumP else np.zeros((0, 3)),
            W=np.concatenate(cumW) if cumW else np.zeros(0),
            path=list(path), cam_pos=snap["cam_pos"], cam_look=snap["cam_look"],
            title=f"{cat}   |   MF-IG {wz_str}   (t={snap['step']*CFG.SIM.dt:.1f}s)", sim=snap.get("sim_rgb")))

    # write out one accumulation clip plus its still frames (faithful-MEM variant)
    def _emit(frames, stem, caption):
        frames = list(frames) + [frames[-1]] * VIDEO_MF.HOLD
        p = f"{VIDEO_MF.OUT}/{_slug(cat)}_{stem}.mp4"
        try:
            imageio.mimsave(p, frames, fps=VIDEO_MF.FPS, quality=8, macro_block_size=1)
            display(HTML(f"<b>{caption}</b>")); display(Video(p, embed=True, width=720))
            print(f"  saved -> {p}")
        except Exception as e:
            g = f"{VIDEO_MF.OUT}/{_slug(cat)}_{stem}.gif"
            imageio.mimsave(g, frames, duration=1.0/VIDEO_MF.FPS)
            from IPython.display import Image as IPImage
            display(HTML(f"<b>{caption}</b>")); display(IPImage(g))
            print(f"  mp4 failed ({e!r}); saved -> {g}")

    if VIDEO_MF.SEPARATE:                                   # one player per angle + the sim, separately
        for i, view in enumerate(VIDEO_MF.VIEWS):
            fr = [render_video_frame(sp["XYZ"], sp["W"], cubes, sp["path"], sp["cam_pos"], sp["cam_look"],
                                     K, sp["title"], vmax, views=[view], view_names=[f"view {i+1}"]) for sp in specs]
            _emit(fr, f"view{i+1}", f"MF saliency — view {i+1} (elev {view[0]}, azim {view[1]})")
        sim_fr = [np.asarray(sp["sim"]) for sp in specs if sp["sim"] is not None]
        if sim_fr:
            _emit(sim_fr, "sim", "Genesis sim (≈ view 1)")
    else:                                                  # combined multi-panel video (legacy)
        fr = [render_video_frame(sp["XYZ"], sp["W"], cubes, sp["path"], sp["cam_pos"], sp["cam_look"],
                                 K, sp["title"], vmax, sim_rgb=sp["sim"]) for sp in specs]
        _emit(fr, "combined", cat)

print("§14b ready: multi-frame video-IG accumulation. Run a per-condition §14b cell below (one trial each).")

In [ ]:
# §14b-tune  Denser MF-IG along the trajectory -> smoother accumulation. RUN AFTER §14b-setup, BEFORE §14b-1..4.
#   SAMPLE_EVERY_Q : 3 -> 1   run MF-IG on EVERY VLM query (≈3x more projected windows along the path)
#   IG_STEPS       : kept at 8 (bump to 16 for a finer Riemann sum / less per-frame noise)
#   VIDEO_EVERY    : 8 -> 4   snapshot the video twice as often — smoother playback, NOT more IG
VIDEO_MF.SAMPLE_EVERY_Q = 1
VIDEO_MF.IG_STEPS       = 8
VIDEO_MF.VIDEO_EVERY    = 4
print(f"MF-IG density bumped: SAMPLE_EVERY_Q={VIDEO_MF.SAMPLE_EVERY_Q}, IG_STEPS={VIDEO_MF.IG_STEPS}, "
      f"VIDEO_EVERY={VIDEO_MF.VIDEO_EVERY}. Each query now runs IG over {VIDEO_MF.WINDOW_FRAMES} frames "
      f"(≈{VIDEO_MF.WINDOW_FRAMES}x the per-query work of §14).")

In [ ]:
# §14b-1  MF Video: L=red,R=blue | friendly
run_saliency_video_mf("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[0])

In [ ]:
# §14b-2  MF Video: L=red,R=blue | hostile
run_saliency_video_mf("L=red,R=blue", RED, BLUE, CFG.AMBIGUOUS_PROMPTS[1])

In [ ]:
# §14b-3  MF Video: L=blue,R=red | friendly
run_saliency_video_mf("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[0])

In [ ]:
# §14b-4  MF Video: L=blue,R=red | hostile
run_saliency_video_mf("L=blue,R=red", BLUE, RED, CFG.AMBIGUOUS_PROMPTS[1])

## §15 — Cinematic **Blender Cycles** render: two trials × two looks, saliency as a **vibrant projected heat field**

Renders the **full trial** (not a still) of the **faithful MEM** driver as a path-traced video, for **both** ambiguous instructions — *"Go to the friendly one."* and *"Go to the hostile one."* — in the **same** arena (Red left, Blue right), so only the VLM's chosen target and its saliency change between the two simulations.

Each trial is rendered from **two cameras** (an **isometric** follow-cam + the robot **POV**) and stitched into a combined `iso|pov` clip, in **two material looks**:

- **without saliency** — the NB4 cinematic look: matte-black **tiled** floor, dark metallic agent, **glowing** coloured cubes.
- **with saliency** — a **frosted-glass** (semi-transparent, cloudy-white) agent on a matte-**white ceramic** floor, with both targets as **matte-black** cubes, overlaid with the saliency heat below.

**Saliency = one accumulating world-space heat field, projected onto every surface (both cameras).** Each attributed onboard frame's **chosen-$w_z$ IG** map is unprojected (depth + intrinsics) to the 3D points where it lands; **all** hits — floor *and* target faces — are splatted into a single world-space x-y field that **accumulates over the trial** (like the §14 projected-saliency figure). That one field is then projected **top-down onto the floor and onto the target cubes**, so the heat lands on the target surfaces exactly like the floor. It's drawn with a **vibrant turbo colormap** (blue → cyan → green → yellow → red) that **mixes over** the base material where heat is present — so it stays saturated and clearly visible on the white floor / matte-black cubes instead of washing out. Zero accumulation leaves the base untouched (clean floor, black cube — the "VLM blind spot"). Because the field *is* the unprojected hit points, the glow aligns with the floor/cube faces by construction. There are **no lasers**.

Tuning knobs live in `LOOK_SAL`: `heat_strength` (glow brightness), `heat_mask_gain` (how strongly the heat replaces the base — higher = more saturated / pops more), and `heat_gamma` (low-end lift); the heat-grid resolution is `CINE.HEAT_GRID`.

A printed **geometry self-test** (`[align …]`: floor-z ≈ 0, cube clusters at known x,y) and a **saliency check** (hot-centroid nearest the chosen target) verify alignment each run.

Run order: §15a (install Blender, once) → §15b (floor texture) → §15c (write render script) → §15d (capture utilities) → §15e (drive + render). Output: `cine_{friendly,hostile}_{plain,sal}_{iso,pov,combo}.mp4` (12 clips) under `CINE.OUT`. `CINE.QUICK=False` here (final quality: 1280×720 @ 200 spp, 30 fps); set `CINE.QUICK=True` for a fast 640×360 preview.


In [ ]:
# §15a  Download + extract standalone Blender 4.5 LTS (idempotent, its own Python). Browser UA (download.blender.org
# 403s the default urllib UA) and falls back across official mirrors. Verifies the file is the real
# ~375 MB tarball, not an HTML error page.
import os, glob, subprocess
BLENDER_ROOT='/content/blender'; os.makedirs(BLENDER_ROOT, exist_ok=True)
UA='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
# locate an already-extracted Blender binary, if one exists
def _find_bin():
    c=glob.glob(f'{BLENDER_ROOT}/blender-*/blender'); return c[0] if c else None

BLENDER_BIN=_find_bin()
if BLENDER_BIN is None:
    tar=f'{BLENDER_ROOT}/blender.tar.xz'
    bases=['https://download.blender.org/release',
           'https://ftp.nluug.nl/pub/graphics/blender/release',     # official mirror
           'https://mirrors.ocf.berkeley.edu/blender/release',      # official mirror
           'https://mirror.clarkson.edu/blender/release']           # official mirror
    versions=['4.5.4','4.5.3','4.5.2','4.5.1','4.5.0']
    def good(p):  # real tarball is hundreds of MB; reject error pages
        return os.path.exists(p) and os.path.getsize(p) > 100_000_000
    # try downloading via the wget CLI
    def wget(url):
        if os.path.exists(tar): os.remove(tar)
        r=subprocess.run(['wget','--user-agent',UA,'--tries=3','--timeout=90','-O',tar,url])
        return r.returncode==0 and good(tar)
    # fallback download path via urllib
    def urllib_dl(url):
        import urllib.request
        if os.path.exists(tar): os.remove(tar)
        try:
            req=urllib.request.Request(url, headers={'User-Agent':UA})
            with urllib.request.urlopen(req, timeout=90) as r, open(tar,'wb') as f:
                while True:
                    chunk=r.read(1<<20)
                    if not chunk: break
                    f.write(chunk)
            return good(tar)
        except Exception as e:
            print('   urllib failed:', e); return False
    done=False
    for ver in versions:
        major='.'.join(ver.split('.')[:2])
        for base in bases:
            url=f'{base}/Blender{major}/blender-{ver}-linux-x64.tar.xz'
            print('downloading', url)
            if wget(url) or urllib_dl(url):
                print('   OK (%.0f MB)' % (os.path.getsize(tar)/1e6)); done=True; break
            print('   failed')
        if done: break
    assert done, 'Could not download Blender from any mirror — check Colab network / try another version.'
    subprocess.run(['tar','-xf',tar,'-C',BLENDER_ROOT], check=True)
    BLENDER_BIN=_find_bin()

assert BLENDER_BIN and os.path.exists(BLENDER_BIN), 'Blender binary not found after extract.'
print('Blender binary:', BLENDER_BIN)
!{BLENDER_BIN} -b --version 2>/dev/null | head -1
# quick device probe (lists the CUDA/OPTIX devices Cycles can see)
probe='/content/_probe.py'
open(probe,'w').write(
 "import bpy\n"
 "p=bpy.context.preferences.addons['cycles'].preferences\n"
 "for dt in ('OPTIX','CUDA'):\n"
 "    try:\n"
 "        p.compute_device_type=dt; p.get_devices()\n"
 "        print('  ',dt,'->',[d.name for d in p.devices if d.type==dt])\n"
 "    except Exception as e: print('  ',dt,'err',e)\n")
!{BLENDER_BIN} -b -P {probe} 2>/dev/null | grep -E 'OPTIX|CUDA'

In [ ]:
# §15b  Full-floor grid texture: white thin lines on black, anti-aliased. Maps 1:1 to the floor plane.
import numpy as np, imageio.v2 as imageio
FLOOR_SIZE   = 40.0     # metres (plane edge); keep cubes (<~4 m out) well inside
GRID_CELL_M  = 1.0      # spacing between grid lines, metres
GRID_PX      = 4096     # texture resolution
LINE_PX      = 0.6      # on-texture line half-extent in pixels (extremely thin lines)

# anti-aliased white-on-black floor grid texture
def make_grid_png(path):
    img = np.zeros((GRID_PX, GRID_PX), np.float32)
    px_per_m = GRID_PX / FLOOR_SIZE
    coords = np.arange(GRID_PX, dtype=np.float32)
    # distance (px) from each column/row to the nearest grid line
    line_world = np.arange(0, FLOOR_SIZE + 1e-6, GRID_CELL_M)
    line_px = (line_world * px_per_m)
    # per-axis mask: 1 on a grid line, fading to 0 within LINE_PX
    def line_mask(axis_px):
        dmin = np.min(np.abs(axis_px[:,None] - line_px[None,:]), axis=1)
        return np.clip(1.0 - (dmin / LINE_PX), 0.0, 1.0)   # 1 on the line, fades within LINE_PX
    mx = line_mask(coords); my = line_mask(coords)
    grid = np.maximum(mx[None,:], my[:,None])               # vertical OR horizontal lines
    rgba = np.zeros((GRID_PX, GRID_PX, 4), np.float32); rgba[...,3]=1.0
    for c in range(3): rgba[...,c]=grid
    imageio.imwrite(path, (np.clip(rgba,0,1)*255).astype(np.uint8))
    return path

make_grid_png('/content/grid.png')
print('wrote /content/grid.png', f'({GRID_CELL_M} m cells, {FLOOR_SIZE} m floor)')

In [ ]:
%%writefile /content/render_scene_sal.py
# §15c  render_scene_sal.py — runs INSIDE standalone Blender 4.5 (its own Python 3.11). Path-traces each
# frame of a Go2 trial with Cycles (GPU/OptiX). Supports TWO material LOOKS, chosen by meta flags:
#   • "plain"  (no saliency): matte-black tiled floor, dark metallic agent, GLOWING coloured cubes
#                             (identical to the NB4 cinematic look). No saliency overlay.
#   • "sal"    (saliency):    matte-WHITE ceramic floor, FROSTED-GLASS (semi-transparent) agent and
#                             MATTE-BLACK target cubes, with ONE saliency overlay:
#       SURFACE HEAT (meta["show_surface_heat"], BOTH cameras): a single world-space accumulated heat
#       field (all surface hits — floor AND target faces) is projected TOP-DOWN onto the floor AND the
#       target cubes via a shared world-coordinate mapping, so the heat lands on the cube surfaces just
#       like the floor. Rendered as a VIBRANT turbo colormap (blue->red, like §14) that MIXES OVER the
#       base where heat is present (so it pops on the white floor / black cubes, not a washed-out add).
#       Zero accumulation = base shows through (clean floor, matte cube). There are NO lasers.
# The heat field IS the unprojected (depth+intrinsics) hit points, so the glow aligns with the floor /
# cube faces by construction (no volumetric domain box, no horizon slab).
# Bundle: meta.json, faces.npy, verts.npy(T,N,3), <cams>.npy(T,6), cubes.json, grid.png,
#         heat_floor.npy(T,G,G) [accumulated, normalised world-space x-y heat field].
# Invoked as: blender -b -P render_scene_sal.py -- <BUNDLE_DIR> <OUT_DIR> <CAMS.npy> <LENS_mm>
import bpy, sys, os, json
import numpy as np
import mathutils

# Blender CLI args that follow the '--' separator
def _argv_after_dd():
    return sys.argv[sys.argv.index("--") + 1:] if "--" in sys.argv else []

# select OPTIX/CUDA for Cycles rendering, falling back to CPU
def setup_gpu(scene, want="OPTIX"):
    prefs = bpy.context.preferences.addons["cycles"].preferences
    order = [want] + [d for d in ("OPTIX", "CUDA") if d != want]
    chosen = None
    for dt in order:
        try: prefs.compute_device_type = dt
        except Exception as e: print(f"[gpu] {dt} not assignable: {e}"); continue
        try: prefs.get_devices()
        except Exception as e: print(f"[gpu] get_devices {dt}: {e}"); continue
        gpu_devs = [d for d in prefs.devices if d.type == dt]
        if gpu_devs:
            for d in prefs.devices: d.use = (d.type == dt)
            chosen = dt; print(f"[gpu] using {dt}: " + ", ".join(d.name for d in gpu_devs)); break
        else: print(f"[gpu] no devices of type {dt}")
    scene.cycles.device = "GPU" if chosen else "CPU"
    if not chosen: print("[gpu] WARNING: CPU fallback (slow).")
    return chosen

# set the first matching input socket on a shader node
def _setp(inp, names, val):
    for nm in names:
        if nm in inp: inp[nm].default_value = val; return True
    return False

def add_heat_overlay(nt, base_surface, meta, heat_img, ref_obj=None):
    """Project the shared world-space accumulated heat image onto a surface as a VIVID turbo glow that
    MIXES OVER the base where heat is present — so it stands out on a bright white floor instead of being
    washed out by an additive emission. ref_obj=None -> use this object's own (world-aligned) coords;
    ref_obj=floor -> sample the SAME world field on the target cubes, so the heat lands on their surfaces
    exactly like the floor. Returns the new surface shader socket to feed the material output.
    Zero accumulation -> mix Fac 0 -> base shows through untouched (clean floor / matte cube)."""
    x0, x1, y0, y1 = [float(v) for v in meta.get("heat_extent", (-1.5, 5.0, -2.5, 2.5))]
    tc = nt.nodes.new("ShaderNodeTexCoord")
    if ref_obj is not None: tc.object = ref_obj
    mp = nt.nodes.new("ShaderNodeMapping")
    mp.inputs["Scale"].default_value    = (1.0/(x1-x0), 1.0/(y1-y0), 1.0)
    mp.inputs["Location"].default_value = (-x0/(x1-x0), -y0/(y1-y0), 0.0)
    ht = nt.nodes.new("ShaderNodeTexImage")
    ht.image = heat_img; ht.extension = "CLIP"; ht.interpolation = "Linear"     # CLIP -> no glow outside the active area
    bw = nt.nodes.new("ShaderNodeRGBToBW")
    pw = nt.nodes.new("ShaderNodeMath"); pw.operation = "POWER"
    pw.inputs[1].default_value = float(meta.get("heat_gamma", 0.7))
    ramp = nt.nodes.new("ShaderNodeValToRGB"); cr = ramp.color_ramp; cr.interpolation = "LINEAR"
    cr.elements[0].position = 0.0;  cr.elements[0].color = (0.02, 0.05, 0.55, 1.0)   # low -> vivid blue
    cr.elements[1].position = 1.0;  cr.elements[1].color = (0.90, 0.05, 0.03, 1.0)   # peak -> red
    for pos, col in ((0.22, (0.00, 0.55, 0.99)),    # cyan
                     (0.42, (0.10, 0.88, 0.28)),    # green
                     (0.62, (0.98, 0.93, 0.10)),    # yellow
                     (0.80, (1.00, 0.46, 0.03))):   # orange
        e = cr.elements.new(pos); e.color = (*col, 1.0)
    emit = nt.nodes.new("ShaderNodeEmission")
    emit.inputs["Strength"].default_value = float(meta.get("heat_strength", 5.0))
    mask = nt.nodes.new("ShaderNodeMath"); mask.operation = "MULTIPLY"; mask.use_clamp = True
    mask.inputs[1].default_value = float(meta.get("heat_mask_gain", 3.0))        # how fast heat replaces the base
    mix = nt.nodes.new("ShaderNodeMixShader")
    nt.links.new(tc.outputs["Object"], mp.inputs["Vector"])
    nt.links.new(mp.outputs["Vector"], ht.inputs["Vector"])
    nt.links.new(ht.outputs["Color"], bw.inputs["Color"])
    nt.links.new(bw.outputs["Val"], pw.inputs[0])
    nt.links.new(pw.outputs["Value"], ramp.inputs["Fac"])
    nt.links.new(pw.outputs["Value"], mask.inputs[0])
    nt.links.new(ramp.outputs["Color"], emit.inputs["Color"])
    nt.links.new(mask.outputs["Value"], mix.inputs["Fac"])
    nt.links.new(base_surface, mix.inputs[1]); nt.links.new(emit.outputs["Emission"], mix.inputs[2])
    return mix.outputs["Shader"]

def make_floor(meta, grid_png):
    """floor_style='tile'  -> dark glossy tiles, thin EMISSIVE grid lines (NB4 plain look).
       floor_style='ceramic'-> matte WHITE ceramic, thin darker-grey GROUT lines (saliency look)."""
    bpy.ops.mesh.primitive_plane_add(size=meta["floor_size"], location=(0, 0, 0))
    floor = bpy.context.active_object; floor.name = "floor"
    mat = bpy.data.materials.new("floor_mat"); mat.use_nodes = True; nt = mat.node_tree
    for n in list(nt.nodes): nt.nodes.remove(n)
    out = nt.nodes.new("ShaderNodeOutputMaterial"); mix = nt.nodes.new("ShaderNodeMixShader")
    tile = nt.nodes.new("ShaderNodeBsdfPrincipled")
    tex = nt.nodes.new("ShaderNodeTexImage")
    img = bpy.data.images.load(grid_png)
    try: img.colorspace_settings.name = "Non-Color"
    except Exception: pass
    tex.image = img; tex.extension = "EXTEND"; tex.interpolation = "Linear"
    tile.inputs["Base Color"].default_value = (*meta["floor_color"], 1.0)
    tile.inputs["Roughness"].default_value = meta["floor_roughness"]; tile.inputs["Metallic"].default_value = 0.0
    if meta.get("floor_style", "tile") == "ceramic":
        _setp(tile.inputs, ("Coat Weight",), float(meta.get("floor_coat", 0.18)))  # faint ceramic sheen
        _setp(tile.inputs, ("Coat Roughness",), 0.25)
        line = nt.nodes.new("ShaderNodeBsdfPrincipled")          # grout lines = matte grey BSDF (NOT emissive)
        line.inputs["Base Color"].default_value = (*meta.get("line_color", (0.55, 0.55, 0.58)), 1.0)
        line.inputs["Roughness"].default_value = 0.7; line.inputs["Metallic"].default_value = 0.0
        nt.links.new(line.outputs["BSDF"], mix.inputs[2])
    else:
        line = nt.nodes.new("ShaderNodeEmission")                # NB4 plain: thin glowing lines
        line.inputs["Color"].default_value = (*meta["line_color"], 1.0)
        line.inputs["Strength"].default_value = meta["line_glow"]
        nt.links.new(line.outputs["Emission"], mix.inputs[2])
    nt.links.new(tex.outputs["Color"], mix.inputs["Fac"])
    nt.links.new(tile.outputs["BSDF"], mix.inputs[1])
    surface = mix.outputs["Shader"]
    heat_img = None
    if meta.get("show_surface_heat", False):                     # accumulated saliency GLOW on the floor
        G = int(meta.get("heat_grid", 144))
        heat_img = bpy.data.images.new("heat_floor", width=G, height=G, alpha=False, float_buffer=True)
        try: heat_img.colorspace_settings.name = "Non-Color"
        except Exception: pass
        surface = add_heat_overlay(nt, surface, meta, heat_img, ref_obj=None)   # ref None -> floor's own (world) coords
    nt.links.new(surface, out.inputs["Surface"])
    floor.data.materials.append(mat); return floor, heat_img

# build the robot mesh from this frame's exported vertices
def make_robot(meta, verts0, faces):
    mesh = bpy.data.meshes.new("robot_mesh")
    mesh.from_pydata([tuple(v) for v in verts0.tolist()], [], [tuple(f) for f in faces.tolist()])
    mesh.validate(verbose=False); mesh.update()
    mesh.polygons.foreach_set("use_smooth", [True] * len(mesh.polygons)); mesh.update()
    obj = bpy.data.objects.new("robot", mesh); bpy.context.scene.collection.objects.link(obj)
    mat = bpy.data.materials.new("robot_mat"); mat.use_nodes = True
    p = mat.node_tree.nodes.get("Principled BSDF")
    if meta.get("robot_frosted", False):                        # FROSTED GLASS, semi-transparent cloudy-white
        p.inputs["Base Color"].default_value = (*meta.get("frost_color", (0.88, 0.91, 0.96)), 1.0)
        _setp(p.inputs, ("Transmission Weight", "Transmission"), float(meta.get("frost_transmission", 0.85)))
        p.inputs["Roughness"].default_value = float(meta.get("frost_roughness", 0.42))
        p.inputs["Metallic"].default_value = 0.0
        _setp(p.inputs, ("IOR",), float(meta.get("frost_ior", 1.45)))
        _setp(p.inputs, ("Subsurface Weight", "Subsurface"), float(meta.get("frost_subsurface", 0.12)))
        _setp(p.inputs, ("Subsurface Radius",), (0.6, 0.6, 0.7))
        _setp(p.inputs, ("Subsurface Scale",), 0.05)
        if "Subsurface Color" in p.inputs:
            p.inputs["Subsurface Color"].default_value = (0.95, 0.96, 1.0, 1.0)
        _setp(p.inputs, ("Coat Weight",), float(meta.get("frost_coat", 0.15)))
    else:                                                        # NB4 plain: dark matte metallic
        p.inputs["Base Color"].default_value = (*meta["robot_color"], 1.0)
        p.inputs["Metallic"].default_value = meta["robot_metallic"]
        p.inputs["Roughness"].default_value = meta["robot_roughness"]
        if "Coat Weight" in p.inputs and meta.get("robot_coat", 0.0):
            p.inputs["Coat Weight"].default_value = meta["robot_coat"]
    obj.data.materials.append(mat); return obj

# build the two goal-cube meshes with their material look + heat overlay
def make_cubes(meta, cubes, floor=None, heat_img=None):
    style = meta.get("cube_style", "glow"); show_heat = meta.get("show_surface_heat", False)
    for i, c in enumerate(cubes):
        size = float(c["size"]); x, y = float(c["pos"][0]), float(c["pos"][1])
        bpy.ops.mesh.primitive_cube_add(size=size, location=(x, y, size / 2.0))
        cube = bpy.context.active_object; cube.name = f"cube_{i}"; col = tuple(c["color"])
        mat = bpy.data.materials.new(f"cube_mat_{i}"); mat.use_nodes = True
        nt = mat.node_tree; p = nt.nodes.get("Principled BSDF"); out = nt.nodes.get("Material Output")
        if style == "matte":                                     # saliency look: MATTE-BLACK target (heat pops on it)
            p.inputs["Base Color"].default_value = (*meta.get("cube_color", (0.02, 0.02, 0.025)), 1.0)
            p.inputs["Roughness"].default_value = float(meta.get("cube_roughness", 0.6))
            p.inputs["Metallic"].default_value = 0.0
            if show_heat and heat_img is not None:               # SAME world heat field, projected onto the cube faces
                surf = add_heat_overlay(nt, p.outputs["BSDF"], meta, heat_img, ref_obj=floor)
                nt.links.new(surf, out.inputs["Surface"])
        elif style == "frosted":                                 # (legacy) colourless frosted glass
            p.inputs["Base Color"].default_value = (*meta.get("cube_frost_color", (0.90, 0.92, 0.96)), 1.0)
            _setp(p.inputs, ("Transmission Weight", "Transmission"), float(meta.get("cube_frost_transmission", 0.88)))
            p.inputs["Roughness"].default_value = float(meta.get("cube_frost_roughness", 0.38))
            p.inputs["Metallic"].default_value = 0.0
            _setp(p.inputs, ("IOR",), float(meta.get("cube_frost_ior", 1.45)))
            _setp(p.inputs, ("Coat Weight",), 0.1)
        else:                                                    # NB4 plain: glowing coloured cube
            p.inputs["Base Color"].default_value = (*col, 1.0)
            p.inputs["Roughness"].default_value = meta.get("cube_roughness", 0.4)
            if "Emission Color" in p.inputs: p.inputs["Emission Color"].default_value = (*col, 1.0)
            if "Emission Strength" in p.inputs:
                p.inputs["Emission Strength"].default_value = float(c.get("glow", meta.get("cube_glow", 8.0)))
        cube.data.materials.append(mat)
        cube.data.polygons.foreach_set("use_smooth", [False] * len(cube.data.polygons)); cube.data.update()

# add the scene's directional light
def make_light(meta):
    la = bpy.data.lights.new("key", "AREA"); la.energy = meta["light_energy"]; la.size = meta["light_size"]
    la.color = tuple(meta.get("light_color", (1.0, 1.0, 1.0)))
    lo = bpy.data.objects.new("key", la); bpy.context.scene.collection.objects.link(lo)
    lo.location = tuple(meta["light_pos"])
    d = mathutils.Vector(tuple(meta["light_target"])) - mathutils.Vector(tuple(meta["light_pos"]))
    lo.rotation_euler = d.to_track_quat("-Z", "Y").to_euler(); return lo

# set the world background colour/strength
def make_world(scene, meta):
    world = bpy.data.worlds.new("world"); scene.world = world; world.use_nodes = True
    bg = world.node_tree.nodes.get("Background")
    bg.inputs[0].default_value = (*meta["world_color"], 1.0); bg.inputs[1].default_value = meta["world_strength"]

# add and configure the render camera
def make_camera(scene, meta, lens=None):
    cam_data = bpy.data.cameras.new("cam"); cam_data.sensor_fit = "HORIZONTAL"; cam_data.sensor_width = 36.0
    cam_data.lens = float(lens if lens is not None else meta["cam_lens"])
    cam_obj = bpy.data.objects.new("cam", cam_data); scene.collection.objects.link(cam_obj); scene.camera = cam_obj
    return cam_obj

# point the camera at a world-space look-at point
def aim_camera(cam_obj, pos, look):
    cam_obj.location = tuple(float(x) for x in pos)
    d = mathutils.Vector(tuple(float(x) for x in look)) - mathutils.Vector(tuple(float(x) for x in pos))
    cam_obj.rotation_euler = d.to_track_quat("-Z", "Y").to_euler()

def update_floor_heat(heat_img, grid):
    """Write the accumulated floor heat grid (G,G in [0,1]) into the emission texture. Grid is stored
    [iy,ix] with iy=0 at the bottom (v=0), matching Blender's pixel order (bottom row first)."""
    if heat_img is None: return
    g = np.asarray(grid, np.float32).ravel()                        # iy-major, ix-minor
    buf = np.empty((g.shape[0], 4), np.float32)
    buf[:, 0] = g; buf[:, 1] = g; buf[:, 2] = g; buf[:, 3] = 1.0
    heat_img.pixels.foreach_set(buf.reshape(-1)); heat_img.update()

# parse CLI args, build the scene once, then render every frame
def main():
    args = _argv_after_dd()
    if len(args) < 2:
        print("usage: blender -b -P render_scene_sal.py -- <BUNDLE> <OUT> [CAMS.npy] [LENS]"); sys.exit(1)
    BUNDLE, OUT = args[0], args[1]
    cams_path = args[2] if len(args) > 2 else os.path.join(BUNDLE, "cams.npy")
    if not os.path.isabs(cams_path): cams_path = os.path.join(BUNDLE, cams_path)
    lens_override = float(args[3]) if len(args) > 3 else None
    os.makedirs(OUT, exist_ok=True)

    meta = json.load(open(os.path.join(BUNDLE, "meta.json")))
    faces = np.load(os.path.join(BUNDLE, "faces.npy")); verts = np.load(os.path.join(BUNDLE, "verts.npy"))
    cams = np.load(cams_path); cubes = json.load(open(os.path.join(BUNDLE, "cubes.json")))
    grid_png = os.path.join(BUNDLE, "grid.png")

    show_heat = bool(meta.get("show_surface_heat", False))
    heat_floor = None
    if show_heat:
        hf = os.path.join(BUNDLE, "heat_floor.npy")
        heat_floor = np.load(hf).astype(np.float32) if os.path.exists(hf) else None      # (T,G,G)

    T = min(verts.shape[0], cams.shape[0])
    if heat_floor is not None: T = min(T, heat_floor.shape[0])
    print(f"[render] frames={T} verts/frame={verts.shape[1]} faces={faces.shape[0]} "
          f"variant={meta.get('variant','?')} surface_heat={'yes' if show_heat else 'no'}")

    bpy.ops.wm.read_factory_settings(use_empty=True)
    scene = bpy.context.scene
    scene.render.engine = "CYCLES"; scene.cycles.samples = meta["spp"]
    setup_gpu(scene, want=meta.get("device", "OPTIX"))
    scene.cycles.use_denoising = bool(meta.get("use_denoising", True))
    try: scene.cycles.denoiser = meta.get("denoiser", "OPENIMAGEDENOISE")
    except Exception as e: print("[render] denoiser failed:", e); scene.cycles.use_denoising = False
    try: scene.cycles.caustics_refractive = True
    except Exception: pass
    if meta.get("robot_frosted") or meta.get("cube_style") == "frosted":   # glass needs more bounces
        for attr, val in (("max_bounces", 16), ("transmission_bounces", 12), ("transparent_max_bounces", 12)):
            try: setattr(scene.cycles, attr, max(getattr(scene.cycles, attr), val))
            except Exception: pass
    scene.render.resolution_x = meta["res"][0]; scene.render.resolution_y = meta["res"][1]
    scene.render.resolution_percentage = 100
    scene.render.image_settings.file_format = "PNG"; scene.render.film_transparent = False
    try: scene.view_settings.view_transform = meta.get("view_transform", "AgX")
    except Exception as e: print("[render] view_transform failed:", e)

    make_world(scene, meta)
    floor, heat_img = make_floor(meta, grid_png)
    robot = make_robot(meta, verts[0], faces)
    make_cubes(meta, cubes, floor=floor, heat_img=heat_img)        # cubes sample the SAME live heat image
    make_light(meta)
    cam_obj = make_camera(scene, meta, lens=lens_override)

    for i in range(T):
        robot.data.vertices.foreach_set("co", verts[i].reshape(-1).astype(np.float32)); robot.data.update()
        if show_heat and heat_floor is not None:                   # one image update drives floor + cubes
            update_floor_heat(heat_img, heat_floor[i])
        aim_camera(cam_obj, cams[i, :3], cams[i, 3:])
        scene.render.filepath = os.path.join(OUT, f"frame_{i:05d}.png")
        bpy.ops.render.render(write_still=True)
        if i % 10 == 0: print(f"[render] frame {i+1}/{T}", flush=True)
    print(f"[render] DONE -> {OUT}")

if __name__ == "__main__":
    main()


In [ ]:
# §15d  Cinematic CAPTURE: faithful MEM driver (planner_v2) acts; single-onboard-frame chosen-w_z IG
#        is projected into 3D using DEPTH + camera intrinsics (no volumetric cloud). The projection feeds
#        a single saliency representation, the SURFACE HEAT (both cameras): every surface hit (floor AND
#        target faces) is splatted into ONE world-space x-y heat field that accumulates over the trial.
#        That same field is later projected top-down onto the floor AND onto the (matte-black) target
#        cubes, so the targets show the heat on their surfaces exactly like the floor — a vibrant turbo
#        glow that builds up over time, like the §14 projected-saliency examples. The heat splats ARE the
#        unprojected hit points, so the glow aligns with the floor/cubes by construction. Also captures
#        robot mesh geometry + iso/pov camera tracks, and runs a geometry self-test. (No lasers.)
import os, gc, json, random, numpy as np, torch
from types import SimpleNamespace
from PIL import Image as PILImage

# Fresh random seed each run (logged so a good-looking trial is reproducible). The ceiling is 2**31-1,
# not 2**32, because the planner is later called with seed=SEED+t per timestep — this leaves headroom so
# those per-step seeds never overflow the 32-bit range some samplers expect. Hard-code SEED back to a
# logged value to replay a specific run.
SEED = random.randrange(1, 2**31 - 1)
print(f"[cine] seed = {SEED}")

CINE = SimpleNamespace(
    QUICK       = False,            # True: fast preview (lower res/spp). False: final quality.
    HEAT_GRID   = 144,            # world-space floor heat-grid resolution (per axis)
    IG_STEPS    = 8,               # IG Riemann steps per attributed query
    IG_EVERY_Q  = 2,               # attribute every k-th VLM query along the trajectory
    MAX_SECONDS = 6.0,
    SEED        = SEED,            # randomized above; varies trajectory + which hot points get sampled
    CUBE_SIZE   = 0.45,            # render-only cube edge (sim cubes keep CFG.ARENA.box_size)
    SAL_MAX_DEPTH = 8.0,           # drop pixels with no surface within this range (would shoot to infinity)
    WORK        = "/content",      # Blender + bundle scratch (Colab)
    OUT         = f"{CFG.SALIENCY_DIR}/cinematic")
os.makedirs(CINE.OUT, exist_ok=True)

ISO_OFFSET  = np.array([3.0, 3.0, 2.4])   # isometric follow-cam offset from the robot base (world)
ISO_LOOK_DZ = 0.30

# ---- robot geometry math (Genesis quat w,x,y,z -> world verts) ----
def _to_np(x):
    return np.asarray(x.detach().cpu().numpy() if hasattr(x, "detach")
                      else (x.cpu().numpy() if hasattr(x, "cpu") else x), dtype=float)

# quaternion -> 3x3 rotation matrix
def _quat2mat(q):
    w, x, y, z = q; n = np.sqrt(w*w + x*x + y*y + z*z)
    if n == 0: return np.eye(3)
    w, x, y, z = w/n, x/n, y/n, z/n
    return np.array([[1-2*(y*y+z*z), 2*(x*y-z*w),   2*(x*z+y*w)],
                     [2*(x*y+z*w),   1-2*(x*x+z*z), 2*(y*z-x*w)],
                     [2*(x*z-y*w),   2*(y*z+x*w),   1-2*(x*x+y*y)]])

# apply a (rotation, translation) pose to a batch of points
def _apply_T(p, q, V):
    return (_quat2mat(q) @ np.asarray(V, float).T).T + np.asarray(p, float).reshape(3)

# capture the robot's local-frame vertices/faces once
def build_robot_template(robot):
    vgeoms = list(getattr(robot, "vgeoms", []))
    if not vgeoms:
        vgeoms = [vg for lk in robot.links for vg in getattr(lk, "vgeoms", [])]
    if not vgeoms:
        raise RuntimeError("No visual geoms on robot — cannot reconstruct mesh.")
    template, faces, off = [], [], 0
    for vg in vgeoms:
        tm = vg.get_trimesh()
        V = np.asarray(tm.vertices, np.float32); F = np.asarray(tm.faces, np.int64) + off
        link = getattr(vg, "link", None) or (vg.get_link() if hasattr(vg, "get_link") else None)
        ip = _to_np(getattr(vg, "init_pos")).reshape(3); iq = _to_np(getattr(vg, "init_quat")).reshape(4)
        template.append(dict(link=link, V=V, ip=ip, iq=iq, sl=slice(off, off+len(V))))
        faces.append(F); off += len(V)
    return template, np.concatenate(faces, 0).astype(np.int32), off

# world-space robot vertices for every recorded frame
def extract_world_verts(template, Ntot):
    V = np.empty((Ntot, 3), np.float32); cache = {}
    for it in template:
        k = id(it["link"])
        if k not in cache:
            cache[k] = (_to_np(it["link"].get_pos()).reshape(-1)[:3], _to_np(it["link"].get_quat()).reshape(-1)[:4])
        lp, lq = cache[k]
        V[it["sl"]] = _apply_T(lp, lq, _apply_T(it["ip"], it["iq"], it["V"])).astype(np.float32)
    return V

# ---- SURFACE-ONLY saliency projection (same verified geometry as §13, NO horizon/back-wall) ----
def project_saliency_surface(sal, depth, cam_pos, lookat, K, max_depth=None):
    """Unproject ONLY finite-depth (surface) pixels to world XYZ, weighted by saliency. Identical
    camera math to the §13 projection that visibly aligns with the cubes, but sky/horizon pixels are
    DROPPED (not painted onto a back wall): saliency toward the horizon never lands on a surface.
    Returns (XYZ Nx3, w N)."""
    max_depth = CINE.SAL_MAX_DEPTH if max_depth is None else max_depth
    d = np.asarray(depth, float)
    if d.ndim == 3: d = d[..., 0]
    H, W = d.shape[:2]
    sal = np.asarray(sal, float)
    if sal.ndim == 3: sal = sal[..., 0]
    if sal.shape[:2] != (H, W):
        sal = np.asarray(PILImage.fromarray((np.clip(sal,0,1)*255).astype("uint8")).resize((W,H)))/255.0
    _, T_wc = U.look_at_extrinsics(cam_pos, lookat, up=(0, 0, 1))
    R = T_wc[:3, :3]; cam = T_wc[:3, 3]
    vv, uu = np.mgrid[0:H, 0:W].astype(float)
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    w = sal.reshape(-1); dd = d.reshape(-1)
    with np.errstate(invalid="ignore", divide="ignore"):
        x = (uu-cx)/fx*d; y = (vv-cy)/fy*d; z = d
        world = (R @ np.stack([x, y, z], -1).reshape(-1, 3).T).T + cam
    surf = np.isfinite(dd) & (dd > 1e-3) & (dd < max_depth)
    return world[surf].astype(np.float32), w[surf].astype(np.float32)

# goal-cube geometry, shared with the §13/§14 projection helpers
def _cube_geo(left, right):
    out = []
    for o in (left, right):
        cx, cy = float(o["pos"][0]), float(o["pos"][1])
        sx, sy, sz = (float(o["size"][0]), float(o["size"][1]), float(o["size"][2])) \
            if "size" in o and np.ndim(o["size"]) else (CFG.ARENA.box_size[0], CFG.ARENA.box_size[1], CFG.ARENA.box_size[2])
        out.append((color_name(o["color"]).capitalize(), (cx, cy, sx/2, sy/2, sz)))
    return out

def verify_projection_alignment(depth, cam_pos, lookat, K, cube_geo, tag=""):
    """GEOMETRY self-test (saliency-independent): unproject EVERY surface pixel with weight 1 and check
    the reconstructed cloud matches the known scene — floor pixels at z≈0, cube pixels clustered at the
    cubes' known (x,y). This is the direct proof that the heat splats align with the 3D environment."""
    P, _ = project_saliency_surface(np.ones(np.asarray(depth).shape[:2], np.float32), depth, cam_pos, lookat, K)
    if len(P) == 0:
        print(f"  [align{tag}] no surface pixels in view"); return
    floor = P[P[:, 2] < 0.06]
    z_rms = float(np.sqrt(np.mean(floor[:, 2]**2))) if len(floor) else float("nan")
    parts = [f"floor-z RMS={z_rms:.3f}m (want ~0)"]
    for nm, (cx, cy, hx, hy, sz) in cube_geo:
        on = P[(np.abs(P[:,0]-cx)<hx+0.12)&(np.abs(P[:,1]-cy)<hy+0.12)&(P[:,2]>0.05)&(P[:,2]<sz+0.12)]
        parts.append(f"{nm}: {len(on)}px Δxy=({np.median(on[:,0])-cx:+.2f},{np.median(on[:,1])-cy:+.2f})m"
                     if len(on) else f"{nm}: not in view")
    print(f"  [align{tag}] " + " | ".join(parts))

# ---- ACCUMULATING surface heat (both cameras): ONE world-space heat field, projected onto floor + targets ----
def build_surface_heat(sal_keys, n_render, grid=144, margin=0.6, sig=2.2):
    """Splat ALL projected saliency hits (floor AND target faces) into a single world-space x-y heat grid
    and accumulate over the trial (monotone, like the §14 figure). The same field is later projected
    top-down onto the floor and onto the target cubes, so the targets show the heat on their surfaces
    exactly like the floor. Normalised by the end-state max so the glow builds up from cold to full.
    Returns heat_floor(T,G,G)≤1 float16, extent=(x0,x1,y0,y1)."""
    pts = []
    for (f, P, W) in sal_keys:
        P = np.asarray(P, float).reshape(-1, 3); W = np.clip(np.asarray(W, float).reshape(-1), 0, None)
        if len(P): pts.append((P, W))
    if pts:
        allP = np.concatenate([p for p, _ in pts])
        x0, x1 = allP[:, 0].min() - margin, allP[:, 0].max() + margin
        y0, y1 = allP[:, 1].min() - margin, allP[:, 1].max() + margin
    else:
        x0, x1, y0, y1 = -1.5, 5.0, -2.5, 2.5
    if x1 - x0 < 1e-3: x1 = x0 + 1.0
    if y1 - y0 < 1e-3: y1 = y0 + 1.0
    extent = (float(x0), float(x1), float(y0), float(y1)); G = int(grid)
    r = int(max(1, round(sig * 2))); ay, ax = np.mgrid[-r:r+1, -r:r+1]
    stamp = np.exp(-(ax**2 + ay**2) / (2 * sig**2)).astype(np.float32)         # soft Gaussian splat
    incs, key_f = [], []
    for (f, P, W) in sal_keys:
        P = np.asarray(P, float).reshape(-1, 3); W = np.clip(np.asarray(W, float).reshape(-1), 0, None)
        g = np.zeros((G, G), np.float32)
        if len(P):                                                            # ALL hits (floor + cube faces) -> grid
            fx = (P[:, 0] - x0) / (x1 - x0); fy = (P[:, 1] - y0) / (y1 - y0)
            ix = np.clip((fx * (G-1)).astype(int), 0, G-1); iy = np.clip((fy * (G-1)).astype(int), 0, G-1)
            for dy in range(-r, r+1):
                yy = np.clip(iy + dy, 0, G-1)
                for dx in range(-r, r+1):
                    xx = np.clip(ix + dx, 0, G-1); np.add.at(g, (yy, xx), W * stamp[dy+r, dx+r])
        incs.append(g); key_f.append(int(f))
    key_f = np.asarray(key_f)
    heat = np.zeros((n_render, G, G), np.float32)
    acc = np.zeros((G, G), np.float32); ki = 0
    for i in range(n_render):                                                  # hold accumulation between keyframes
        while ki < len(key_f) and key_f[ki] <= i:
            acc = acc + incs[ki]; ki += 1
        heat[i] = acc
    hm = float(heat[-1].max()) if n_render and heat[-1].max() > 0 else 1.0
    return (heat / hm).astype(np.float16), extent

def saliency_alignment_report(sal_keys, cube_geo, chosen_xy):
    """Confirm the accumulated saliency hot-centroid sits nearer the CHOSEN target than the other."""
    if not sal_keys: print("  [sal-align] no saliency keyframes"); return
    P = np.concatenate([np.asarray(k[1], float).reshape(-1, 3) for k in sal_keys])
    W = np.concatenate([np.clip(np.asarray(k[2], float).reshape(-1), 0, None) for k in sal_keys])
    hot = W >= np.quantile(W, 0.85)
    cen = np.average(P[hot, :2], axis=0, weights=W[hot]) if hot.any() else P[:, :2].mean(0)
    ds = {nm: float(np.linalg.norm(cen-np.array(c[:2]))) for nm, c in cube_geo}
    near = min(ds, key=ds.get)
    chosen = min(cube_geo, key=lambda nc: np.linalg.norm(np.array(nc[1][:2])-np.array(chosen_xy)))[0]
    print(f"  [sal-align] hot-centroid=({cen[0]:.2f},{cen[1]:.2f}) nearest={near} chosen={chosen} "
          f"-> {'PASS' if near==chosen else 'CHECK'} | dists={ {k:round(v,2) for k,v in ds.items()} }")

def record_cinematic_trial(env, policy, left, right, instruction, seed):
    """Drive ONE trial with the faithful MEM planner; capture geometry, iso/pov cams, and the
    single-frame chosen-w_z saliency -> accumulating surface-heat grid + per-target glow (aligned)."""
    robot = env.robot; template, faces, Ntot = build_robot_template(robot)
    cube_geo = _cube_geo(left, right)
    query_every = max(1, int(round(50.0 / CFG.VLM_RATE_HZ)))
    stride = max(1, int(round(50.0 / (18 if CINE.QUICK else 30))))
    max_steps = int(CINE.MAX_SECONDS / env.dt); K = onboard_K()
    planner_v2.reset(); obs, _ = env.reset(); cmd = np.zeros(3, float)
    p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
    rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
    verts_seq, cams_iso, cams_pov, sal_keys, qn, did_selftest = [], [], [], [], 0, False
    with torch.no_grad():
        for t in range(max_steps):
            now = t * env.dt
            frame = PILImage.fromarray(last_rgb[..., :3].astype("uint8")).convert("RGB")
            if t % query_every == 0:
                r = planner_v2.act(frame, instruction, sim_time=now, temperature=CFG.VLM_TEMPERATURE, seed=seed + t)
                cmd = CFG.LOWPASS_ALPHA*np.array([r["v_x"], r["v_y"], r["w_z"]], float) + (1-CFG.LOWPASS_ALPHA)*cmd
                if depth is not None and qn % CINE.IG_EVERY_Q == 0:
                    try:
                        cp, cl = onboard_pose(env)
                        if not did_selftest:                       # one-time geometry alignment proof
                            verify_projection_alignment(depth, cp, cl, K, cube_geo, tag=" t0"); did_selftest = True
                        maps, _ = integrated_gradients_chosen([frame], r.get("gen_ids") or [], None, False,
                                    instruction=instruction, component="w_z", steps=CINE.IG_STEPS,
                                    out_size=tuple(CFG.CAM.res))
                        P, Wt = project_saliency_surface(maps[0], depth, cp, cl, K)
                        if len(P): sal_keys.append((len(verts_seq), P, Wt))
                    except Exception as e:
                        print("   [cine-ig] skip:", repr(e))
                qn += 1
            cmd[0] = np.clip(cmd[0], *CFG.CMD.lin_vel_x_range); cmd[1] = np.clip(cmd[1], *CFG.CMD.lin_vel_y_range)
            cmd[2] = np.clip(cmd[2], *CFG.CMD.ang_vel_range)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            a = policy(obs); obs, _, _, _ = env.step(a)
            env.commands[:, 0], env.commands[:, 1], env.commands[:, 2] = map(float, cmd)
            p, l = onboard_pose(env); set_level_pose(env.camera, p, l)
            rgb, depth = U.render_camera(env.camera, want_depth=True); last_rgb = rgb
            base = np.asarray(robot.get_pos().cpu()).reshape(-1, 3)[0]
            if t % stride == 0:
                verts_seq.append(extract_world_verts(template, Ntot))
                cams_iso.append(np.concatenate([base + ISO_OFFSET, base + np.array([0, 0, ISO_LOOK_DZ])]).astype(np.float32))
                pp, pl = onboard_pose(env)
                cams_pov.append(np.concatenate([np.asarray(pp, float), np.asarray(pl, float)]).astype(np.float32))
            if any(np.linalg.norm(base[:2]-np.array(o["pos"][:2])) < CFG.ARENA.commit_radius for o in (left, right)):
                break
    n_render = len(verts_seq)
    base_xy = np.asarray(robot.get_pos().cpu()).reshape(-1, 3)[0][:2]
    heat_floor, heat_extent = build_surface_heat(sal_keys, n_render, grid=CINE.HEAT_GRID)
    print(f"  captured {n_render} render frames, {len(sal_keys)} IG keyframes, "
          f"heat {heat_floor.shape} extent={[round(e,2) for e in heat_extent]}")
    saliency_alignment_report(sal_keys, cube_geo, chosen_xy=base_xy)
    return dict(verts=np.asarray(verts_seq, np.float32), cams_iso=np.asarray(cams_iso, np.float32),
                cams_pov=np.asarray(cams_pov, np.float32), faces=faces, render_fps=(18 if CINE.QUICK else 30),
                heat_floor=heat_floor, heat_extent=heat_extent, n_keys=len(sal_keys), cube_geo=cube_geo)

print("Cinematic capture ready (faithful driver + accumulating vibrant surface-heat glow; no lasers).")


In [ ]:
# §15e  Cinematic RUN: for BOTH ambiguous instructions ("friendly" / "hostile") drive ONE faithful-MEM
#        trial, capture geometry + the accumulating world-space surface-heat field ONCE, then path-trace it
#        in Blender Cycles from two cameras (isometric + onboard POV) in TWO material LOOKS:
#          • plain  = NB4 cinematic (matte-black tiles, dark metallic agent, glowing cubes), NO saliency
#          • sal    = frosted-glass agent + white ceramic floor + MATTE-BLACK targets, with saliency shown
#                     as a VIBRANT accumulating heat field (turbo blue->red, like §14) projected onto BOTH
#                     the floor AND the target surfaces, in BOTH iso and pov. (No lasers.)
#        Each look is rendered iso + pov, then stitched into a combined iso|pov video.
#        Outputs:  cine_{sim}_{variant}_{iso,pov,combo}.mp4   under CINE.OUT   (2×2×3 = 12 clips).
import os, gc, json, glob, subprocess, shutil, numpy as np
import imageio.v2 as imageio
from IPython.display import Video, display

if CINE.QUICK:
    RES, SPP = (640, 360), 64
else:
    RES, SPP = (1280, 720), 200
RED_RGB  = [1.00, 0.04, 0.04]
BLUE_RGB = [0.05, 0.12, 1.00]
# NOTE: this dict shadows §13a's global VIEWS tuple — re-run §13a/§14a before returning there.
VIEWS = {"iso": dict(lens=34.0), "pov": dict(lens=20.0)}

# --- shared render settings ---
LOOK_COMMON = dict(
    res=list(RES), spp=SPP, device="OPTIX", view_transform="AgX",
    denoiser="OPENIMAGEDENOISE", use_denoising=True,
    floor_size=40.0, cam_lens=34.0,
    light_size=6.0, light_pos=[4.0, 6.0, 7.0], light_target=[0.0, 0.0, 0.3])

# --- WITHOUT saliency: identical to the NB4 cinematic look ---
LOOK_PLAIN = dict(LOOK_COMMON,
    floor_style="tile",
    floor_color=[0.030, 0.030, 0.036], floor_roughness=0.22,
    line_color=[0.45, 0.47, 0.55], line_glow=0.4,
    robot_frosted=False,
    robot_color=[0.016, 0.016, 0.020], robot_metallic=0.55, robot_roughness=0.38, robot_coat=0.15,
    cube_style="glow", cube_roughness=0.5, cube_glow=18.0,
    world_color=[0.045, 0.05, 0.06], world_strength=0.35,
    light_energy=2200.0, light_color=[1.0, 0.98, 0.95])

# --- WITH saliency: frosted-glass agent + matte-white ceramic floor + MATTE-BLACK target cubes ---
#     Saliency = ONE accumulating world-space heat field (turbo blue->red, like §14), projected onto BOTH
#     the floor and the (matte-black) targets and MIXED OVER the base so it pops. show_surface_heat is
#     toggled per variant in the loop below.
LOOK_SAL = dict(LOOK_COMMON,
    floor_style="ceramic",
    floor_color=[0.82, 0.82, 0.84], floor_roughness=0.55, floor_coat=0.18,
    line_color=[0.55, 0.55, 0.58], line_glow=0.0,                       # matte grout lines (non-emissive)
    robot_frosted=True, frost_color=[0.88, 0.91, 0.96], frost_roughness=0.42,
    frost_transmission=0.85, frost_ior=1.45, frost_subsurface=0.12, frost_coat=0.15,
    cube_style="matte", cube_color=[0.02, 0.02, 0.025], cube_roughness=0.6,   # MATTE-BLACK targets
    world_color=[0.050, 0.055, 0.065], world_strength=0.60,             # brighter so the white room reads
    light_energy=2600.0, light_color=[1.0, 0.99, 0.97],
    # --- accumulating SURFACE HEAT (floor + targets): VIBRANT turbo, MIXED OVER the base so it stands out ---
    heat_strength=5.0,          # emissive glow strength of the heat
    heat_gamma=0.7,             # low-end lift (more of the field shows colour)
    heat_mask_gain=3.0)         # how fast heat replaces the bright base (higher = more saturated, pops more)

def save_geo(bundle, res, cubes):
    """Write the view-independent geometry ONCE per sim: robot faces/verts, iso & pov camera tracks,
    the accumulating world-space surface-heat field, the cube placements and the floor grid. meta.json
    is written per (variant, view)."""
    if os.path.isdir(bundle): shutil.rmtree(bundle)
    os.makedirs(bundle)
    for k in ("faces", "verts", "cams_iso", "cams_pov", "heat_floor"):
        np.save(f"{bundle}/{k}.npy", res[k])
    json.dump(cubes, open(f"{bundle}/cubes.json", "w"))
    shutil.copy(f"{CINE.WORK}/grid.png", f"{bundle}/grid.png")

# write the per-render meta.json (material look + resolution + extras)
def write_meta(bundle, look, fps, extra=None):
    meta = dict(look); meta["res"] = list(RES); meta["spp"] = SPP; meta["fps"] = fps
    if extra: meta.update(extra)
    json.dump(meta, open(f"{bundle}/meta.json", "w"))
    return meta

# invoke standalone Blender on render_scene_sal.py and stream its log
def run_blender(bundle, out, cams_file, lens):
    if os.path.isdir(out): shutil.rmtree(out)
    os.makedirs(out)
    cmd = [BLENDER_BIN, "-b", "-P", f"{CINE.WORK}/render_scene_sal.py", "--", bundle, out, cams_file, str(lens)]
    print("  >", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if any(k in line for k in ("[render] frame", "[render] DONE", "[gpu]", "[align", "Error", "Failed", "Traceback")):
            print("   ", line.rstrip())
    proc.wait(); print("  blender exit", proc.returncode); return proc.returncode

# stitch rendered PNG frames into an mp4
def frames_to_mp4(out_dir, mp4, fps):
    files = sorted(glob.glob(f"{out_dir}/frame_*.png"))
    assert files, f"no frames in {out_dir}"
    with imageio.get_writer(mp4, fps=fps, macro_block_size=None, quality=8) as w:
        for f in files: w.append_data(imageio.imread(f)[..., :3])
    return len(files)

# stitch two frame sequences into one side-by-side mp4
def side_by_side(dir_a, dir_b, mp4, fps):
    fa = sorted(glob.glob(f"{dir_a}/frame_*.png")); fb = sorted(glob.glob(f"{dir_b}/frame_*.png"))
    n = min(len(fa), len(fb))
    with imageio.get_writer(mp4, fps=fps, macro_block_size=None, quality=8) as w:
        for i in range(n):
            a = imageio.imread(fa[i])[..., :3]; b = imageio.imread(fb[i])[..., :3]
            if a.shape[0] != b.shape[0]:
                hh = min(a.shape[0], b.shape[0]); a, b = a[:hh], b[:hh]
            w.append_data(np.concatenate([a, b], axis=1))     # iso | pov
    return n

# ----------------------------------------------------------------------------------
# Two simulations share the SAME arena (RED left, BLUE right); only the instruction —
# hence the VLM's chosen target and its saliency — differs between them.
# ----------------------------------------------------------------------------------
SIMS     = [("friendly", CFG.AMBIGUOUS_PROMPTS[0]), ("hostile", CFG.AMBIGUOUS_PROMPTS[1])]
VARIANTS = [("plain", LOOK_PLAIN), ("sal", LOOK_SAL)]
all_mp4s = {}

for sim_name, instr in SIMS:
    print(f"\n{'#'*78}\n#  CINEMATIC SIM: {sim_name.upper():8s} | instruction: {instr!r}\n{'#'*78}")
    env, left, right = build_arena(left_color=RED, right_color=BLUE)
    policy = load_policy(env)
    res = record_cinematic_trial(env, policy, left, right, instr, seed=CINE.SEED)
    del env, policy; gc.collect(); torch.cuda.empty_cache()

    cubes = []
    for o in (left, right):
        col = RED_RGB if color_name(o["color"]) == "red" else BLUE_RGB
        cubes.append(dict(pos=[float(o["pos"][0]), float(o["pos"][1])], size=CINE.CUBE_SIZE,
                          color=col, glow=LOOK_PLAIN["cube_glow"]))   # color/glow used by plain look only

    bundle = f"{CINE.WORK}/cine_bundle_{sim_name}"; save_geo(bundle, res, cubes)
    fps = res["render_fps"]

    for var_name, look in VARIANTS:
        is_sal = (var_name == "sal")
        tag = f"{sim_name}_{var_name}"
        kind = ("WITH saliency (vibrant heat projected on floor + matte-black targets, both views)"
                if is_sal else "WITHOUT saliency (NB4 cinematic)")
        print(f"\n==== render  {tag}  — {kind} ====")
        mp4s = {}
        for view, vc in VIEWS.items():
            extra = dict(variant=var_name,
                         show_surface_heat=is_sal,                       # accumulating glow: BOTH views
                         heat_grid=int(CINE.HEAT_GRID),
                         heat_extent=[float(v) for v in res["heat_extent"]])
            write_meta(bundle, look, fps, extra)                          # swap LOOK + heat overlay
            out = f"{CINE.WORK}/cine_{tag}_{view}"
            if run_blender(bundle, out, f"{bundle}/cams_{view}.npy", vc["lens"]) == 0:
                mp4 = f"{CINE.OUT}/cine_{tag}_{view}.mp4"; n = frames_to_mp4(out, mp4, fps)
                mp4s[view] = mp4; print(f"  [{tag} {view}] {mp4} ({n} frames)")
            else:
                print(f"  >>> {tag} {view} render FAILED — see log above.")
        if "iso" in mp4s and "pov" in mp4s:
            combo = f"{CINE.OUT}/cine_{tag}_combo.mp4"
            side_by_side(f"{CINE.WORK}/cine_{tag}_iso", f"{CINE.WORK}/cine_{tag}_pov", combo, fps)
            mp4s["combo"] = combo; print(f"  [{tag} combo] {combo} (iso | pov)")
        all_mp4s[tag] = mp4s

# ---- display every clip, grouped sim -> variant -> (combo, iso, pov) ----
for sim_name, _ in SIMS:
    for var_name, _ in VARIANTS:
        tag = f"{sim_name}_{var_name}"; mp4s = all_mp4s.get(tag, {})
        kind = ("WITH saliency · vibrant heat on floor + matte-black targets (both views)"
                if var_name == "sal" else "WITHOUT saliency · NB4 cinematic")
        print(f"\n{'='*72}\n  {sim_name.upper()}  —  {kind}\n{'='*72}")
        for view in ("combo", "iso", "pov"):
            if view in mp4s:
                print(f"[{tag} · {view}]"); display(Video(mp4s[view], embed=True, width=900))

print("\nAll cinematic renders complete:", sum(len(v) for v in all_mp4s.values()), "clips under", CINE.OUT)


In [ ]:
# to delete runtime and save credits
from google.colab import runtime
runtime.unassign()

## ✅ Acceptance checklist

- [ ] A100 + CUDA confirmed (§0)
- [ ] Headless rendering + dependencies installed (§1–§2)
- [ ] PPO `locomote` policy trained / restored (§9)
- [ ] Policy + MEM planner loaded (§10)
- [ ] VLM emits **continuous** `(v_x,v_y,w_z)` fed directly to PPO (§7, §11)
- [ ] **Short-term visual memory**: recent frames sent as a Qwen video (`used_video=True` in the
      planner's returned dict), or multi-image fallback (§7, §12)
- [ ] **Long-term language memory** rewrites + compresses each step and stays under the char cap (§7, §12)
- [ ] Memory ablations reproducible via `MEM_SHORT_ENABLED` / `MEM_LANG_ENABLED` (§3, §13-ABL)
- [ ] Post-hoc `bucketize()` labels each command (§8, §12)
- [ ] Integrated gradients attributed to the onboard frame(s), projected into the 3D arena (§13)

**MEM ablation grid** (set in §3, re-run §10 + §11): both off = no-memory baseline; short-only =
"Only Video Memory"; lang-only = "Only Text Memory"; both on = the full MEM-style hybrid.

**To change memory size:** `MEM_SHORT_FRAMES` / `MEM_SHORT_STRIDE` (visual horizon),
`MEM_LANG_MAX_CHARS` (summary length). **To change the task / contract:** edit `CFG.NEUTRAL_PROMPT`
etc. and `CFG.CONTRACT` in §3 (keep JSON keys `v_x,v_y,w_z` and `memory`).

**Ground-truth experiment settings (as run):**
*Driver* — MEM-V2 planner (§12a): 4 recent frames ~0.333 s apart as native Qwen video, no anchor
frame, text memory ≤ 600 chars, queries at 3 Hz, temperature 0.7, command low-pass α = 0.35.
*Behavioural evaluation* (§12) — N = 50 trials/category (batched ×6), 6 s timeout,
0.7 m commit radius, drift trials committed to the nearest target.
*Saliency* (§13–§15) — IG on the **chosen `w_z`** (teacher-forced, sign-inclusive), 8 Riemann
steps; ablation grid N = 20 trials/category, attribution every 3rd query; accumulation videos
(§14) 8 s trials with every query attributed after the tune cells; cinematic (§15) 6 s trials,
every 2nd query, final quality (`CINE.QUICK=False`).

**Note:** `qre_utils.sanity_gate()` (§5) is a standalone physics/render sanity check available for
ad-hoc debugging; it isn't called automatically anywhere in the flow above.
